In [7]:
%%time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
import requests
import time
import os
import json
import csv
import urllib.parse
import re
from datetime import datetime
from PIL import Image
import io

# =================== 설정 구역 ===================
# 여기서 검색어와 기타 설정을 변경하세요
SEARCH_KEYWORD = "신림동 햄버거"  # 원하는 검색어로 변경
MAX_RESTAURANTS = 10  # 크롤링할 최대 가게 수
MAX_REVIEWS_PER_RESTAURANT = 500  # 가게당 최대 리뷰 수
# ================================================

# 서비스 설정
service = Service(port=9999)
# 크롬 인터넷 설정
options = webdriver.ChromeOptions()
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36')
options.add_argument('window-size=1380,900')
# 드라이버 생성
driver = webdriver.Chrome(service=service, options=options)
# WebDriverWait 초기화
wait = WebDriverWait(driver, 10)

# 결과 저장할 리스트
all_restaurant_data = []

def create_directory_structure(keyword):
    """가게별 이미지 저장용 디렉토리 구조 생성"""
    base_dir = f"{keyword}_test2_sample_data"
    os.makedirs(base_dir, exist_ok=True)
    return base_dir

def create_restaurant_directories(base_dir, restaurant_name):
    """가게별 폴더 구조 생성"""
    safe_name = safe_filename(restaurant_name)
    restaurant_dir = os.path.join(base_dir, safe_name)
    menu_images_dir = os.path.join(restaurant_dir, "메뉴_이미지")
    review_images_dir = os.path.join(restaurant_dir, "리뷰_이미지")
    
    os.makedirs(menu_images_dir, exist_ok=True)
    os.makedirs(review_images_dir, exist_ok=True)
    
    return restaurant_dir, menu_images_dir, review_images_dir

def download_image_improved(url, filename, images_dir):
    """이미지 다운로드 (개선된 버전 - 이미지 검증 포함)"""
    try:
        # User-Agent 헤더 추가
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
            'Referer': 'https://map.naver.com/'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            # 이미지 데이터 검증
            try:
                # PIL로 이미지 검증
                img = Image.open(io.BytesIO(response.content))
                img.verify()  # 이미지가 유효한지 검증
                
                # 파일 저장
                filepath = os.path.join(images_dir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                print(f"  이미지 다운로드 성공: {filename}")
                return filepath
            except Exception as img_error:
                print(f"  이미지 검증 실패 ({filename}): {img_error}")
                return None
        else:
            print(f"  HTTP 오류 ({filename}): {response.status_code}")
            return None
    except Exception as e:
        print(f"  이미지 다운로드 실패 ({filename}): {e}")
        return None

def safe_filename(filename):
    """파일명에서 특수문자 제거"""
    return re.sub(r'[^\w\s-]', '', filename).strip()[:50]

def get_restaurant_list():
    """왼쪽 패널에서 가게 리스트 수집"""
    print("가게 리스트 수집 중...")
    
    # 스크롤하여 모든 가게 로드
    try:
        scroll_container = driver.find_element(By.CSS_SELECTOR, "div.Ryr1F")
        print("스크롤 컨테이너 찾음")
        
        previous_count = 0
        max_attempts = 15
        
        for attempt in range(max_attempts):
            restaurant_elements = driver.find_elements(By.CSS_SELECTOR, "li.UEzoS")
            current_count = len(restaurant_elements)
            
            print(f"현재 로드된 가게 수: {current_count}")
            
            if current_count == previous_count:
                if attempt >= 3:
                    print("더 이상 로드할 가게가 없음")
                    break
            else:
                previous_count = current_count
            
            # 스크롤 실행
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
            time.sleep(2)
        
        return restaurant_elements
        
    except Exception as e:
        print(f"가게 리스트 수집 실패: {e}")
        return []

def click_restaurant(restaurant_element, index):
    """가게 클릭하여 세부 정보 패널 열기"""
    try:
        # 가게 링크 찾기
        link_element = restaurant_element.find_element(By.CSS_SELECTOR, "a.place_bluelink")
        
        # 스크롤하여 요소가 보이도록 함
        driver.execute_script("arguments[0].scrollIntoView(true);", link_element)
        time.sleep(1)
        
        # 클릭
        ActionChains(driver).move_to_element(link_element).click().perform()
        print(f"가게 {index} 클릭 완료")
        
        # 세부 정보 패널 로딩 대기
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"가게 {index} 클릭 실패: {e}")
        return False

def switch_to_detail_iframe():
    """오른쪽 세부 정보 iframe으로 전환"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 세부 정보 iframe 찾기
        detail_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#entryIframe")))
        driver.switch_to.frame(detail_iframe)
        print("세부 정보 iframe으로 전환 완료")
        return True
        
    except Exception as e:
        print(f"세부 정보 iframe 전환 실패: {e}")
        return False

def get_home_info():
    """홈 탭에서 기본 정보 수집"""
    home_info = {}
    
    try:
        # 가게 이름
        name_element = driver.find_element(By.CSS_SELECTOR, "span.GHAhO")
        home_info['name'] = name_element.text.strip()
        print(f"가게명: {home_info['name']}")
        
        # 가게 분류
        category_element = driver.find_element(By.CSS_SELECTOR, "span.lnJFt")
        home_info['category'] = category_element.text.strip()
        print(f"분류: {home_info['category']}")
        
        # 가게 주소
        try:
            address_element = driver.find_element(By.CSS_SELECTOR, "span.LDgIH")
            home_info['address'] = address_element.text.strip()
            print(f"주소: {home_info['address']}")
        except:
            home_info['address'] = "주소 정보 없음"
        
        # 전화번호 (있다면)
        try:
            phone_element = driver.find_element(By.CSS_SELECTOR, "span.xlx7Q")
            home_info['phone'] = phone_element.text.strip()
        except:
            home_info['phone'] = "전화번호 정보 없음"
        
        # 영업시간 (있다면)
        try:
            hours_element = driver.find_element(By.CSS_SELECTOR, "time.H3ua4")
            home_info['hours'] = hours_element.text.strip()
        except:
            home_info['hours'] = "영업시간 정보 없음"
            
    except Exception as e:
        print(f"홈 정보 수집 실패: {e}")
        
    return home_info

def find_and_click_menu_tab():
    """메뉴 탭을 정확하게 찾아서 클릭"""
    try:
        print("메뉴 탭 찾는 중...")
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 메뉴 탭 찾기
        menu_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 메뉴 탭인지 판단
                is_menu_tab = (
                    "menu" in tab_href.lower() or
                    "메뉴" in tab_text or
                    "menu" in tab_text
                )
                
                if is_menu_tab:
                    menu_tab = tab
                    print(f"메뉴 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 메뉴 탭 클릭
        if menu_tab:
            try:
                # 스크롤하여 요소가 보이도록 함
                driver.execute_script("arguments[0].scrollIntoView(true);", menu_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", menu_tab)
                time.sleep(3)
                print("메뉴 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"메뉴 탭 클릭 실패: {e}")
                return False
        else:
            print("메뉴 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"메뉴 탭 찾기 실패: {e}")
        return False

def get_menu_info(menu_images_dir, restaurant_name):
    """메뉴 탭에서 메뉴 정보 수집"""
    menu_list = []
    
    try:
        # 메뉴 더보기 버튼 클릭
        try:
            more_button_selectors = [
                "a.fvwqf",
                "button.fvwqf",
                "a[class*='more']",
                "//a[contains(text(), '더보기')]"
            ]
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_button = driver.find_element(By.XPATH, selector)
                    else:
                        more_button = driver.find_element(By.CSS_SELECTOR, selector)
                    
                    if more_button.is_displayed() and more_button.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", more_button)
                        time.sleep(2)
                        print("메뉴 더보기 버튼 클릭 완료")
                        break
                except:
                    continue
        except:
            print("메뉴 더보기 버튼이 없거나 이미 모든 메뉴가 표시됨")
        
        # 메뉴 항목들 찾기
        menu_elements = driver.find_elements(By.CSS_SELECTOR, "li.E2jtL")
        
        for i, menu_element in enumerate(menu_elements):
            menu_info = {}
            
            try:
                # 메뉴명
                name_element = menu_element.find_element(By.CSS_SELECTOR, "span.lPzHi")
                menu_info['name'] = name_element.text.strip()
                
                # 메뉴 가격
                try:
                    price_element = menu_element.find_element(By.CSS_SELECTOR, "div.GXS1X em")
                    menu_info['price'] = price_element.text.strip()
                except:
                    menu_info['price'] = "가격 정보 없음"
                
                # 메뉴 설명
                try:
                    desc_element = menu_element.find_element(By.CSS_SELECTOR, "div.TRxGt")
                    menu_info['description'] = desc_element.text.strip()
                except:
                    menu_info['description'] = "설명 없음"
                
                # 메뉴 이미지
                try:
                    img_selectors = ["img.K0PDV", "img"]
                    img_element = None
                    
                    for selector in img_selectors:
                        try:
                            img_element = menu_element.find_element(By.CSS_SELECTOR, selector)
                            break
                        except:
                            continue
                    
                    if img_element:
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            # 이미지 파일명 생성 (특수문자 제거)
                            safe_menu_name = safe_filename(menu_info['name'])[:20]
                            img_filename = f"메뉴_{i+1:02d}_{safe_menu_name}.jpg"
                            
                            # 이미지 다운로드
                            downloaded_path = download_image_improved(img_url, img_filename, menu_images_dir)
                            menu_info['image_path'] = downloaded_path
                        else:
                            menu_info['image_path'] = None
                    else:
                        menu_info['image_path'] = None
                except:
                    menu_info['image_path'] = None
                
                menu_list.append(menu_info)
                print(f"메뉴 {i+1}: {menu_info['name']} - {menu_info['price']}")
                
            except Exception as e:
                print(f"메뉴 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"메뉴 정보 수집 실패: {e}")
    
    return menu_list

def find_and_click_review_tab():
    """리뷰 탭을 정확하게 찾아서 클릭"""
    try:
        print("리뷰 탭 찾는 중...")
        
        # 페이지 스크롤하여 탭이 보이도록 함
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 리뷰 탭 찾기
        review_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 리뷰 탭인지 판단
                is_review_tab = (
                    "review" in tab_href.lower() or
                    "리뷰" in tab_text or
                    "review" in tab_text
                )
                
                if is_review_tab:
                    review_tab = tab
                    print(f"리뷰 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 리뷰 탭 클릭
        if review_tab:
            try:
                # 요소가 보이도록 스크롤
                driver.execute_script("arguments[0].scrollIntoView(true);", review_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", review_tab)
                time.sleep(3)
                print("리뷰 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"리뷰 탭 클릭 실패: {e}")
                return False
        else:
            print("리뷰 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"리뷰 탭 찾기 실패: {e}")
        return False

def click_all_review_more_buttons():
    """모든 리뷰 더보기 버튼 클릭 (개선된 버전)"""
    try:
        print("리뷰 더보기 버튼들 찾는 중...")
        
        # 지정된 셀렉터로 더보기 버튼 찾기
        more_button_selectors = [
            "div.lfH3O.fvwqf",  # 사용자가 제공한 정확한 셀렉터
            "div.fvwqf",
            "a.fvwqf",
            "button.fvwqf",
            "//a[contains(text(), '더보기')]",
            "//button[contains(text(), '더보기')]",
            "//div[contains(text(), '더보기')]"
        ]
        
        clicked_count = 0
        max_attempts = 10  # 최대 10번 시도
        
        for attempt in range(max_attempts):
            print(f"더보기 버튼 찾기 시도 {attempt + 1}/{max_attempts}")
            
            button_found = False
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_buttons = driver.find_elements(By.XPATH, selector)
                    else:
                        more_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    
                    for button in more_buttons:
                        if button.is_displayed() and button.is_enabled():
                            try:
                                # 버튼이 보이도록 스크롤
                                driver.execute_script("arguments[0].scrollIntoView(true);", button)
                                time.sleep(1)
                                
                                # 버튼 텍스트 확인
                                button_text = button.text.strip()
                                print(f"발견된 버튼 텍스트: '{button_text}'")
                                
                                if "더보기" in button_text or "more" in button_text.lower():
                                    # JavaScript로 클릭
                                    driver.execute_script("arguments[0].click();", button)
                                    time.sleep(3)  # 로딩 대기
                                    clicked_count += 1
                                    button_found = True
                                    print(f"더보기 버튼 클릭 완료 ({clicked_count}번째)")
                                    break
                            except Exception as click_error:
                                print(f"버튼 클릭 실패: {click_error}")
                                continue
                    
                    if button_found:
                        break
                        
                except Exception as e:
                    continue
            
            if not button_found:
                print("더 이상 더보기 버튼을 찾을 수 없음")
                break
                
            # 페이지 하단으로 스크롤하여 새로운 리뷰 로드 확인
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        print(f"총 {clicked_count}개의 더보기 버튼을 클릭했습니다")
        return clicked_count > 0
        
    except Exception as e:
        print(f"리뷰 더보기 버튼 처리 실패: {e}")
        return False

def collect_review_images_with_slide(restaurant_name, review_images_dir):
    """리뷰 이미지 수집 (슬라이드 포함) - 개선된 버전"""
    downloaded_images = []
    
    try:
        print("리뷰 이미지 수집 시작...")
        
        # 리뷰 이미지 컨테이너 찾기
        review_containers = driver.find_elements(By.CSS_SELECTOR, "div.HH5sZ")
        print(f"리뷰 이미지 컨테이너 발견: {len(review_containers)}개")
        
        total_images = 0
        
        for container_idx, container in enumerate(review_containers):
            try:
                print(f"컨테이너 {container_idx + 1} 처리 중...")
                
                # 컨테이너가 보이도록 스크롤
                driver.execute_script("arguments[0].scrollIntoView(true);", container)
                time.sleep(1)
                
                # 현재 컨테이너의 이미지들 수집
                collected_in_container = 0
                max_slides = 10  # 최대 10번 슬라이드
                
                for slide_count in range(max_slides):
                    # 현재 보이는 이미지들 찾기
                    current_images = container.find_elements(By.CSS_SELECTOR, "a.place_thumb img.K0PDV")
                    
                    for img_idx, img in enumerate(current_images):
                        try:
                            img_url = img.get_attribute("src")
                            
                            if img_url and 'http' in img_url and 'pstatic.net' in img_url:
                                filename = f"리뷰_{container_idx+1:02d}_{slide_count+1:02d}_{img_idx+1:02d}.jpg"
                                
                                # 이미지가 이미 다운로드되었는지 확인 (URL 기준)
                                if img_url not in [img_info.get('url') for img_info in downloaded_images]:
                                    downloaded_path = download_image_improved(img_url, filename, review_images_dir)
                                    if downloaded_path:
                                        downloaded_images.append({
                                            'path': downloaded_path,
                                            'url': img_url,
                                            'filename': filename
                                        })
                                        collected_in_container += 1
                                        total_images += 1
                        
                        except Exception as e:
                            print(f"  이미지 처리 실패: {e}")
                            continue
                    
                    # 다음 슬라이드로 이동 시도
                    try:
                        next_button = container.find_element(By.CSS_SELECTOR, "span.nK_aH")
                        if next_button.is_displayed() and next_button.is_enabled():
                            # 다음 버튼 클릭
                            driver.execute_script("arguments[0].click();", next_button)
                            time.sleep(2)
                            print(f"  슬라이드 {slide_count + 1} -> {slide_count + 2}")
                        else:
                            print(f"  더 이상 슬라이드할 수 없음 (슬라이드 {slide_count + 1})")
                            break
                    except:
                        print(f"  슬라이드 버튼을 찾을 수 없음 (슬라이드 {slide_count + 1})")
                        break
                
                print(f"컨테이너 {container_idx + 1}에서 수집된 이미지: {collected_in_container}개")
                
            except Exception as e:
                print(f"컨테이너 {container_idx + 1} 처리 실패: {e}")
                continue
        
        print(f"리뷰 이미지 다운로드 완료: 총 {total_images}개")
        return [img['path'] for img in downloaded_images]
        
    except Exception as e:
        print(f"리뷰 이미지 수집 실패: {e}")
        return []

def get_review_info(review_images_dir, restaurant_name, max_reviews=None):
    """리뷰 탭에서 리뷰 정보 수집 (개선된 버전)"""
    if max_reviews is None:
        max_reviews = MAX_REVIEWS_PER_RESTAURANT
    
    review_list = []
    
    try:
        # 모든 리뷰 더보기 버튼 클릭
        click_all_review_more_buttons()
        
        # 리뷰 이미지 수집 (슬라이드 포함)
        review_slide_images = collect_review_images_with_slide(restaurant_name, review_images_dir)
        
        # 최종 스크롤하여 모든 리뷰 로드
        print("최종 스크롤로 모든 리뷰 로드 중...")
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # 리뷰 항목들 찾기 (여러 셀렉터 시도)
        review_selectors = [
            "li.place_apply_pui.EjjAW",
            "li.place_apply_pui",
            "div.pui__vn15t2",
            "li[class*='review']",
            "div[class*='review']"
        ]
        
        review_elements = []
        for selector in review_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    review_elements = elements[:max_reviews]
                    print(f"리뷰 요소 찾음: {selector} - {len(review_elements)}개")
                    break
            except:
                continue
        
        if not review_elements:
            print("리뷰 요소를 찾을 수 없습니다.")
            return review_list
        
        print(f"총 {len(review_elements)}개의 리뷰 처리 시작")
        
        for i, review_element in enumerate(review_elements):
            review_info = {}
            
            try:
                # 리뷰 텍스트 (여러 셀렉터 시도)
                text_selectors = [
                    "span.zPfVt", 
                    "div.pui__vn15t2 span",
                    "div.pui__vn15t2",
                    "span[class*='review']", 
                    "div[class*='content']",
                    "div.YeINN",
                    "span.YeINN"
                ]
                
                review_text = "리뷰 텍스트 없음"
                
                for text_selector in text_selectors:
                    try:
                        text_elements = review_element.find_elements(By.CSS_SELECTOR, text_selector)
                        for text_element in text_elements:
                            text_content = text_element.text.strip()
                            # 의미있는 텍스트가 있는지 확인 (길이 10자 이상)
                            if text_content and len(text_content) > 10:
                                review_text = text_content
                                break
                        if review_text != "리뷰 텍스트 없음":
                            break
                    except:
                        continue
                
                # 여전히 텍스트를 못찾았다면 전체 텍스트에서 추출 시도
                if review_text == "리뷰 텍스트 없음":
                    try:
                        full_text = review_element.text.strip()
                        # 긴 텍스트가 있다면 첫 번째 문장 추출
                        if len(full_text) > 20:
                            sentences = full_text.split('\n')
                            for sentence in sentences:
                                if len(sentence.strip()) > 10:
                                    review_text = sentence.strip()
                                    break
                    except:
                        pass
                
                review_info['text'] = review_text
                
                # 리뷰 날짜 (개선된 버전)
                date_selectors = [
                    "time", 
                    "span.pui__gfuUIT", 
                    "div.pui__QKE5Pr", 
                    "span[class*='date']",
                    "div[class*='date']",
                    "span.time",
                    ".pui__time"
                ]
                review_date = "날짜 정보 없음"
                
                for date_selector in date_selectors:
                    try:
                        date_element = review_element.find_element(By.CSS_SELECTOR, date_selector)
                        date_text = date_element.text.strip()
                        
                        # 날짜 패턴 추출 (더 다양한 패턴 추가)
                        date_patterns = [
                            r'(\d{4}\.\d{1,2}\.\d{1,2})',  # 2024.01.15
                            r'(\d{4}-\d{1,2}-\d{1,2})',   # 2024-01-15
                            r'(\d{1,2}\.\d{1,2})',        # 01.15
                            r'(\d+일전)',                  # 3일전
                            r'(\d+주전)',                  # 2주전
                            r'(\d+개월전)',                # 1개월전
                            r'(\d+년전)',                  # 1년전
                            r'(오늘)',                     # 오늘
                            r'(어제)',                     # 어제
                            r'(\d{4}년\s*\d{1,2}월\s*\d{1,2}일)'  # 2024년 1월 15일
                        ]
                        
                        for pattern in date_patterns:
                            date_match = re.search(pattern, date_text)
                            if date_match:
                                review_date = date_match.group(1)
                                break
                        
                        if review_date != "날짜 정보 없음":
                            break
                        elif date_text and len(date_text) > 0 and len(date_text) < 30:
                            review_date = date_text
                            break
                    except:
                        continue
                
                review_info['date'] = review_date
                
                # 리뷰 평점 (별점) 수집
                try:
                    rating_selectors = [
                        "div.pui__rating span",
                        "span[class*='rating']",
                        "div[class*='star']",
                        ".rating"
                    ]
                    
                    review_rating = "평점 정보 없음"
                    for rating_selector in rating_selectors:
                        try:
                            rating_element = review_element.find_element(By.CSS_SELECTOR, rating_selector)
                            rating_text = rating_element.text.strip()
                            if rating_text and ("점" in rating_text or "★" in rating_text):
                                review_rating = rating_text
                                break
                        except:
                            continue
                    
                    review_info['rating'] = review_rating
                except:
                    review_info['rating'] = "평점 정보 없음"
                
                # 리뷰 이미지들 (개별 리뷰에서)
                review_images = []
                try:
                    img_elements = review_element.find_elements(By.CSS_SELECTOR, "img")
                    for j, img_element in enumerate(img_elements):
                        img_url = img_element.get_attribute("src")
                        if img_url and ("review" in img_url or "ldb-phinf" in img_url or "pstatic.net" in img_url):
                            # 안전한 파일명 생성
                            safe_name = safe_filename(restaurant_name)[:20]
                            img_filename = f"개별리뷰_{i+1:02d}_{j+1:02d}.jpg"
                            
                            downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                            if downloaded_path:
                                review_images.append(downloaded_path)
                        
                except Exception as e:
                    print(f"리뷰 {i+1} 개별 이미지 수집 실패: {e}")
                
                review_info['individual_images'] = review_images
                review_list.append(review_info)
                
                # 진행 상황 출력
                if i % 10 == 0:
                    print(f"리뷰 처리 진행률: {i+1}/{len(review_elements)}")
                
                print(f"리뷰 {i+1}: {review_info['text'][:50]}..." if len(review_info['text']) > 50 else f"리뷰 {i+1}: {review_info['text']}")
                
            except Exception as e:
                print(f"리뷰 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"리뷰 정보 수집 실패: {e}")
        
    print(f"총 {len(review_list)}개의 리뷰 수집 완료")
    return review_list

def go_back_to_list():
    """목록으로 돌아가기"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 검색 결과 iframe으로 다시 전환
        search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
        driver.switch_to.frame(search_iframe)
        print("목록으로 돌아가기 완료")
        return True
        
    except Exception as e:
        print(f"목록으로 돌아가기 실패: {e}")
        return False

# =================== 메인 실행 부분 ===================
try:
    print("=" * 60)
    print(f"네이버 지도 크롤링 시작")
    print(f"검색어: {SEARCH_KEYWORD}")
    print(f"최대 가게 수: {MAX_RESTAURANTS}")
    print(f"가게당 최대 리뷰 수: {MAX_REVIEWS_PER_RESTAURANT}")
    print("=" * 60)
    
    # 1. 검색어 설정 및 접속
    encoded_keyword = urllib.parse.quote(SEARCH_KEYWORD)
    URL = f"https://map.naver.com/p/search/{encoded_keyword}"
    
    print(f"네이버 지도 접속 중: {URL}")
    driver.get(URL)
    time.sleep(5)
    
    # 2. 디렉토리 생성
    base_dir = create_directory_structure(SEARCH_KEYWORD)
    print(f"저장 디렉토리 생성: {base_dir}")
    
    # 3. 검색 결과 iframe 접근
    print("검색 결과 iframe으로 전환 중...")
    search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
    driver.switch_to.frame(search_iframe)
    
    # 4. 가게 리스트 수집
    restaurant_elements = get_restaurant_list()
    total_restaurants = len(restaurant_elements)
    print(f"총 {total_restaurants}개 가게 발견")
    
    # 5. 각 가게별로 세부 정보 수집
    max_restaurants = min(MAX_RESTAURANTS, total_restaurants)
    
    for i in range(max_restaurants):
        print(f"\n{'='*20} 가게 {i+1}/{max_restaurants} {'='*20}")
        
        try:
            # 가게 클릭
            if not click_restaurant(restaurant_elements[i], i+1):
                continue
            
            # 세부 정보 iframe으로 전환
            if not switch_to_detail_iframe():
                continue
            
            # 홈 정보 수집
            print("홈 정보 수집 중...")
            home_info = get_home_info()
            
            # 가게별 폴더 구조 생성
            restaurant_dir, menu_images_dir, review_images_dir = create_restaurant_directories(
                base_dir, home_info.get('name', f'restaurant_{i+1}')
            )
            
            # 메뉴 정보 수집
            menu_info = []
            if find_and_click_menu_tab():
                print("메뉴 정보 수집 중...")
                menu_info = get_menu_info(menu_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("메뉴 탭을 찾을 수 없어 메뉴 정보 수집을 건너뜁니다.")
            
            # 리뷰 정보 수집
            review_info = []
            if find_and_click_review_tab():
                print("리뷰 정보 수집 중...")
                review_info = get_review_info(review_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("리뷰 탭을 찾을 수 없어 리뷰 정보 수집을 건너뜁니다.")
            
            # 전체 정보 결합
            restaurant_data = {
                'index': i + 1,
                'home_info': home_info,
                'menu_info': menu_info,
                'review_info': review_info,
                'restaurant_dir': restaurant_dir,
                'collected_at': datetime.now().isoformat()
            }
            
            all_restaurant_data.append(restaurant_data)
            print(f"가게 {i+1} 정보 수집 완료!")
            print(f"- 메뉴 개수: {len(menu_info)}")
            print(f"- 리뷰 개수: {len(review_info)}")
            
            # 목록으로 돌아가기
            go_back_to_list()
            time.sleep(2)
            
        except Exception as e:
            print(f"가게 {i+1} 처리 중 오류: {e}")
            # 목록으로 돌아가기 시도
            try:
                go_back_to_list()
            except:
                pass
            continue
    
    # 6. 결과 저장
    print(f"\n{'='*60}")
    print(f"총 {len(all_restaurant_data)}개 가게 상세 정보 수집 완료!")
    
    # JSON 파일 저장
    safe_keyword = safe_filename(SEARCH_KEYWORD)
    json_filename = os.path.join(base_dir, f"{safe_keyword}_detailed_restaurants.json")
    with open(json_filename, 'w', encoding='utf-8') as jsonfile:
        json.dump(all_restaurant_data, jsonfile, ensure_ascii=False, indent=2)
    
    # CSV 파일 저장 (한글 헤더로 개선)
    csv_filename = os.path.join(base_dir, f"{safe_keyword}_restaurants_summary.csv")
    with open(csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:  # utf-8-sig로 BOM 추가
        fieldnames = ['순번', '가게명', '분류', '주소', '전화번호', '메뉴수', '리뷰수', '폴더경로']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            row = {
                '순번': restaurant['index'],
                '가게명': restaurant['home_info'].get('name', ''),
                '분류': restaurant['home_info'].get('category', ''),
                '주소': restaurant['home_info'].get('address', ''),
                '전화번호': restaurant['home_info'].get('phone', ''),
                '메뉴수': len(restaurant['menu_info']),
                '리뷰수': len(restaurant['review_info']),
                '폴더경로': restaurant.get('restaurant_dir', '')
            }
            writer.writerow(row)
    
    # 리뷰 상세 정보 CSV 저장
    reviews_csv_filename = os.path.join(base_dir, f"{safe_keyword}_reviews_detail.csv")
    with open(reviews_csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['가게명', '리뷰번호', '리뷰내용', '작성날짜', '평점', '이미지수']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            restaurant_name = restaurant['home_info'].get('name', '')
            for idx, review in enumerate(restaurant['review_info']):
                row = {
                    '가게명': restaurant_name,
                    '리뷰번호': idx + 1,
                    '리뷰내용': review.get('text', ''),
                    '작성날짜': review.get('date', ''),
                    '평점': review.get('rating', ''),
                    '이미지수': len(review.get('individual_images', []))
                }
                writer.writerow(row)
    
    # 통계 정보 출력
    total_menus = sum(len(restaurant['menu_info']) for restaurant in all_restaurant_data)
    total_reviews = sum(len(restaurant['review_info']) for restaurant in all_restaurant_data)
    total_images = 0
    
    # 이미지 파일 개수 세기
    for restaurant in all_restaurant_data:
        restaurant_dir = restaurant.get('restaurant_dir', '')
        if os.path.exists(restaurant_dir):
            for root, dirs, files in os.walk(restaurant_dir):
                image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
                total_images += len(image_files)
    
    print(f"\n파일 저장 완료:")
    print(f"- 상세 정보 (JSON): {json_filename}")
    print(f"- 요약 정보 (CSV): {csv_filename}")
    print(f"- 리뷰 상세 (CSV): {reviews_csv_filename}")
    print(f"- 이미지 폴더: 각 가게별 폴더")
    print(f"\n수집 통계:")
    print(f"- 총 가게 수: {len(all_restaurant_data)}")
    print(f"- 총 메뉴 수: {total_menus}")
    print(f"- 총 리뷰 수: {total_reviews}")
    print(f"- 총 이미지 수: {total_images}")
    
    print(f"\n{'='*60}")
    print("크롤링 완료!")

except Exception as e:
    print(f"전체 프로세스 오류 발생: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    # 드라이버 종료
    try:
        pass
#         driver.quit()
        print("웹드라이버 종료 완료")
    except:
        pass

네이버 지도 크롤링 시작
검색어: 신림동 햄버거
최대 가게 수: 10
가게당 최대 리뷰 수: 500
네이버 지도 접속 중: https://map.naver.com/p/search/%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0
웹드라이버 종료 완료


KeyboardInterrupt: 

In [6]:
## 0708 다시 진짜 찐 마지막!

In [13]:
from time import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
import requests
import time
import os
import json
import csv
import urllib.parse
import re
from datetime import datetime
from PIL import Image
import io

start_time = time.time()
# =================== 설정 구역 ===================
# 여기서 검색어와 기타 설정을 변경하세요
SEARCH_KEYWORD = "신림동 햄버거"  # 원하는 검색어로 변경
MAX_RESTAURANTS = 5  # 크롤링할 최대 가게 수
MAX_REVIEWS_PER_RESTAURANT = 500  # 가게당 최대 리뷰 수
# ================================================

# 서비스 설정
service = Service(port=9999)
# 크롬 인터넷 설정
options = webdriver.ChromeOptions()
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36')
options.add_argument('window-size=1380,900')
# 드라이버 생성
driver = webdriver.Chrome(service=service, options=options)
# WebDriverWait 초기화
wait = WebDriverWait(driver, 10)

# 결과 저장할 리스트
all_restaurant_data = []

def create_directory_structure(keyword):
    """가게별 이미지 저장용 디렉토리 구조 생성"""
    base_dir = f"{keyword}_test2_sample2_data"
    os.makedirs(base_dir, exist_ok=True)
    return base_dir

def create_restaurant_directories(base_dir, restaurant_name):
    """가게별 폴더 구조 생성"""
    safe_name = safe_filename(restaurant_name)
    restaurant_dir = os.path.join(base_dir, safe_name)
    menu_images_dir = os.path.join(restaurant_dir, "메뉴_이미지")
    review_images_dir = os.path.join(restaurant_dir, "리뷰_이미지")
    
    os.makedirs(menu_images_dir, exist_ok=True)
    os.makedirs(review_images_dir, exist_ok=True)
    
    return restaurant_dir, menu_images_dir, review_images_dir

def download_image_improved(url, filename, images_dir):
    """이미지 다운로드 (개선된 버전 - 이미지 검증 포함)"""
    try:
        # User-Agent 헤더 추가
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
            'Referer': 'https://map.naver.com/'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            # 이미지 데이터 검증
            try:
                # PIL로 이미지 검증
                img = Image.open(io.BytesIO(response.content))
                img.verify()  # 이미지가 유효한지 검증
                
                # 파일 저장
                filepath = os.path.join(images_dir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                print(f"  이미지 다운로드 성공: {filename}")
                return filepath
            except Exception as img_error:
                print(f"  이미지 검증 실패 ({filename}): {img_error}")
                return None
        else:
            print(f"  HTTP 오류 ({filename}): {response.status_code}")
            return None
    except Exception as e:
        print(f"  이미지 다운로드 실패 ({filename}): {e}")
        return None

def safe_filename(filename):
    """파일명에서 특수문자 제거"""
    return re.sub(r'[^\w\s-]', '', filename).strip()[:50]

def get_restaurant_list():
    """왼쪽 패널에서 가게 리스트 수집"""
    print("가게 리스트 수집 중...")
    
    # 스크롤하여 모든 가게 로드
    try:
        scroll_container = driver.find_element(By.CSS_SELECTOR, "div.Ryr1F")
        print("스크롤 컨테이너 찾음")
        
        previous_count = 0
        max_attempts = 15
        
        for attempt in range(max_attempts):
            restaurant_elements = driver.find_elements(By.CSS_SELECTOR, "li.UEzoS")
            current_count = len(restaurant_elements)
            
            print(f"현재 로드된 가게 수: {current_count}")
            
            if current_count == previous_count:
                if attempt >= 3:
                    print("더 이상 로드할 가게가 없음")
                    break
            else:
                previous_count = current_count
            
            # 스크롤 실행
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
            time.sleep(2)
        
        return restaurant_elements
        
    except Exception as e:
        print(f"가게 리스트 수집 실패: {e}")
        return []

def click_restaurant(restaurant_element, index):
    """가게 클릭하여 세부 정보 패널 열기"""
    try:
        # 가게 링크 찾기
        link_element = restaurant_element.find_element(By.CSS_SELECTOR, "a.place_bluelink")
        
        # 스크롤하여 요소가 보이도록 함
        driver.execute_script("arguments[0].scrollIntoView(true);", link_element)
        time.sleep(1)
        
        # 클릭
        ActionChains(driver).move_to_element(link_element).click().perform()
        print(f"가게 {index} 클릭 완료")
        
        # 세부 정보 패널 로딩 대기
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"가게 {index} 클릭 실패: {e}")
        return False

def switch_to_detail_iframe():
    """오른쪽 세부 정보 iframe으로 전환"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 세부 정보 iframe 찾기
        detail_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#entryIframe")))
        driver.switch_to.frame(detail_iframe)
        print("세부 정보 iframe으로 전환 완료")
        return True
        
    except Exception as e:
        print(f"세부 정보 iframe 전환 실패: {e}")
        return False

def get_home_info():
    """홈 탭에서 기본 정보 수집"""
    home_info = {}
    
    try:
        # 가게 이름
        name_element = driver.find_element(By.CSS_SELECTOR, "span.GHAhO")
        home_info['name'] = name_element.text.strip()
        print(f"가게명: {home_info['name']}")
        
        # 가게 분류
        category_element = driver.find_element(By.CSS_SELECTOR, "span.lnJFt")
        home_info['category'] = category_element.text.strip()
        print(f"분류: {home_info['category']}")
        
        # 가게 주소
        try:
            address_element = driver.find_element(By.CSS_SELECTOR, "span.LDgIH")
            home_info['address'] = address_element.text.strip()
            print(f"주소: {home_info['address']}")
        except:
            home_info['address'] = "주소 정보 없음"
        
        # 전화번호 (있다면)
        try:
            phone_element = driver.find_element(By.CSS_SELECTOR, "span.xlx7Q")
            home_info['phone'] = phone_element.text.strip()
        except:
            home_info['phone'] = "전화번호 정보 없음"
        
        # 영업시간 (있다면)
        try:
            hours_element = driver.find_element(By.CSS_SELECTOR, "time.H3ua4")
            home_info['hours'] = hours_element.text.strip()
        except:
            home_info['hours'] = "영업시간 정보 없음"
            
    except Exception as e:
        print(f"홈 정보 수집 실패: {e}")
        
    return home_info

def find_and_click_menu_tab():
    """메뉴 탭을 정확하게 찾아서 클릭"""
    try:
        print("메뉴 탭 찾는 중...")
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 메뉴 탭 찾기
        menu_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 메뉴 탭인지 판단
                is_menu_tab = (
                    "menu" in tab_href.lower() or
                    "메뉴" in tab_text or
                    "menu" in tab_text
                )
                
                if is_menu_tab:
                    menu_tab = tab
                    print(f"메뉴 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 메뉴 탭 클릭
        if menu_tab:
            try:
                # 스크롤하여 요소가 보이도록 함
                driver.execute_script("arguments[0].scrollIntoView(true);", menu_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", menu_tab)
                time.sleep(3)
                print("메뉴 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"메뉴 탭 클릭 실패: {e}")
                return False
        else:
            print("메뉴 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"메뉴 탭 찾기 실패: {e}")
        return False

def get_menu_info(menu_images_dir, restaurant_name):
    """메뉴 탭에서 메뉴 정보 수집"""
    menu_list = []
    
    try:
        # 메뉴 더보기 버튼 클릭
        try:
            more_button_selectors = [
                "a.fvwqf",
                "button.fvwqf",
                "a[class*='more']",
                "//a[contains(text(), '더보기')]"
            ]
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_button = driver.find_element(By.XPATH, selector)
                    else:
                        more_button = driver.find_element(By.CSS_SELECTOR, selector)
                    
                    if more_button.is_displayed() and more_button.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", more_button)
                        time.sleep(2)
                        print("메뉴 더보기 버튼 클릭 완료")
                        break
                except:
                    continue
        except:
            print("메뉴 더보기 버튼이 없거나 이미 모든 메뉴가 표시됨")
        
        # 메뉴 항목들 찾기
        menu_elements = driver.find_elements(By.CSS_SELECTOR, "li.E2jtL")
        
        for i, menu_element in enumerate(menu_elements):
            menu_info = {}
            
            try:
                # 메뉴명
                name_element = menu_element.find_element(By.CSS_SELECTOR, "span.lPzHi")
                menu_info['name'] = name_element.text.strip()
                
                # 메뉴 가격
                try:
                    price_element = menu_element.find_element(By.CSS_SELECTOR, "div.GXS1X em")
                    menu_info['price'] = price_element.text.strip()
                except:
                    menu_info['price'] = "가격 정보 없음"
                
                # 메뉴 설명
                try:
                    desc_element = menu_element.find_element(By.CSS_SELECTOR, "div.TRxGt")
                    menu_info['description'] = desc_element.text.strip()
                except:
                    menu_info['description'] = "설명 없음"
                
                # 메뉴 이미지
                try:
                    img_selectors = ["img.K0PDV", "img"]
                    img_element = None
                    
                    for selector in img_selectors:
                        try:
                            img_element = menu_element.find_element(By.CSS_SELECTOR, selector)
                            break
                        except:
                            continue
                    
                    if img_element:
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            # 이미지 파일명 생성 (특수문자 제거)
                            safe_menu_name = safe_filename(menu_info['name'])[:20]
                            img_filename = f"메뉴_{i+1:02d}_{safe_menu_name}.jpg"
                            
                            # 이미지 다운로드
                            downloaded_path = download_image_improved(img_url, img_filename, menu_images_dir)
                            menu_info['image_path'] = downloaded_path
                        else:
                            menu_info['image_path'] = None
                    else:
                        menu_info['image_path'] = None
                except:
                    menu_info['image_path'] = None
                
                menu_list.append(menu_info)
                print(f"메뉴 {i+1}: {menu_info['name']} - {menu_info['price']}")
                
            except Exception as e:
                print(f"메뉴 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"메뉴 정보 수집 실패: {e}")
    
    return menu_list

def find_and_click_review_tab():
    """리뷰 탭을 정확하게 찾아서 클릭"""
    try:
        print("리뷰 탭 찾는 중...")
        
        # 페이지 스크롤하여 탭이 보이도록 함
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 리뷰 탭 찾기
        review_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 리뷰 탭인지 판단
                is_review_tab = (
                    "review" in tab_href.lower() or
                    "리뷰" in tab_text or
                    "review" in tab_text
                )
                
                if is_review_tab:
                    review_tab = tab
                    print(f"리뷰 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 리뷰 탭 클릭
        if review_tab:
            try:
                # 요소가 보이도록 스크롤
                driver.execute_script("arguments[0].scrollIntoView(true);", review_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", review_tab)
                time.sleep(3)
                print("리뷰 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"리뷰 탭 클릭 실패: {e}")
                return False
        else:
            print("리뷰 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"리뷰 탭 찾기 실패: {e}")
        return False

def click_all_review_more_buttons():
    """모든 리뷰 더보기 버튼 클릭 (개선된 버전)"""
    try:
        print("리뷰 더보기 버튼들 찾는 중...")
        
        # 지정된 셀렉터로 더보기 버튼 찾기
        more_button_selectors = [
            "div.lfH3O.fvwqf",  # 사용자가 제공한 정확한 셀렉터
            "div.fvwqf",
            "a.fvwqf",
            "button.fvwqf",
            "//a[contains(text(), '더보기')]",
            "//button[contains(text(), '더보기')]",
            "//div[contains(text(), '더보기')]"
        ]
        
        clicked_count = 0
        max_attempts = 10  # 최대 10번 시도
        
        for attempt in range(max_attempts):
            print(f"더보기 버튼 찾기 시도 {attempt + 1}/{max_attempts}")
            
            button_found = False
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_buttons = driver.find_elements(By.XPATH, selector)
                    else:
                        more_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    
                    for button in more_buttons:
                        if button.is_displayed() and button.is_enabled():
                            try:
                                # 버튼이 보이도록 스크롤
                                driver.execute_script("arguments[0].scrollIntoView(true);", button)
                                time.sleep(1)
                                
                                # 버튼 텍스트 확인
                                button_text = button.text.strip()
                                print(f"발견된 버튼 텍스트: '{button_text}'")
                                
                                if "더보기" in button_text or "more" in button_text.lower():
                                    # JavaScript로 클릭
                                    driver.execute_script("arguments[0].click();", button)
                                    time.sleep(3)  # 로딩 대기
                                    clicked_count += 1
                                    button_found = True
                                    print(f"더보기 버튼 클릭 완료 ({clicked_count}번째)")
                                    break
                            except Exception as click_error:
                                print(f"버튼 클릭 실패: {click_error}")
                                continue
                    
                    if button_found:
                        break
                        
                except Exception as e:
                    continue
            
            if not button_found:
                print("더 이상 더보기 버튼을 찾을 수 없음")
                break
                
            # 페이지 하단으로 스크롤하여 새로운 리뷰 로드 확인
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        print(f"총 {clicked_count}개의 더보기 버튼을 클릭했습니다")
        return clicked_count > 0
        
    except Exception as e:
        print(f"리뷰 더보기 버튼 처리 실패: {e}")
        return False

def collect_review_images_with_slide(restaurant_name, review_images_dir):
    """리뷰 이미지 수집 (슬라이드 포함) - 정확한 셀렉터 사용"""
    downloaded_images = []
    
    try:
        print("리뷰 이미지 수집 시작...")
        
        # 정확한 리뷰 이미지 컨테이너 찾기 (프로필 사진 제외)
        # div.flicking-camera 안의 div.HH5sZ만 선택
        camera_containers = driver.find_elements(By.CSS_SELECTOR, "div.flicking-camera")
        
        total_images = 0
        
        for camera_idx, camera_container in enumerate(camera_containers):
            try:
                # 각 카메라 컨테이너 안의 HH5sZ 요소들 찾기
                review_containers = camera_container.find_elements(By.CSS_SELECTOR, "div.HH5sZ")
                print(f"카메라 컨테이너 {camera_idx + 1}에서 리뷰 이미지 컨테이너 발견: {len(review_containers)}개")
                
                for container_idx, container in enumerate(review_containers):
                    try:
                        print(f"리뷰 이미지 컨테이너 {container_idx + 1} 처리 중...")
                        
                        # 컨테이너가 보이도록 스크롤
                        driver.execute_script("arguments[0].scrollIntoView(true);", container)
                        time.sleep(1)
                        
                        # 현재 컨테이너의 이미지들 수집
                        collected_in_container = 0
                        max_slides = 10  # 최대 10번 슬라이드
                        
                        for slide_count in range(max_slides):
                            # 현재 보이는 이미지들 찾기 (프로필 사진 제외하고 실제 리뷰 이미지만)
                            current_images = container.find_elements(By.CSS_SELECTOR, "img")
                            
                            for img_idx, img in enumerate(current_images):
                                try:
                                    img_url = img.get_attribute("src")
                                    img_alt = img.get_attribute("alt") or ""
                                    
                                    # 프로필 사진이 아닌 실제 리뷰 이미지만 필터링
                                    if (img_url and 'http' in img_url and 
                                        ('pstatic.net' in img_url or 'blogfiles' in img_url) and
                                        'profile' not in img_url.lower() and
                                        'avatar' not in img_url.lower() and
                                        'user' not in img_alt.lower() and
                                        'profile' not in img_alt.lower()):
                                        
                                        filename = f"리뷰이미지_{camera_idx+1:02d}_{container_idx+1:02d}_{slide_count+1:02d}_{img_idx+1:02d}.jpg"
                                        
                                        # 이미지가 이미 다운로드되었는지 확인 (URL 기준)
                                        if img_url not in [img_info.get('url') for img_info in downloaded_images]:
                                            downloaded_path = download_image_improved(img_url, filename, review_images_dir)
                                            if downloaded_path:
                                                downloaded_images.append({
                                                    'path': downloaded_path,
                                                    'url': img_url,
                                                    'filename': filename
                                                })
                                                collected_in_container += 1
                                                total_images += 1
                                
                                except Exception as e:
                                    print(f"  이미지 처리 실패: {e}")
                                    continue
                            
                            # 다음 슬라이드로 이동 시도
                            try:
                                next_button = container.find_element(By.CSS_SELECTOR, "span.nK_aH, button[class*='next'], a[class*='next']")
                                if next_button.is_displayed() and next_button.is_enabled():
                                    # 다음 버튼 클릭
                                    driver.execute_script("arguments[0].click();", next_button)
                                    time.sleep(2)
                                    print(f"  슬라이드 {slide_count + 1} -> {slide_count + 2}")
                                else:
                                    print(f"  더 이상 슬라이드할 수 없음 (슬라이드 {slide_count + 1})")
                                    break
                            except:
                                print(f"  슬라이드 버튼을 찾을 수 없음 (슬라이드 {slide_count + 1})")
                                break
                        
                        print(f"컨테이너 {container_idx + 1}에서 수집된 이미지: {collected_in_container}개")
                        
                    except Exception as e:
                        print(f"컨테이너 {container_idx + 1} 처리 실패: {e}")
                        continue
                        
            except Exception as e:
                print(f"카메라 컨테이너 {camera_idx + 1} 처리 실패: {e}")
                continue
        
        print(f"리뷰 이미지 다운로드 완료: 총 {total_images}개")
        return [img['path'] for img in downloaded_images]
        
    except Exception as e:
        print(f"리뷰 이미지 수집 실패: {e}")
        return []

def get_review_info(review_images_dir, restaurant_name, max_reviews=None):
    """리뷰 탭에서 리뷰 정보 수집 (개선된 버전)"""
    if max_reviews is None:
        max_reviews = MAX_REVIEWS_PER_RESTAURANT
    
    review_list = []
    
    try:
        # 모든 리뷰 더보기 버튼 클릭
        click_all_review_more_buttons()
        
        # 리뷰 이미지 수집 (슬라이드 포함)
        review_slide_images = collect_review_images_with_slide(restaurant_name, review_images_dir)
        
        # 최종 스크롤하여 모든 리뷰 로드
        print("최종 스크롤로 모든 리뷰 로드 중...")
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # 리뷰 항목들 찾기 (여러 셀렉터 시도)
        review_selectors = [
            "li.place_apply_pui.EjjAW",
            "li.place_apply_pui",
            "div.pui__vn15t2",
            "li[class*='review']",
            "div[class*='review']"
        ]
        
        review_elements = []
        for selector in review_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    review_elements = elements[:max_reviews]
                    print(f"리뷰 요소 찾음: {selector} - {len(review_elements)}개")
                    break
            except:
                continue
        
        if not review_elements:
            print("리뷰 요소를 찾을 수 없습니다.")
            return review_list
        
        print(f"총 {len(review_elements)}개의 리뷰 처리 시작")
        
        for i, review_element in enumerate(review_elements):
            review_info = {}
            
            try:
                # 리뷰 텍스트 - 정확한 셀렉터 사용
                review_text = "리뷰 텍스트 없음"
                try:
                    # div.pui__vn15t2에서 리뷰 텍스트 찾기
                    text_element = review_element.find_element(By.CSS_SELECTOR, "div.pui__vn15t2")
                    review_text = text_element.text.strip()
                    if not review_text or len(review_text) < 5:
                        # 다른 텍스트 셀렉터들도 시도
                        text_selectors = [
                            "span.zPfVt", 
                            "div.pui__vn15t2 span",
                            "span[class*='review']", 
                            "div[class*='content']"
                        ]
                        
                        for text_selector in text_selectors:
                            try:
                                text_elements = review_element.find_elements(By.CSS_SELECTOR, text_selector)
                                for text_elem in text_elements:
                                    text_content = text_elem.text.strip()
                                    if text_content and len(text_content) > 10:
                                        review_text = text_content
                                        break
                                if review_text != "리뷰 텍스트 없음":
                                    break
                            except:
                                continue
                except:
                    # 여전히 텍스트를 못찾았다면 전체 텍스트에서 추출 시도
                    try:
                        full_text = review_element.text.strip()
                        if len(full_text) > 20:
                            sentences = full_text.split('\n')
                            for sentence in sentences:
                                if len(sentence.strip()) > 10:
                                    review_text = sentence.strip()
                                    break
                    except:
                        pass
                
                review_info['text'] = review_text
                
                # 리뷰 날짜 - 정확한 셀렉터 사용
                review_date = "날짜 정보 없음"
                try:
                    # span.pui__gfuUIT 안의 span.pui__blind에서 날짜 찾기
                    date_container = review_element.find_element(By.CSS_SELECTOR, "span.pui__gfuUIT")
                    date_element = date_container.find_element(By.CSS_SELECTOR, "span.pui__blind")
                    date_text = date_element.text.strip()
                    
                    if date_text:
                        # 날짜 패턴 추출
                        date_patterns = [
                            r'(\d{4}\.\d{1,2}\.\d{1,2})',  # 2024.01.15
                            r'(\d{4}-\d{1,2}-\d{1,2})',   # 2024-01-15
                            r'(\d{1,2}\.\d{1,2})',        # 01.15
                            r'(\d+일전)',                  # 3일전
                            r'(\d+주전)',                  # 2주전
                            r'(\d+개월전)',                # 1개월전
                            r'(\d+년전)',                  # 1년전
                            r'(오늘)',                     # 오늘
                            r'(어제)',                     # 어제
                            r'(\d{4}년\s*\d{1,2}월\s*\d{1,2}일)'  # 2024년 1월 15일
                        ]
                        
                        for pattern in date_patterns:
                            date_match = re.search(pattern, date_text)
                            if date_match:
                                review_date = date_match.group(1)
                                break
                        
                        if review_date == "날짜 정보 없음" and len(date_text) < 30:
                            review_date = date_text
                except:
                    # 대체 날짜 셀렉터들 시도
                    date_selectors = [
                        "time", 
                        "div.pui__QKE5Pr", 
                        "span[class*='date']",
                        "div[class*='date']",
                        "span.time"
                    ]
                    
                    for date_selector in date_selectors:
                        try:
                            date_element = review_element.find_element(By.CSS_SELECTOR, date_selector)
                            date_text = date_element.text.strip()
                            
                            if date_text and len(date_text) < 30:
                                review_date = date_text
                                break
                        except:
                            continue
                
                review_info['date'] = review_date
                
                # 리뷰 평점 (별점) 수집
                try:
                    rating_selectors = [
                        "div.pui__rating span",
                        "span[class*='rating']",
                        "div[class*='star']",
                        ".rating"
                    ]
                    
                    review_rating = "평점 정보 없음"
                    for rating_selector in rating_selectors:
                        try:
                            rating_element = review_element.find_element(By.CSS_SELECTOR, rating_selector)
                            rating_text = rating_element.text.strip()
                            if rating_text and ("점" in rating_text or "★" in rating_text):
                                review_rating = rating_text
                                break
                        except:
                            continue
                    
                    review_info['rating'] = review_rating
                except:
                    review_info['rating'] = "평점 정보 없음"
                
                # 리뷰 이미지들 (개별 리뷰에서) - 프로필 사진 제외
                review_images = []
                try:
                    # 개별 리뷰 내의 이미지들 (프로필 사진 제외)
                    img_elements = review_element.find_elements(By.CSS_SELECTOR, "img")
                    for j, img_element in enumerate(img_elements):
                        img_url = img_element.get_attribute("src")
                        img_alt = img_element.get_attribute("alt") or ""
                        img_class = img_element.get_attribute("class") or ""
                        
                        # 프로필 사진이 아닌 실제 리뷰 이미지만 필터링
                        if (img_url and 
                            ("review" in img_url or "ldb-phinf" in img_url or "pstatic.net" in img_url) and
                            "profile" not in img_url.lower() and
                            "avatar" not in img_url.lower() and
                            "user" not in img_alt.lower() and
                            "profile" not in img_alt.lower() and
                            "pui__q2fg8o" not in img_class and  # 프로필 사진 클래스 제외
                            "pui__A7NplK" not in img_class):   # 프로필 사진 클래스 제외
                            
                            # 안전한 파일명 생성
                            safe_name = safe_filename(restaurant_name)[:20]
                            img_filename = f"개별리뷰_{i+1:02d}_{j+1:02d}.jpg"
                            
                            downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                            if downloaded_path:
                                review_images.append(downloaded_path)
                        
                except Exception as e:
                    print(f"리뷰 {i+1} 개별 이미지 수집 실패: {e}")
                
                review_info['individual_images'] = review_images
                review_list.append(review_info)
                
                # 진행 상황 출력
                if i % 10 == 0:
                    print(f"리뷰 처리 진행률: {i+1}/{len(review_elements)}")
                
                print(f"리뷰 {i+1}: {review_info['text'][:50]}..." if len(review_info['text']) > 50 else f"리뷰 {i+1}: {review_info['text']}")
                
            except Exception as e:
                print(f"리뷰 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"리뷰 정보 수집 실패: {e}")
        
    print(f"총 {len(review_list)}개의 리뷰 수집 완료")
    return review_list

def go_back_to_list():
    """목록으로 돌아가기"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 검색 결과 iframe으로 다시 전환
        search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
        driver.switch_to.frame(search_iframe)
        print("목록으로 돌아가기 완료")
        return True
        
    except Exception as e:
        print(f"목록으로 돌아가기 실패: {e}")
        return False

# =================== 메인 실행 부분 ===================
try:
    print("=" * 60)
    print(f"네이버 지도 크롤링 시작")
    print(f"검색어: {SEARCH_KEYWORD}")
    print(f"최대 가게 수: {MAX_RESTAURANTS}")
    print(f"가게당 최대 리뷰 수: {MAX_REVIEWS_PER_RESTAURANT}")
    print("=" * 60)
    
    # 1. 검색어 설정 및 접속
    encoded_keyword = urllib.parse.quote(SEARCH_KEYWORD)
    URL = f"https://map.naver.com/p/search/{encoded_keyword}"
    
    print(f"네이버 지도 접속 중: {URL}")
    driver.get(URL)
    time.sleep(5)
    
    # 2. 디렉토리 생성
    base_dir = create_directory_structure(SEARCH_KEYWORD)
    print(f"저장 디렉토리 생성: {base_dir}")
    
    # 3. 검색 결과 iframe 접근
    print("검색 결과 iframe으로 전환 중...")
    search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
    driver.switch_to.frame(search_iframe)
    
    # 4. 가게 리스트 수집
    restaurant_elements = get_restaurant_list()
    total_restaurants = len(restaurant_elements)
    print(f"총 {total_restaurants}개 가게 발견")
    
    # 5. 각 가게별로 세부 정보 수집
    max_restaurants = min(MAX_RESTAURANTS, total_restaurants)
    
    for i in range(max_restaurants):
        print(f"\n{'='*20} 가게 {i+1}/{max_restaurants} {'='*20}")
        
        try:
            # 가게 클릭
            if not click_restaurant(restaurant_elements[i], i+1):
                continue
            
            # 세부 정보 iframe으로 전환
            if not switch_to_detail_iframe():
                continue
            
            # 홈 정보 수집
            print("홈 정보 수집 중...")
            home_info = get_home_info()
            
            # 가게별 폴더 구조 생성
            restaurant_dir, menu_images_dir, review_images_dir = create_restaurant_directories(
                base_dir, home_info.get('name', f'restaurant_{i+1}')
            )
            
            # 메뉴 정보 수집
            menu_info = []
            if find_and_click_menu_tab():
                print("메뉴 정보 수집 중...")
                menu_info = get_menu_info(menu_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("메뉴 탭을 찾을 수 없어 메뉴 정보 수집을 건너뜁니다.")
            
            # 리뷰 정보 수집
            review_info = []
            if find_and_click_review_tab():
                print("리뷰 정보 수집 중...")
                review_info = get_review_info(review_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("리뷰 탭을 찾을 수 없어 리뷰 정보 수집을 건너뜁니다.")
            
            # 전체 정보 결합
            restaurant_data = {
                'index': i + 1,
                'home_info': home_info,
                'menu_info': menu_info,
                'review_info': review_info,
                'restaurant_dir': restaurant_dir,
                'collected_at': datetime.now().isoformat()
            }
            
            all_restaurant_data.append(restaurant_data)
            print(f"가게 {i+1} 정보 수집 완료!")
            print(f"- 메뉴 개수: {len(menu_info)}")
            print(f"- 리뷰 개수: {len(review_info)}")
            
            # 목록으로 돌아가기
            go_back_to_list()
            time.sleep(2)
            
        except Exception as e:
            print(f"가게 {i+1} 처리 중 오류: {e}")
            # 목록으로 돌아가기 시도
            try:
                go_back_to_list()
            except:
                pass
            continue
    
    # 6. 결과 저장
    print(f"\n{'='*60}")
    print(f"총 {len(all_restaurant_data)}개 가게 상세 정보 수집 완료!")
    
    # JSON 파일 저장
    safe_keyword = safe_filename(SEARCH_KEYWORD)
    json_filename = os.path.join(base_dir, f"{safe_keyword}_detailed_restaurants.json")
    with open(json_filename, 'w', encoding='utf-8') as jsonfile:
        json.dump(all_restaurant_data, jsonfile, ensure_ascii=False, indent=2)
    
    # CSV 파일 저장 (한글 헤더로 개선)
    csv_filename = os.path.join(base_dir, f"{safe_keyword}_restaurants_summary.csv")
    with open(csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:  # utf-8-sig로 BOM 추가
        fieldnames = ['순번', '가게명', '분류', '주소', '전화번호', '메뉴수', '리뷰수', '폴더경로']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            row = {
                '순번': restaurant['index'],
                '가게명': restaurant['home_info'].get('name', ''),
                '분류': restaurant['home_info'].get('category', ''),
                '주소': restaurant['home_info'].get('address', ''),
                '전화번호': restaurant['home_info'].get('phone', ''),
                '메뉴수': len(restaurant['menu_info']),
                '리뷰수': len(restaurant['review_info']),
                '폴더경로': restaurant.get('restaurant_dir', '')
            }
            writer.writerow(row)
    
    # 리뷰 상세 정보 CSV 저장
    reviews_csv_filename = os.path.join(base_dir, f"{safe_keyword}_reviews_detail.csv")
    with open(reviews_csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['가게명', '리뷰번호', '리뷰내용', '작성날짜', '평점', '이미지수']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            restaurant_name = restaurant['home_info'].get('name', '')
            for idx, review in enumerate(restaurant['review_info']):
                row = {
                    '가게명': restaurant_name,
                    '리뷰번호': idx + 1,
                    '리뷰내용': review.get('text', ''),
                    '작성날짜': review.get('date', ''),
                    '평점': review.get('rating', ''),
                    '이미지수': len(review.get('individual_images', []))
                }
                writer.writerow(row)
    
    # 통계 정보 출력
    total_menus = sum(len(restaurant['menu_info']) for restaurant in all_restaurant_data)
    total_reviews = sum(len(restaurant['review_info']) for restaurant in all_restaurant_data)
    total_images = 0
    
    # 이미지 파일 개수 세기
    for restaurant in all_restaurant_data:
        restaurant_dir = restaurant.get('restaurant_dir', '')
        if os.path.exists(restaurant_dir):
            for root, dirs, files in os.walk(restaurant_dir):
                image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
                total_images += len(image_files)
    
    print(f"\n파일 저장 완료:")
    print(f"- 상세 정보 (JSON): {json_filename}")
    print(f"- 요약 정보 (CSV): {csv_filename}")
    print(f"- 리뷰 상세 (CSV): {reviews_csv_filename}")
    print(f"- 이미지 폴더: 각 가게별 폴더")
    print(f"\n수집 통계:")
    print(f"- 총 가게 수: {len(all_restaurant_data)}")
    print(f"- 총 메뉴 수: {total_menus}")
    print(f"- 총 리뷰 수: {total_reviews}")
    print(f"- 총 이미지 수: {total_images}")
    
    print(f"\n{'='*60}")
    print("크롤링 완료!")

except Exception as e:
    print(f"전체 프로세스 오류 발생: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    # 드라이버 종료
    try:
        pass
#         driver.quit()
#         print("웹드라이버 종료 완료")
    except:
        pass
end_time = time.time()

print(f"실행 시간: {end_time - start_time:.2f}초")

네이버 지도 크롤링 시작
검색어: 신림동 햄버거
최대 가게 수: 5
가게당 최대 리뷰 수: 500
네이버 지도 접속 중: https://map.naver.com/p/search/%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0
저장 디렉토리 생성: 신림동 햄버거_test2_sample2_data
검색 결과 iframe으로 전환 중...
가게 리스트 수집 중...
스크롤 컨테이너 찾음
현재 로드된 가게 수: 10
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
더 이상 로드할 가게가 없음
총 50개 가게 발견

==================== 가게 1/5 ====================
가게 1 클릭 완료
세부 정보 iframe으로 전환 완료
홈 정보 수집 중...
가게명: 아토커피 신림점
분류: 카페,디저트
주소: 서울 관악구 관천로 79 1층
메뉴 탭 찾는 중...
탭 요소들 찾음: 6개
탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1539251371/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081555&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 2: 텍스트='소식', href='https://pcmap.place.naver.com/restaurant/1539251371/feed?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081555&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 3: 텍스트='메뉴', href='https://pcm

  이미지 다운로드 성공: 리뷰이미지_04_02_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 2에서 수집된 이미지: 1개
카메라 컨테이너 5에서 리뷰 이미지 컨테이너 발견: 1개
리뷰 이미지 컨테이너 1 처리 중...
  이미지 다운로드 성공: 리뷰이미지_05_01_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 1에서 수집된 이미지: 1개
카메라 컨테이너 6에서 리뷰 이미지 컨테이너 발견: 1개
리뷰 이미지 컨테이너 1 처리 중...
  이미지 다운로드 성공: 리뷰이미지_06_01_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 1에서 수집된 이미지: 1개
카메라 컨테이너 7에서 리뷰 이미지 컨테이너 발견: 4개
리뷰 이미지 컨테이너 1 처리 중...
  이미지 다운로드 성공: 리뷰이미지_07_01_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 1에서 수집된 이미지: 1개
리뷰 이미지 컨테이너 2 처리 중...
  이미지 다운로드 성공: 리뷰이미지_07_02_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 2에서 수집된 이미지: 1개
리뷰 이미지 컨테이너 3 처리 중...
  이미지 다운로드 성공: 리뷰이미지_07_03_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 3에서 수집된 이미지: 1개
리뷰 이미지 컨테이너 4 처리 중...
  이미지 다운로드 성공: 리뷰이미지_07_04_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 4에서 수집된 이미지: 1개
카메라 컨테이너 8에서 리뷰 이미지 컨테이너 발견: 2개
리뷰 이미지 컨테이너 1 처리 중...
  이미지 다운로드 성공: 리뷰이미지_08_01_01_01.jpg
  슬라이드 버튼을 찾을 수 없음 (슬라이드 1)
컨테이너 1에서 수집된 이미지: 1개
리뷰 이미지 컨테이너 2 처리 중...
  이미지 다운

  이미지 다운로드 성공: 개별리뷰_47_02.jpg
  이미지 다운로드 성공: 개별리뷰_47_03.jpg
리뷰 47: 화장실도 내부에 있어서 24시간 카페 이용할 때 편리할거 같아요 :)
요즘 24시 카페들이...
  이미지 다운로드 성공: 개별리뷰_48_02.jpg
리뷰 48: 수박주스랑 오렌지밤 먹었는데 수박주스는 정말 진짜 온리 수박을 갈아넣은 맛이예요! 무조건 ...
  이미지 다운로드 성공: 개별리뷰_49_02.jpg
리뷰 49: 저는 딸기라떼 먹고 동생은 프로틴주스 먹었는데 진짜 너무 맛있었어요 딸기라떼는 진짜 생딸기...
  이미지 다운로드 성공: 개별리뷰_50_02.jpg
리뷰 50: 음료도 맛있고 매장이 힙해서 데이트하기 좋았어요!!! 굿굿~~~!
  이미지 다운로드 성공: 개별리뷰_51_01.jpg
  이미지 다운로드 성공: 개별리뷰_51_02.jpg
  이미지 다운로드 성공: 개별리뷰_51_03.jpg
  이미지 다운로드 성공: 개별리뷰_51_04.jpg
  이미지 다운로드 성공: 개별리뷰_51_05.jpg
  이미지 다운로드 성공: 개별리뷰_51_06.jpg
리뷰 처리 진행률: 51/110
리뷰 51: 신림역에 24시간 카페가 있어서 왔습니다!
디저트 메뉴도 햄버거, 와플, 떡볶이 등등
매우...
  이미지 다운로드 성공: 개별리뷰_52_01.jpg
  이미지 다운로드 성공: 개별리뷰_52_02.jpg
리뷰 52: 집 근처 24시간 카페라니 행복하다
단백질 음료가 있어서 초코맛 먹음
다음엔 커피맛을 알아...
  이미지 다운로드 성공: 개별리뷰_53_01.jpg
  이미지 다운로드 성공: 개별리뷰_53_02.jpg
  이미지 다운로드 성공: 개별리뷰_53_03.jpg
리뷰 53: ♥︎
  이미지 다운로드 성공: 개별리뷰_54_01.jpg
  이미지 다운로드 성공: 개별리뷰_54_02.jpg
  이미지 다운로드 성공: 개별리뷰_54_03.jpg
리뷰 54: ❤️
  이미지 다운로드 성공: 개별리뷰_55_01.jpg
  이미지 다운로드 성공

KeyboardInterrupt: 

In [14]:
## 또 다시.. 리뷰 이미지에서 이상한 로고들이 들어와서 다시 정리함

In [15]:
from time import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
import requests
import time
import os
import json
import csv
import urllib.parse
import re
from datetime import datetime
from PIL import Image
import io

start = time.time()
# =================== 설정 구역 ===================
# 여기서 검색어와 기타 설정을 변경하세요
SEARCH_KEYWORD = "신림동 햄버거"  # 원하는 검색어로 변경
MAX_RESTAURANTS = 5  # 크롤링할 최대 가게 수
MAX_REVIEWS_PER_RESTAURANT = 50  # 가게당 최대 리뷰 수
# ================================================

# 서비스 설정
service = Service(port=9999)
# 크롬 인터넷 설정
options = webdriver.ChromeOptions()
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36')
options.add_argument('window-size=1380,900')
# 드라이버 생성
driver = webdriver.Chrome(service=service, options=options)
# WebDriverWait 초기화
wait = WebDriverWait(driver, 10)

# 결과 저장할 리스트
all_restaurant_data = []

def create_directory_structure(keyword):
    """가게별 이미지 저장용 디렉토리 구조 생성"""
    base_dir = f"{keyword}_test2_sample2_data"
    os.makedirs(base_dir, exist_ok=True)
    return base_dir

def create_restaurant_directories(base_dir, restaurant_name):
    """가게별 폴더 구조 생성"""
    safe_name = safe_filename(restaurant_name)
    restaurant_dir = os.path.join(base_dir, safe_name)
    menu_images_dir = os.path.join(restaurant_dir, "메뉴_이미지")
    review_images_dir = os.path.join(restaurant_dir, "리뷰_이미지")
    
    os.makedirs(menu_images_dir, exist_ok=True)
    os.makedirs(review_images_dir, exist_ok=True)
    
    return restaurant_dir, menu_images_dir, review_images_dir

def download_image_improved(url, filename, images_dir):
    """이미지 다운로드 (개선된 버전 - 이미지 검증 포함)"""
    try:
        # User-Agent 헤더 추가
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
            'Referer': 'https://map.naver.com/'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            # 이미지 데이터 검증
            try:
                # PIL로 이미지 검증
                img = Image.open(io.BytesIO(response.content))
                img.verify()  # 이미지가 유효한지 검증
                
                # 파일 저장
                filepath = os.path.join(images_dir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                print(f"  이미지 다운로드 성공: {filename}")
                return filepath
            except Exception as img_error:
                print(f"  이미지 검증 실패 ({filename}): {img_error}")
                return None
        else:
            print(f"  HTTP 오류 ({filename}): {response.status_code}")
            return None
    except Exception as e:
        print(f"  이미지 다운로드 실패 ({filename}): {e}")
        return None

def safe_filename(filename):
    """파일명에서 특수문자 제거"""
    return re.sub(r'[^\w\s-]', '', filename).strip()[:50]

def get_restaurant_list():
    """왼쪽 패널에서 가게 리스트 수집"""
    print("가게 리스트 수집 중...")
    
    # 스크롤하여 모든 가게 로드
    try:
        scroll_container = driver.find_element(By.CSS_SELECTOR, "div.Ryr1F")
        print("스크롤 컨테이너 찾음")
        
        previous_count = 0
        max_attempts = 15
        
        for attempt in range(max_attempts):
            restaurant_elements = driver.find_elements(By.CSS_SELECTOR, "li.UEzoS")
            current_count = len(restaurant_elements)
            
            print(f"현재 로드된 가게 수: {current_count}")
            
            if current_count == previous_count:
                if attempt >= 3:
                    print("더 이상 로드할 가게가 없음")
                    break
            else:
                previous_count = current_count
            
            # 스크롤 실행
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
            time.sleep(2)
        
        return restaurant_elements
        
    except Exception as e:
        print(f"가게 리스트 수집 실패: {e}")
        return []

def click_restaurant(restaurant_element, index):
    """가게 클릭하여 세부 정보 패널 열기"""
    try:
        # 가게 링크 찾기
        link_element = restaurant_element.find_element(By.CSS_SELECTOR, "a.place_bluelink")
        
        # 스크롤하여 요소가 보이도록 함
        driver.execute_script("arguments[0].scrollIntoView(true);", link_element)
        time.sleep(1)
        
        # 클릭
        ActionChains(driver).move_to_element(link_element).click().perform()
        print(f"가게 {index} 클릭 완료")
        
        # 세부 정보 패널 로딩 대기
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"가게 {index} 클릭 실패: {e}")
        return False

def switch_to_detail_iframe():
    """오른쪽 세부 정보 iframe으로 전환"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 세부 정보 iframe 찾기
        detail_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#entryIframe")))
        driver.switch_to.frame(detail_iframe)
        print("세부 정보 iframe으로 전환 완료")
        return True
        
    except Exception as e:
        print(f"세부 정보 iframe 전환 실패: {e}")
        return False

def get_home_info():
    """홈 탭에서 기본 정보 수집"""
    home_info = {}
    
    try:
        # 가게 이름
        name_element = driver.find_element(By.CSS_SELECTOR, "span.GHAhO")
        home_info['name'] = name_element.text.strip()
        print(f"가게명: {home_info['name']}")
        
        # 가게 분류
        category_element = driver.find_element(By.CSS_SELECTOR, "span.lnJFt")
        home_info['category'] = category_element.text.strip()
        print(f"분류: {home_info['category']}")
        
        # 가게 주소
        try:
            address_element = driver.find_element(By.CSS_SELECTOR, "span.LDgIH")
            home_info['address'] = address_element.text.strip()
            print(f"주소: {home_info['address']}")
        except:
            home_info['address'] = "주소 정보 없음"
        
        # 전화번호 (있다면)
        try:
            phone_element = driver.find_element(By.CSS_SELECTOR, "span.xlx7Q")
            home_info['phone'] = phone_element.text.strip()
        except:
            home_info['phone'] = "전화번호 정보 없음"
        
        # 영업시간 (있다면)
        try:
            hours_element = driver.find_element(By.CSS_SELECTOR, "time.H3ua4")
            home_info['hours'] = hours_element.text.strip()
        except:
            home_info['hours'] = "영업시간 정보 없음"
            
    except Exception as e:
        print(f"홈 정보 수집 실패: {e}")
        
    return home_info

def find_and_click_menu_tab():
    """메뉴 탭을 정확하게 찾아서 클릭"""
    try:
        print("메뉴 탭 찾는 중...")
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 메뉴 탭 찾기
        menu_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 메뉴 탭인지 판단
                is_menu_tab = (
                    "menu" in tab_href.lower() or
                    "메뉴" in tab_text or
                    "menu" in tab_text
                )
                
                if is_menu_tab:
                    menu_tab = tab
                    print(f"메뉴 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 메뉴 탭 클릭
        if menu_tab:
            try:
                # 스크롤하여 요소가 보이도록 함
                driver.execute_script("arguments[0].scrollIntoView(true);", menu_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", menu_tab)
                time.sleep(3)
                print("메뉴 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"메뉴 탭 클릭 실패: {e}")
                return False
        else:
            print("메뉴 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"메뉴 탭 찾기 실패: {e}")
        return False

def get_menu_info(menu_images_dir, restaurant_name):
    """메뉴 탭에서 메뉴 정보 수집"""
    menu_list = []
    
    try:
        # 메뉴 더보기 버튼 클릭
        try:
            more_button_selectors = [
                "a.fvwqf",
                "button.fvwqf",
                "a[class*='more']",
                "//a[contains(text(), '더보기')]"
            ]
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_button = driver.find_element(By.XPATH, selector)
                    else:
                        more_button = driver.find_element(By.CSS_SELECTOR, selector)
                    
                    if more_button.is_displayed() and more_button.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", more_button)
                        time.sleep(2)
                        print("메뉴 더보기 버튼 클릭 완료")
                        break
                except:
                    continue
        except:
            print("메뉴 더보기 버튼이 없거나 이미 모든 메뉴가 표시됨")
        
        # 메뉴 항목들 찾기
        menu_elements = driver.find_elements(By.CSS_SELECTOR, "li.E2jtL")
        
        for i, menu_element in enumerate(menu_elements):
            menu_info = {}
            
            try:
                # 메뉴명
                name_element = menu_element.find_element(By.CSS_SELECTOR, "span.lPzHi")
                menu_info['name'] = name_element.text.strip()
                
                # 메뉴 가격
                try:
                    price_element = menu_element.find_element(By.CSS_SELECTOR, "div.GXS1X em")
                    menu_info['price'] = price_element.text.strip()
                except:
                    menu_info['price'] = "가격 정보 없음"
                
                # 메뉴 설명
                try:
                    desc_element = menu_element.find_element(By.CSS_SELECTOR, "div.TRxGt")
                    menu_info['description'] = desc_element.text.strip()
                except:
                    menu_info['description'] = "설명 없음"
                
                # 메뉴 이미지
                try:
                    img_selectors = ["img.K0PDV", "img"]
                    img_element = None
                    
                    for selector in img_selectors:
                        try:
                            img_element = menu_element.find_element(By.CSS_SELECTOR, selector)
                            break
                        except:
                            continue
                    
                    if img_element:
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            # 이미지 파일명 생성 (특수문자 제거)
                            safe_menu_name = safe_filename(menu_info['name'])[:20]
                            img_filename = f"메뉴_{i+1:02d}_{safe_menu_name}.jpg"
                            
                            # 이미지 다운로드
                            downloaded_path = download_image_improved(img_url, img_filename, menu_images_dir)
                            menu_info['image_path'] = downloaded_path
                        else:
                            menu_info['image_path'] = None
                    else:
                        menu_info['image_path'] = None
                except:
                    menu_info['image_path'] = None
                
                menu_list.append(menu_info)
                print(f"메뉴 {i+1}: {menu_info['name']} - {menu_info['price']}")
                
            except Exception as e:
                print(f"메뉴 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"메뉴 정보 수집 실패: {e}")
    
    return menu_list

def find_and_click_review_tab():
    """리뷰 탭을 정확하게 찾아서 클릭"""
    try:
        print("리뷰 탭 찾는 중...")
        
        # 페이지 스크롤하여 탭이 보이도록 함
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 리뷰 탭 찾기
        review_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 리뷰 탭인지 판단
                is_review_tab = (
                    "review" in tab_href.lower() or
                    "리뷰" in tab_text or
                    "review" in tab_text
                )
                
                if is_review_tab:
                    review_tab = tab
                    print(f"리뷰 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 리뷰 탭 클릭
        if review_tab:
            try:
                # 요소가 보이도록 스크롤
                driver.execute_script("arguments[0].scrollIntoView(true);", review_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", review_tab)
                time.sleep(3)
                print("리뷰 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"리뷰 탭 클릭 실패: {e}")
                return False
        else:
            print("리뷰 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"리뷰 탭 찾기 실패: {e}")
        return False

def click_all_review_more_buttons():
    """모든 리뷰 더보기 버튼 클릭 (개선된 버전)"""
    try:
        print("리뷰 더보기 버튼들 찾는 중...")
        
        # 지정된 셀렉터로 더보기 버튼 찾기
        more_button_selectors = [
            "div.lfH3O.fvwqf",  # 사용자가 제공한 정확한 셀렉터
            "div.fvwqf",
            "a.fvwqf",
            "button.fvwqf",
            "//a[contains(text(), '더보기')]",
            "//button[contains(text(), '더보기')]",
            "//div[contains(text(), '더보기')]"
        ]
        
        clicked_count = 0
        max_attempts = 10  # 최대 10번 시도
        
        for attempt in range(max_attempts):
            print(f"더보기 버튼 찾기 시도 {attempt + 1}/{max_attempts}")
            
            button_found = False
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_buttons = driver.find_elements(By.XPATH, selector)
                    else:
                        more_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    
                    for button in more_buttons:
                        if button.is_displayed() and button.is_enabled():
                            try:
                                # 버튼이 보이도록 스크롤
                                driver.execute_script("arguments[0].scrollIntoView(true);", button)
                                time.sleep(1)
                                
                                # 버튼 텍스트 확인
                                button_text = button.text.strip()
                                print(f"발견된 버튼 텍스트: '{button_text}'")
                                
                                if "더보기" in button_text or "more" in button_text.lower():
                                    # JavaScript로 클릭
                                    driver.execute_script("arguments[0].click();", button)
                                    time.sleep(3)  # 로딩 대기
                                    clicked_count += 1
                                    button_found = True
                                    print(f"더보기 버튼 클릭 완료 ({clicked_count}번째)")
                                    break
                            except Exception as click_error:
                                print(f"버튼 클릭 실패: {click_error}")
                                continue
                    
                    if button_found:
                        break
                        
                except Exception as e:
                    continue
            
            if not button_found:
                print("더 이상 더보기 버튼을 찾을 수 없음")
                break
                
            # 페이지 하단으로 스크롤하여 새로운 리뷰 로드 확인
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        print(f"총 {clicked_count}개의 더보기 버튼을 클릭했습니다")
        return clicked_count > 0
        
    except Exception as e:
        print(f"리뷰 더보기 버튼 처리 실패: {e}")
        return False

def collect_review_images_with_slide(restaurant_name, review_images_dir):
    """리뷰 이미지 수집 (슬라이드 포함) - alt="방문자리뷰사진" 우선 사용"""
    downloaded_images = []
    
    try:
        print("리뷰 이미지 수집 시작...")
        
        # 1단계: alt="방문자리뷰사진"인 이미지들을 우선 수집
        visitor_review_images = driver.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
        print(f"방문자리뷰사진 alt 속성을 가진 이미지 발견: {len(visitor_review_images)}개")
        
        total_images = 0
        
        # 방문자리뷰사진 alt 속성을 가진 이미지들 우선 수집
        for img_idx, img in enumerate(visitor_review_images):
            try:
                img_url = img.get_attribute("src")
                
                if img_url and 'http' in img_url:
                    filename = f"방문자리뷰_{img_idx+1:03d}.jpg"
                    
                    # 이미지가 이미 다운로드되었는지 확인 (URL 기준)
                    if img_url not in [img_info.get('url') for img_info in downloaded_images]:
                        downloaded_path = download_image_improved(img_url, filename, review_images_dir)
                        if downloaded_path:
                            downloaded_images.append({
                                'path': downloaded_path,
                                'url': img_url,
                                'filename': filename,
                                'type': 'visitor_review'
                            })
                            total_images += 1
                            print(f"  방문자리뷰사진 다운로드: {filename}")
            
            except Exception as e:
                print(f"  방문자리뷰사진 {img_idx+1} 처리 실패: {e}")
                continue
        
        # 2단계: 기존 방식으로 추가 이미지 수집 (백업용)
        print("추가 리뷰 이미지 수집 중...")
        
        # 정확한 리뷰 이미지 컨테이너 찾기 (프로필 사진 제외)
        camera_containers = driver.find_elements(By.CSS_SELECTOR, "div.flicking-camera")
        
        for camera_idx, camera_container in enumerate(camera_containers):
            try:
                # 각 카메라 컨테이너 안의 HH5sZ 요소들 찾기
                review_containers = camera_container.find_elements(By.CSS_SELECTOR, "div.HH5sZ")
                
                for container_idx, container in enumerate(review_containers):
                    try:
                        # 컨테이너가 보이도록 스크롤
                        driver.execute_script("arguments[0].scrollIntoView(true);", container)
                        time.sleep(1)
                        
                        # 현재 컨테이너의 이미지들 수집
                        max_slides = 5  # 슬라이드 횟수 줄임 (방문자리뷰사진으로 대부분 커버되므로)
                        
                        for slide_count in range(max_slides):
                            # 현재 보이는 이미지들 찾기
                            current_images = container.find_elements(By.CSS_SELECTOR, "img")
                            
                            for img_idx, img in enumerate(current_images):
                                try:
                                    img_url = img.get_attribute("src")
                                    img_alt = img.get_attribute("alt") or ""
                                    
                                    # 방문자리뷰사진이 아닌 경우에만 추가 필터링 적용
                                    if (img_alt == "방문자리뷰사진" or
                                        (img_url and 'http' in img_url and 
                                         ('pstatic.net' in img_url or 'blogfiles' in img_url) and
                                         'profile' not in img_url.lower() and
                                         'avatar' not in img_url.lower() and
                                         'user' not in img_alt.lower() and
                                         'profile' not in img_alt.lower())):
                                        
                                        filename = f"추가리뷰_{camera_idx+1:02d}_{container_idx+1:02d}_{slide_count+1:02d}_{img_idx+1:02d}.jpg"
                                        
                                        # 이미지가 이미 다운로드되었는지 확인 (URL 기준)
                                        if img_url not in [img_info.get('url') for img_info in downloaded_images]:
                                            downloaded_path = download_image_improved(img_url, filename, review_images_dir)
                                            if downloaded_path:
                                                downloaded_images.append({
                                                    'path': downloaded_path,
                                                    'url': img_url,
                                                    'filename': filename,
                                                    'type': 'additional'
                                                })
                                                total_images += 1
                                
                                except Exception as e:
                                    continue
                            
                            # 다음 슬라이드로 이동 시도
                            try:
                                next_button = container.find_element(By.CSS_SELECTOR, "span.nK_aH, button[class*='next'], a[class*='next']")
                                if next_button.is_displayed() and next_button.is_enabled():
                                    driver.execute_script("arguments[0].click();", next_button)
                                    time.sleep(1)
                                else:
                                    break
                            except:
                                break
                        
                    except Exception as e:
                        continue
                        
            except Exception as e:
                continue
        
        print(f"리뷰 이미지 다운로드 완료: 총 {total_images}개")
        print(f"  - 방문자리뷰사진: {len([img for img in downloaded_images if img.get('type') == 'visitor_review'])}개")
        print(f"  - 추가 이미지: {len([img for img in downloaded_images if img.get('type') == 'additional'])}개")
        
        return [img['path'] for img in downloaded_images]
        
    except Exception as e:
        print(f"리뷰 이미지 수집 실패: {e}")
        return []

def get_review_info(review_images_dir, restaurant_name, max_reviews=None):
    """리뷰 탭에서 리뷰 정보 수집 (개선된 버전)"""
    if max_reviews is None:
        max_reviews = MAX_REVIEWS_PER_RESTAURANT
    
    review_list = []
    
    try:
        # 모든 리뷰 더보기 버튼 클릭
        click_all_review_more_buttons()
        
        # 리뷰 이미지 수집 (슬라이드 포함)
        review_slide_images = collect_review_images_with_slide(restaurant_name, review_images_dir)
        
        # 최종 스크롤하여 모든 리뷰 로드
        print("최종 스크롤로 모든 리뷰 로드 중...")
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # 리뷰 항목들 찾기 (여러 셀렉터 시도)
        review_selectors = [
            "li.place_apply_pui.EjjAW",
            "li.place_apply_pui",
            "div.pui__vn15t2",
            "li[class*='review']",
            "div[class*='review']"
        ]
        
        review_elements = []
        for selector in review_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    review_elements = elements[:max_reviews]
                    print(f"리뷰 요소 찾음: {selector} - {len(review_elements)}개")
                    break
            except:
                continue
        
        if not review_elements:
            print("리뷰 요소를 찾을 수 없습니다.")
            return review_list
        
        print(f"총 {len(review_elements)}개의 리뷰 처리 시작")
        
        for i, review_element in enumerate(review_elements):
            review_info = {}
            
            try:
                # 리뷰 텍스트 - 정확한 셀렉터 사용
                review_text = "리뷰 텍스트 없음"
                try:
                    # div.pui__vn15t2에서 리뷰 텍스트 찾기
                    text_element = review_element.find_element(By.CSS_SELECTOR, "div.pui__vn15t2")
                    review_text = text_element.text.strip()
                    if not review_text or len(review_text) < 5:
                        # 다른 텍스트 셀렉터들도 시도
                        text_selectors = [
                            "span.zPfVt", 
                            "div.pui__vn15t2 span",
                            "span[class*='review']", 
                            "div[class*='content']"
                        ]
                        
                        for text_selector in text_selectors:
                            try:
                                text_elements = review_element.find_elements(By.CSS_SELECTOR, text_selector)
                                for text_elem in text_elements:
                                    text_content = text_elem.text.strip()
                                    if text_content and len(text_content) > 10:
                                        review_text = text_content
                                        break
                                if review_text != "리뷰 텍스트 없음":
                                    break
                            except:
                                continue
                except:
                    # 여전히 텍스트를 못찾았다면 전체 텍스트에서 추출 시도
                    try:
                        full_text = review_element.text.strip()
                        if len(full_text) > 20:
                            sentences = full_text.split('\n')
                            for sentence in sentences:
                                if len(sentence.strip()) > 10:
                                    review_text = sentence.strip()
                                    break
                    except:
                        pass
                
                review_info['text'] = review_text
                
                # 리뷰 날짜 - 정확한 셀렉터 사용
                review_date = "날짜 정보 없음"
                try:
                    # span.pui__gfuUIT 안의 span.pui__blind에서 날짜 찾기
                    date_container = review_element.find_element(By.CSS_SELECTOR, "span.pui__gfuUIT")
                    date_element = date_container.find_element(By.CSS_SELECTOR, "span.pui__blind")
                    date_text = date_element.text.strip()
                    
                    if date_text:
                        # 날짜 패턴 추출
                        date_patterns = [
                            r'(\d{4}\.\d{1,2}\.\d{1,2})',  # 2024.01.15
                            r'(\d{4}-\d{1,2}-\d{1,2})',   # 2024-01-15
                            r'(\d{1,2}\.\d{1,2})',        # 01.15
                            r'(\d+일전)',                  # 3일전
                            r'(\d+주전)',                  # 2주전
                            r'(\d+개월전)',                # 1개월전
                            r'(\d+년전)',                  # 1년전
                            r'(오늘)',                     # 오늘
                            r'(어제)',                     # 어제
                            r'(\d{4}년\s*\d{1,2}월\s*\d{1,2}일)'  # 2024년 1월 15일
                        ]
                        
                        for pattern in date_patterns:
                            date_match = re.search(pattern, date_text)
                            if date_match:
                                review_date = date_match.group(1)
                                break
                        
                        if review_date == "날짜 정보 없음" and len(date_text) < 30:
                            review_date = date_text
                except:
                    # 대체 날짜 셀렉터들 시도
                    date_selectors = [
                        "time", 
                        "div.pui__QKE5Pr", 
                        "span[class*='date']",
                        "div[class*='date']",
                        "span.time"
                    ]
                    
                    for date_selector in date_selectors:
                        try:
                            date_element = review_element.find_element(By.CSS_SELECTOR, date_selector)
                            date_text = date_element.text.strip()
                            
                            if date_text and len(date_text) < 30:
                                review_date = date_text
                                break
                        except:
                            continue
                
                review_info['date'] = review_date
                
                # 리뷰 평점 (별점) 수집
                try:
                    rating_selectors = [
                        "div.pui__rating span",
                        "span[class*='rating']",
                        "div[class*='star']",
                        ".rating"
                    ]
                    
                    review_rating = "평점 정보 없음"
                    for rating_selector in rating_selectors:
                        try:
                            rating_element = review_element.find_element(By.CSS_SELECTOR, rating_selector)
                            rating_text = rating_element.text.strip()
                            if rating_text and ("점" in rating_text or "★" in rating_text):
                                review_rating = rating_text
                                break
                        except:
                            continue
                    
                    review_info['rating'] = review_rating
                except:
                    review_info['rating'] = "평점 정보 없음"
                
                # 리뷰 이미지들 (개별 리뷰에서) - alt="방문자리뷰사진" 우선 사용
                review_images = []
                try:
                    # 1. 우선적으로 alt="방문자리뷰사진"인 이미지만 찾기
                    visitor_images = review_element.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
                    
                    for j, img_element in enumerate(visitor_images):
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            img_filename = f"개별방문자리뷰_{i+1:02d}_{j+1:02d}.jpg"
                            downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                            if downloaded_path:
                                review_images.append(downloaded_path)
                    
                    # 2. 방문자리뷰사진이 없는 경우에만 다른 이미지들 시도 (백업용)
                    if not visitor_images:
                        img_elements = review_element.find_elements(By.CSS_SELECTOR, "img")
                        for j, img_element in enumerate(img_elements):
                            img_url = img_element.get_attribute("src")
                            img_alt = img_element.get_attribute("alt") or ""
                            img_class = img_element.get_attribute("class") or ""
                            
                            # 프로필 사진이 아닌 실제 리뷰 이미지만 필터링 (백업용)
                            if (img_url and 
                                ("review" in img_url or "ldb-phinf" in img_url or "pstatic.net" in img_url) and
                                "profile" not in img_url.lower() and
                                "avatar" not in img_url.lower() and
                                "user" not in img_alt.lower() and
                                "profile" not in img_alt.lower() and
                                "pui__q2fg8o" not in img_class and
                                "pui__A7NplK" not in img_class):
                                
                                img_filename = f"개별리뷰_{i+1:02d}_{j+1:02d}.jpg"
                                downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                                if downloaded_path:
                                    review_images.append(downloaded_path)
                        
                except Exception as e:
                    print(f"리뷰 {i+1} 개별 이미지 수집 실패: {e}")
                
                review_info['individual_images'] = review_images
                review_list.append(review_info)
                
                # 진행 상황 출력
                if i % 10 == 0:
                    print(f"리뷰 처리 진행률: {i+1}/{len(review_elements)}")
                
                print(f"리뷰 {i+1}: {review_info['text'][:50]}..." if len(review_info['text']) > 50 else f"리뷰 {i+1}: {review_info['text']}")
                
            except Exception as e:
                print(f"리뷰 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"리뷰 정보 수집 실패: {e}")
        
    print(f"총 {len(review_list)}개의 리뷰 수집 완료")
    return review_list

def go_back_to_list():
    """목록으로 돌아가기"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 검색 결과 iframe으로 다시 전환
        search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
        driver.switch_to.frame(search_iframe)
        print("목록으로 돌아가기 완료")
        return True
        
    except Exception as e:
        print(f"목록으로 돌아가기 실패: {e}")
        return False

# =================== 메인 실행 부분 ===================
try:
    print("=" * 60)
    print(f"네이버 지도 크롤링 시작")
    print(f"검색어: {SEARCH_KEYWORD}")
    print(f"최대 가게 수: {MAX_RESTAURANTS}")
    print(f"가게당 최대 리뷰 수: {MAX_REVIEWS_PER_RESTAURANT}")
    print("=" * 60)
    
    # 1. 검색어 설정 및 접속
    encoded_keyword = urllib.parse.quote(SEARCH_KEYWORD)
    URL = f"https://map.naver.com/p/search/{encoded_keyword}"
    
    print(f"네이버 지도 접속 중: {URL}")
    driver.get(URL)
    time.sleep(5)
    
    # 2. 디렉토리 생성
    base_dir = create_directory_structure(SEARCH_KEYWORD)
    print(f"저장 디렉토리 생성: {base_dir}")
    
    # 3. 검색 결과 iframe 접근
    print("검색 결과 iframe으로 전환 중...")
    search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
    driver.switch_to.frame(search_iframe)
    
    # 4. 가게 리스트 수집
    restaurant_elements = get_restaurant_list()
    total_restaurants = len(restaurant_elements)
    print(f"총 {total_restaurants}개 가게 발견")
    
    # 5. 각 가게별로 세부 정보 수집
    max_restaurants = min(MAX_RESTAURANTS, total_restaurants)
    
    for i in range(max_restaurants):
        print(f"\n{'='*20} 가게 {i+1}/{max_restaurants} {'='*20}")
        
        try:
            # 가게 클릭
            if not click_restaurant(restaurant_elements[i], i+1):
                continue
            
            # 세부 정보 iframe으로 전환
            if not switch_to_detail_iframe():
                continue
            
            # 홈 정보 수집
            print("홈 정보 수집 중...")
            home_info = get_home_info()
            
            # 가게별 폴더 구조 생성
            restaurant_dir, menu_images_dir, review_images_dir = create_restaurant_directories(
                base_dir, home_info.get('name', f'restaurant_{i+1}')
            )
            
            # 메뉴 정보 수집
            menu_info = []
            if find_and_click_menu_tab():
                print("메뉴 정보 수집 중...")
                menu_info = get_menu_info(menu_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("메뉴 탭을 찾을 수 없어 메뉴 정보 수집을 건너뜁니다.")
            
            # 리뷰 정보 수집
            review_info = []
            if find_and_click_review_tab():
                print("리뷰 정보 수집 중...")
                review_info = get_review_info(review_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("리뷰 탭을 찾을 수 없어 리뷰 정보 수집을 건너뜁니다.")
            
            # 전체 정보 결합
            restaurant_data = {
                'index': i + 1,
                'home_info': home_info,
                'menu_info': menu_info,
                'review_info': review_info,
                'restaurant_dir': restaurant_dir,
                'collected_at': datetime.now().isoformat()
            }
            
            all_restaurant_data.append(restaurant_data)
            print(f"가게 {i+1} 정보 수집 완료!")
            print(f"- 메뉴 개수: {len(menu_info)}")
            print(f"- 리뷰 개수: {len(review_info)}")
            
            # 목록으로 돌아가기
            go_back_to_list()
            time.sleep(2)
            
        except Exception as e:
            print(f"가게 {i+1} 처리 중 오류: {e}")
            # 목록으로 돌아가기 시도
            try:
                go_back_to_list()
            except:
                pass
            continue
    
    # 6. 결과 저장
    print(f"\n{'='*60}")
    print(f"총 {len(all_restaurant_data)}개 가게 상세 정보 수집 완료!")
    
    # JSON 파일 저장
    safe_keyword = safe_filename(SEARCH_KEYWORD)
    json_filename = os.path.join(base_dir, f"{safe_keyword}_detailed_restaurants.json")
    with open(json_filename, 'w', encoding='utf-8') as jsonfile:
        json.dump(all_restaurant_data, jsonfile, ensure_ascii=False, indent=2)
    
    # CSV 파일 저장 (한글 헤더로 개선)
    csv_filename = os.path.join(base_dir, f"{safe_keyword}_restaurants_summary.csv")
    with open(csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:  # utf-8-sig로 BOM 추가
        fieldnames = ['순번', '가게명', '분류', '주소', '전화번호', '메뉴수', '리뷰수', '폴더경로']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            row = {
                '순번': restaurant['index'],
                '가게명': restaurant['home_info'].get('name', ''),
                '분류': restaurant['home_info'].get('category', ''),
                '주소': restaurant['home_info'].get('address', ''),
                '전화번호': restaurant['home_info'].get('phone', ''),
                '메뉴수': len(restaurant['menu_info']),
                '리뷰수': len(restaurant['review_info']),
                '폴더경로': restaurant.get('restaurant_dir', '')
            }
            writer.writerow(row)
    
    # 리뷰 상세 정보 CSV 저장
    reviews_csv_filename = os.path.join(base_dir, f"{safe_keyword}_reviews_detail.csv")
    with open(reviews_csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['가게명', '리뷰번호', '리뷰내용', '작성날짜', '평점', '이미지수']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            restaurant_name = restaurant['home_info'].get('name', '')
            for idx, review in enumerate(restaurant['review_info']):
                row = {
                    '가게명': restaurant_name,
                    '리뷰번호': idx + 1,
                    '리뷰내용': review.get('text', ''),
                    '작성날짜': review.get('date', ''),
                    '평점': review.get('rating', ''),
                    '이미지수': len(review.get('individual_images', []))
                }
                writer.writerow(row)
    
    # 통계 정보 출력
    total_menus = sum(len(restaurant['menu_info']) for restaurant in all_restaurant_data)
    total_reviews = sum(len(restaurant['review_info']) for restaurant in all_restaurant_data)
    total_images = 0
    
    # 이미지 파일 개수 세기
    for restaurant in all_restaurant_data:
        restaurant_dir = restaurant.get('restaurant_dir', '')
        if os.path.exists(restaurant_dir):
            for root, dirs, files in os.walk(restaurant_dir):
                image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
                total_images += len(image_files)
    
    print(f"\n파일 저장 완료:")
    print(f"- 상세 정보 (JSON): {json_filename}")
    print(f"- 요약 정보 (CSV): {csv_filename}")
    print(f"- 리뷰 상세 (CSV): {reviews_csv_filename}")
    print(f"- 이미지 폴더: 각 가게별 폴더")
    print(f"\n수집 통계:")
    print(f"- 총 가게 수: {len(all_restaurant_data)}")
    print(f"- 총 메뉴 수: {total_menus}")
    print(f"- 총 리뷰 수: {total_reviews}")
    print(f"- 총 이미지 수: {total_images}")
    
    print(f"\n{'='*60}")
    print("크롤링 완료!")

except Exception as e:
    print(f"전체 프로세스 오류 발생: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    # 드라이버 종료
    try:
        pass
#         driver.quit()
#         print("웹드라이버 종료 완료")
    except:
        pass
    
end = time.time()
print(f"실행 시간: {end_time - start_time:.2f}초")

네이버 지도 크롤링 시작
검색어: 신림동 햄버거
최대 가게 수: 5
가게당 최대 리뷰 수: 50
네이버 지도 접속 중: https://map.naver.com/p/search/%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0
저장 디렉토리 생성: 신림동 햄버거_test2_sample2_data
검색 결과 iframe으로 전환 중...
가게 리스트 수집 중...
스크롤 컨테이너 찾음
현재 로드된 가게 수: 10
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
더 이상 로드할 가게가 없음
총 50개 가게 발견

==================== 가게 1/5 ====================
가게 1 클릭 완료
세부 정보 iframe으로 전환 완료
홈 정보 수집 중...
가게명: 아토커피 신림점
분류: 카페,디저트
주소: 서울 관악구 관천로 79 1층
메뉴 탭 찾는 중...
탭 요소들 찾음: 6개
탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1539251371/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081603&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 2: 텍스트='소식', href='https://pcmap.place.naver.com/restaurant/1539251371/feed?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081603&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 3: 텍스트='메뉴', href='https://pcma

  이미지 다운로드 성공: 방문자리뷰_013.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_013.jpg
  이미지 다운로드 성공: 방문자리뷰_014.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_014.jpg
  이미지 다운로드 성공: 방문자리뷰_015.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_015.jpg
  이미지 다운로드 성공: 방문자리뷰_016.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_016.jpg
  이미지 다운로드 성공: 방문자리뷰_017.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_017.jpg
  이미지 다운로드 성공: 방문자리뷰_018.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_018.jpg
  이미지 다운로드 성공: 방문자리뷰_019.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_019.jpg
  이미지 다운로드 성공: 방문자리뷰_020.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_020.jpg
  이미지 다운로드 성공: 방문자리뷰_021.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_021.jpg
  이미지 다운로드 성공: 방문자리뷰_022.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_022.jpg
  이미지 다운로드 성공: 방문자리뷰_023.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_023.jpg
  이미지 다운로드 성공: 방문자리뷰_024.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_024.jpg
  이미지 다운로드 성공: 방문자리뷰_025.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_025.jpg
  이미지 다운로드 성공: 방문자리뷰_026.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_026.jpg
  이미지 다운로드 성공: 방문자리뷰_027.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_027.jpg
  이미지 다운로드 성공: 방문자리뷰_028.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_028.jpg
  이미지 다운로드 성공: 방문자리뷰_029.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_029.j

메뉴 탭 클릭 완료
메뉴 정보 수집 중...
메뉴 더보기 버튼 클릭 완료
  이미지 다운로드 성공: 메뉴_01_더블 치즈버거단품.jpg
메뉴 1: 더블 치즈버거(단품) - 11,200
  이미지 다운로드 성공: 메뉴_02_해쉬 딥 치킨버거NEW단품.jpg
메뉴 2: 해쉬 딥 치킨버거[NEW](단품) - 9,500
  이미지 다운로드 성공: 메뉴_03_해쉬 딥 치킨버거NEW단품.jpg
메뉴 3: 해쉬 딥 치킨버거[NEW](단품) - 9,500
  이미지 다운로드 성공: 메뉴_04_베이컨 해쉬 치즈버거NEW단품.jpg
메뉴 4: 베이컨 해쉬 치즈버거[NEW](단품) - 9,800
  이미지 다운로드 성공: 메뉴_05_아메리칸 치즈버거단품.jpg
메뉴 5: 아메리칸 치즈버거(단품) - 7,300
  이미지 다운로드 성공: 메뉴_06_화이트마요 치킨버거단품.jpg
메뉴 6: 화이트마요 치킨버거(단품) - 6,900
  이미지 다운로드 성공: 메뉴_07_더블 치즈버거단품.jpg
메뉴 7: 더블 치즈버거(단품) - 11,200
  이미지 다운로드 성공: 메뉴_08_할라피뇨 치킨버거단품.jpg
메뉴 8: 할라피뇨 치킨버거(단품) - 8,800
  이미지 다운로드 성공: 메뉴_09_칩스갈릭 버거Signature단품.jpg
메뉴 9: 칩스&갈릭 버거[Signature](단품) - 8,800
  이미지 다운로드 성공: 메뉴_10_콘 치즈 버거Signature단품.jpg
메뉴 10: 콘 치즈 버거[Signature](단품) - 9,300
  이미지 다운로드 성공: 메뉴_11_해쉬 딥 치킨버거 세트NEW.jpg
메뉴 11: 해쉬 딥 치킨버거 세트[NEW] - 13,800
  이미지 다운로드 성공: 메뉴_12_베이컨 해쉬 치즈버거 세트NEW.jpg
메뉴 12: 베이컨 해쉬 치즈버거 세트[NEW] - 13,900
  이미지 다운로드 성공: 메뉴_13_아메리칸 치즈버거 세트.jpg
메뉴 13: 아메리칸 치즈버거 세트 - 12,300
  이미지 다운로드 성공: 메뉴_14_화이트마요 치킨버

  이미지 다운로드 성공: 개별리뷰_12_05.jpg
  이미지 다운로드 성공: 개별리뷰_12_06.jpg
리뷰 12: 신림왔다가 사먹은 버거맛집 버거락,,,!
닭고기패티랑 소고기패티 메뉴도 다양하고
양도 넘나...
  이미지 다운로드 성공: 개별리뷰_13_02.jpg
리뷰 13: 락치킨은 치킨패티가 바삭하면서도 촉촉했고, 매콤달콤한 소스가 중독성 있었어요~ 베이컨 치즈...
  이미지 다운로드 성공: 개별방문자리뷰_14_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_14_02.jpg
리뷰 14: 매장은 아닉한 분위기예요. 햄버거는 채소가 싱싱하고 패티가 가득해서 양도 충분해요
무엇보다...
  이미지 다운로드 성공: 개별방문자리뷰_15_01.jpg
리뷰 15: 신림역 버거 맛집 버거락 왔어요 ✨ 와! 진짜 재료도 신선하고, 버거락만의 버거들이 너무 ...
  이미지 다운로드 성공: 개별리뷰_16_02.jpg
리뷰 16: 아메리칸치즈버거, 치즈스틱
주문 후 7-8분 안에 메뉴 나옴

생각보다 빵은 두껍고 속이 ...
  이미지 다운로드 성공: 개별리뷰_17_02.jpg
리뷰 17: 수제버거인 만큼 가격이 좀 있는 편이지만 무친 맛입니다.. 쉑쉑 파이브가이즈 뺨싸대기 후립...
  이미지 다운로드 성공: 개별리뷰_18_02.jpg
리뷰 18: 칩스&갈릭 버거 세트를 주문했는데, 두툼한 패티와 은은한 갈릭 소스가 정말 잘 어울렸어요....
  이미지 다운로드 성공: 개별방문자리뷰_19_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_19_02.jpg
  이미지 다운로드 성공: 개별방문자리뷰_19_03.jpg
리뷰 19: 버거 맛있고 신림역과 가까워요
  이미지 다운로드 성공: 개별방문자리뷰_20_01.jpg
리뷰 20: 바로 만들어 정말 맛있는 수제 버거예요~~
모든게 잘 어우러져 맛있는 점심먹었습니다.
혼밥...
  이미지 다운로드 성공: 개별리뷰_21_02.jpg
리뷰 처리 진행률: 21/32
리뷰 21: 동생이랑 햄버거세트 두개 주문해서 먹었는데

더보기 버튼 클릭 완료 (6번째)
더보기 버튼 찾기 시도 7/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (7번째)
더보기 버튼 찾기 시도 8/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (8번째)
더보기 버튼 찾기 시도 9/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (9번째)
더보기 버튼 찾기 시도 10/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (10번째)
총 10개의 더보기 버튼을 클릭했습니다
리뷰 이미지 수집 시작...
방문자리뷰사진 alt 속성을 가진 이미지 발견: 12개
  이미지 다운로드 성공: 방문자리뷰_001.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_001.jpg
  이미지 다운로드 성공: 방문자리뷰_002.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_002.jpg
  이미지 다운로드 성공: 방문자리뷰_003.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_003.jpg
  이미지 다운로드 성공: 방문자리뷰_004.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_004.jpg
  이미지 다운로드 성공: 방문자리뷰_005.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_005.jpg
  이미지 다운로드 성공: 방문자리뷰_006.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_006.jpg
  이미지 다운로드 성공: 방문자리뷰_007.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_007.jpg
  이미지 다운로드 성공: 방문자리뷰_008.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_008.jpg
  이미지 다운로드 성공: 방문자리뷰_009.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_009.jpg
  이미지 다운로드 성공: 방문자리뷰_010.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_010.jpg
  이미지 다운로드 성공: 방문자리뷰_011.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_011.jpg
  이미지 다운로드 성공: 방문자리뷰_012.jpg
  방문자리뷰사진 다운

  이미지 다운로드 성공: 메뉴_25_달콤빠삭한케이준순살강정중.jpg
메뉴 25: [달콤빠삭한]케이준순살강정(중) - 10,000
  이미지 다운로드 성공: 메뉴_26_양파듬뿍화이트어니언강정소.jpg
메뉴 26: [양파듬뿍]화이트어니언강정(소) - 5,500
  이미지 다운로드 성공: 메뉴_27_NEW후라이드메가윙5조각.jpg
메뉴 27: (NEW)후라이드메가윙(5조각) - 9,900
  이미지 다운로드 성공: 메뉴_28_NEW자메이카메가윙5조각.jpg
메뉴 28: (NEW)자메이카메가윙(5조각) - 10,400
  이미지 다운로드 성공: 메뉴_29_신토불이후라이드 치킨반마리.jpg
메뉴 29: [신토불이]후라이드 치킨(반마리) - 10,900
  이미지 다운로드 성공: 메뉴_30_순살후라이드.jpg
메뉴 30: 순살후라이드 - 11,900
  이미지 다운로드 성공: 메뉴_31_양념치킨반마리.jpg
메뉴 31: 양념치킨(반마리) - 11,900
  이미지 다운로드 성공: 메뉴_32_다이어트프로틴틴UP에그마요휠렛버거단품.jpg
메뉴 32: [다이어트.프로틴틴UP]에그마요휠렛버거(단품) - 6,800
  이미지 다운로드 성공: 메뉴_33_불찡어버거.jpg
메뉴 33: 불찡어버거 - 7,000
  이미지 다운로드 성공: 메뉴_34_오찡어버거.jpg
메뉴 34: 오찡어버거 - 6,400
  이미지 다운로드 성공: 메뉴_35_비프해쉬버거.jpg
메뉴 35: 비프해쉬버거 - 8,000
  이미지 다운로드 성공: 메뉴_36_한입좌불닭치즈카츠버거불닭소스.jpg
메뉴 36: [한입좌]불닭치즈카츠버거(불닭소스) - 9,300
  이미지 다운로드 성공: 메뉴_37_NEW휠렛불갈비치즈베이크단품.jpg
메뉴 37: [NEW]휠렛불갈비치즈베이크(단품) - 6,400
  이미지 다운로드 성공: 메뉴_38_NEW베이컨에그치즈베이크단품.jpg
메뉴 38: [NEW]베이컨에그치즈베이크(단품) - 6,600
  이미지 다운로드 성공: 메뉴_39_맛찾사달콤비프치즈베이크단품.jpg
메뉴 39: [

KeyboardInterrupt: 

In [16]:
## 아, 진짜 다시.... 진짜 리뷰 이미지만 추출하기 

from time import time

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
import requests
import time
import os
import json
import csv
import urllib.parse
import re
from datetime import datetime
from PIL import Image
import io

start = time.time()
# =================== 설정 구역 ===================
# 여기서 검색어와 기타 설정을 변경하세요
SEARCH_KEYWORD = "신림동 햄버거"  # 원하는 검색어로 변경
MAX_RESTAURANTS = 50  # 크롤링할 최대 가게 수
MAX_REVIEWS_PER_RESTAURANT = 500  # 가게당 최대 리뷰 수
# ================================================

# 서비스 설정
service = Service(port=9999)
# 크롬 인터넷 설정
options = webdriver.ChromeOptions()
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36')
options.add_argument('window-size=1380,900')
# 드라이버 생성
driver = webdriver.Chrome(service=service, options=options)
# WebDriverWait 초기화
wait = WebDriverWait(driver, 10)

# 결과 저장할 리스트
all_restaurant_data = []

def create_directory_structure(keyword):
    """가게별 이미지 저장용 디렉토리 구조 생성"""
    base_dir = f"{keyword}_last_test!plz_data"
    os.makedirs(base_dir, exist_ok=True)
    return base_dir

def create_restaurant_directories(base_dir, restaurant_name):
    """가게별 폴더 구조 생성"""
    safe_name = safe_filename(restaurant_name)
    restaurant_dir = os.path.join(base_dir, safe_name)
    menu_images_dir = os.path.join(restaurant_dir, "메뉴_이미지")
    review_images_dir = os.path.join(restaurant_dir, "리뷰_이미지")
    
    os.makedirs(menu_images_dir, exist_ok=True)
    os.makedirs(review_images_dir, exist_ok=True)
    
    return restaurant_dir, menu_images_dir, review_images_dir

def download_image_improved(url, filename, images_dir):
    """이미지 다운로드 (개선된 버전 - 이미지 검증 포함)"""
    try:
        # User-Agent 헤더 추가
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
            'Referer': 'https://map.naver.com/'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            # 이미지 데이터 검증
            try:
                # PIL로 이미지 검증
                img = Image.open(io.BytesIO(response.content))
                img.verify()  # 이미지가 유효한지 검증
                
                # 파일 저장
                filepath = os.path.join(images_dir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                print(f"  이미지 다운로드 성공: {filename}")
                return filepath
            except Exception as img_error:
                print(f"  이미지 검증 실패 ({filename}): {img_error}")
                return None
        else:
            print(f"  HTTP 오류 ({filename}): {response.status_code}")
            return None
    except Exception as e:
        print(f"  이미지 다운로드 실패 ({filename}): {e}")
        return None

def safe_filename(filename):
    """파일명에서 특수문자 제거"""
    return re.sub(r'[^\w\s-]', '', filename).strip()[:50]

def get_restaurant_list():
    """왼쪽 패널에서 가게 리스트 수집"""
    print("가게 리스트 수집 중...")
    
    # 스크롤하여 모든 가게 로드
    try:
        scroll_container = driver.find_element(By.CSS_SELECTOR, "div.Ryr1F")
        print("스크롤 컨테이너 찾음")
        
        previous_count = 0
        max_attempts = 15
        
        for attempt in range(max_attempts):
            restaurant_elements = driver.find_elements(By.CSS_SELECTOR, "li.UEzoS")
            current_count = len(restaurant_elements)
            
            print(f"현재 로드된 가게 수: {current_count}")
            
            if current_count == previous_count:
                if attempt >= 3:
                    print("더 이상 로드할 가게가 없음")
                    break
            else:
                previous_count = current_count
            
            # 스크롤 실행
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
            time.sleep(2)
        
        return restaurant_elements
        
    except Exception as e:
        print(f"가게 리스트 수집 실패: {e}")
        return []

def click_restaurant(restaurant_element, index):
    """가게 클릭하여 세부 정보 패널 열기"""
    try:
        # 가게 링크 찾기
        link_element = restaurant_element.find_element(By.CSS_SELECTOR, "a.place_bluelink")
        
        # 스크롤하여 요소가 보이도록 함
        driver.execute_script("arguments[0].scrollIntoView(true);", link_element)
        time.sleep(1)
        
        # 클릭
        ActionChains(driver).move_to_element(link_element).click().perform()
        print(f"가게 {index} 클릭 완료")
        
        # 세부 정보 패널 로딩 대기
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"가게 {index} 클릭 실패: {e}")
        return False

def switch_to_detail_iframe():
    """오른쪽 세부 정보 iframe으로 전환"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 세부 정보 iframe 찾기
        detail_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#entryIframe")))
        driver.switch_to.frame(detail_iframe)
        print("세부 정보 iframe으로 전환 완료")
        return True
        
    except Exception as e:
        print(f"세부 정보 iframe 전환 실패: {e}")
        return False

def get_home_info():
    """홈 탭에서 기본 정보 수집"""
    home_info = {}
    
    try:
        # 가게 이름
        name_element = driver.find_element(By.CSS_SELECTOR, "span.GHAhO")
        home_info['name'] = name_element.text.strip()
        print(f"가게명: {home_info['name']}")
        
        # 가게 분류
        category_element = driver.find_element(By.CSS_SELECTOR, "span.lnJFt")
        home_info['category'] = category_element.text.strip()
        print(f"분류: {home_info['category']}")
        
        # 가게 주소
        try:
            address_element = driver.find_element(By.CSS_SELECTOR, "span.LDgIH")
            home_info['address'] = address_element.text.strip()
            print(f"주소: {home_info['address']}")
        except:
            home_info['address'] = "주소 정보 없음"
        
        # 전화번호 (있다면)
        try:
            phone_element = driver.find_element(By.CSS_SELECTOR, "span.xlx7Q")
            home_info['phone'] = phone_element.text.strip()
        except:
            home_info['phone'] = "전화번호 정보 없음"
        
        # 영업시간 (있다면)
        try:
            hours_element = driver.find_element(By.CSS_SELECTOR, "time.H3ua4")
            home_info['hours'] = hours_element.text.strip()
        except:
            home_info['hours'] = "영업시간 정보 없음"
            
    except Exception as e:
        print(f"홈 정보 수집 실패: {e}")
        
    return home_info

def find_and_click_menu_tab():
    """메뉴 탭을 정확하게 찾아서 클릭"""
    try:
        print("메뉴 탭 찾는 중...")
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 메뉴 탭 찾기
        menu_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 메뉴 탭인지 판단
                is_menu_tab = (
                    "menu" in tab_href.lower() or
                    "메뉴" in tab_text or
                    "menu" in tab_text
                )
                
                if is_menu_tab:
                    menu_tab = tab
                    print(f"메뉴 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 메뉴 탭 클릭
        if menu_tab:
            try:
                # 스크롤하여 요소가 보이도록 함
                driver.execute_script("arguments[0].scrollIntoView(true);", menu_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", menu_tab)
                time.sleep(3)
                print("메뉴 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"메뉴 탭 클릭 실패: {e}")
                return False
        else:
            print("메뉴 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"메뉴 탭 찾기 실패: {e}")
        return False

def get_menu_info(menu_images_dir, restaurant_name):
    """메뉴 탭에서 메뉴 정보 수집"""
    menu_list = []
    
    try:
        # 메뉴 더보기 버튼 클릭
        try:
            more_button_selectors = [
                "a.fvwqf",
                "button.fvwqf",
                "a[class*='more']",
                "//a[contains(text(), '더보기')]"
            ]
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_button = driver.find_element(By.XPATH, selector)
                    else:
                        more_button = driver.find_element(By.CSS_SELECTOR, selector)
                    
                    if more_button.is_displayed() and more_button.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", more_button)
                        time.sleep(2)
                        print("메뉴 더보기 버튼 클릭 완료")
                        break
                except:
                    continue
        except:
            print("메뉴 더보기 버튼이 없거나 이미 모든 메뉴가 표시됨")
        
        # 메뉴 항목들 찾기
        menu_elements = driver.find_elements(By.CSS_SELECTOR, "li.E2jtL")
        
        for i, menu_element in enumerate(menu_elements):
            menu_info = {}
            
            try:
                # 메뉴명
                name_element = menu_element.find_element(By.CSS_SELECTOR, "span.lPzHi")
                menu_info['name'] = name_element.text.strip()
                
                # 메뉴 가격
                try:
                    price_element = menu_element.find_element(By.CSS_SELECTOR, "div.GXS1X em")
                    menu_info['price'] = price_element.text.strip()
                except:
                    menu_info['price'] = "가격 정보 없음"
                
                # 메뉴 설명
                try:
                    desc_element = menu_element.find_element(By.CSS_SELECTOR, "div.TRxGt")
                    menu_info['description'] = desc_element.text.strip()
                except:
                    menu_info['description'] = "설명 없음"
                
                # 메뉴 이미지
                try:
                    img_selectors = ["img.K0PDV", "img"]
                    img_element = None
                    
                    for selector in img_selectors:
                        try:
                            img_element = menu_element.find_element(By.CSS_SELECTOR, selector)
                            break
                        except:
                            continue
                    
                    if img_element:
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            # 이미지 파일명 생성 (특수문자 제거)
                            safe_menu_name = safe_filename(menu_info['name'])[:20]
                            img_filename = f"메뉴_{i+1:02d}_{safe_menu_name}.jpg"
                            
                            # 이미지 다운로드
                            downloaded_path = download_image_improved(img_url, img_filename, menu_images_dir)
                            menu_info['image_path'] = downloaded_path
                        else:
                            menu_info['image_path'] = None
                    else:
                        menu_info['image_path'] = None
                except:
                    menu_info['image_path'] = None
                
                menu_list.append(menu_info)
                print(f"메뉴 {i+1}: {menu_info['name']} - {menu_info['price']}")
                
            except Exception as e:
                print(f"메뉴 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"메뉴 정보 수집 실패: {e}")
    
    return menu_list

def find_and_click_review_tab():
    """리뷰 탭을 정확하게 찾아서 클릭"""
    try:
        print("리뷰 탭 찾는 중...")
        
        # 페이지 스크롤하여 탭이 보이도록 함
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 리뷰 탭 찾기
        review_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 리뷰 탭인지 판단
                is_review_tab = (
                    "review" in tab_href.lower() or
                    "리뷰" in tab_text or
                    "review" in tab_text
                )
                
                if is_review_tab:
                    review_tab = tab
                    print(f"리뷰 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 리뷰 탭 클릭
        if review_tab:
            try:
                # 요소가 보이도록 스크롤
                driver.execute_script("arguments[0].scrollIntoView(true);", review_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", review_tab)
                time.sleep(3)
                print("리뷰 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"리뷰 탭 클릭 실패: {e}")
                return False
        else:
            print("리뷰 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"리뷰 탭 찾기 실패: {e}")
        return False

def click_all_review_more_buttons():
    """모든 리뷰 더보기 버튼 클릭 (개선된 버전)"""
    try:
        print("리뷰 더보기 버튼들 찾는 중...")
        
        # 지정된 셀렉터로 더보기 버튼 찾기
        more_button_selectors = [
            "div.lfH3O.fvwqf",  # 사용자가 제공한 정확한 셀렉터
            "div.fvwqf",
            "a.fvwqf",
            "button.fvwqf",
            "//a[contains(text(), '더보기')]",
            "//button[contains(text(), '더보기')]",
            "//div[contains(text(), '더보기')]"
        ]
        
        clicked_count = 0
        max_attempts = 10  # 최대 10번 시도
        
        for attempt in range(max_attempts):
            print(f"더보기 버튼 찾기 시도 {attempt + 1}/{max_attempts}")
            
            button_found = False
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_buttons = driver.find_elements(By.XPATH, selector)
                    else:
                        more_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    
                    for button in more_buttons:
                        if button.is_displayed() and button.is_enabled():
                            try:
                                # 버튼이 보이도록 스크롤
                                driver.execute_script("arguments[0].scrollIntoView(true);", button)
                                time.sleep(1)
                                
                                # 버튼 텍스트 확인
                                button_text = button.text.strip()
                                print(f"발견된 버튼 텍스트: '{button_text}'")
                                
                                if "더보기" in button_text or "more" in button_text.lower():
                                    # JavaScript로 클릭
                                    driver.execute_script("arguments[0].click();", button)
                                    time.sleep(3)  # 로딩 대기
                                    clicked_count += 1
                                    button_found = True
                                    print(f"더보기 버튼 클릭 완료 ({clicked_count}번째)")
                                    break
                            except Exception as click_error:
                                print(f"버튼 클릭 실패: {click_error}")
                                continue
                    
                    if button_found:
                        break
                        
                except Exception as e:
                    continue
            
            if not button_found:
                print("더 이상 더보기 버튼을 찾을 수 없음")
                break
                
            # 페이지 하단으로 스크롤하여 새로운 리뷰 로드 확인
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        print(f"총 {clicked_count}개의 더보기 버튼을 클릭했습니다")
        return clicked_count > 0
        
    except Exception as e:
        print(f"리뷰 더보기 버튼 처리 실패: {e}")
        return False

def collect_review_images_with_slide(restaurant_name, review_images_dir):
    """리뷰 이미지 수집 - 오직 alt="방문자리뷰사진"만 수집"""
    downloaded_images = []
    
    try:
        print("방문자리뷰사진만 수집 시작...")
        
        # 오직 alt="방문자리뷰사진"인 이미지들만 수집
        visitor_review_images = driver.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
        print(f"방문자리뷰사진 발견: {len(visitor_review_images)}개")
        
        total_images = 0
        
        for img_idx, img in enumerate(visitor_review_images):
            try:
                img_url = img.get_attribute("src")
                img_alt = img.get_attribute("alt")
                
                # alt 속성이 정확히 "방문자리뷰사진"인지 다시 한번 확인
                if img_url and img_alt == "방문자리뷰사진":
                    filename = f"방문자리뷰_{img_idx+1:03d}.jpg"
                    
                    # 이미지가 이미 다운로드되었는지 확인 (URL 기준)
                    if img_url not in [img_info.get('url') for img_info in downloaded_images]:
                        downloaded_path = download_image_improved(img_url, filename, review_images_dir)
                        if downloaded_path:
                            downloaded_images.append({
                                'path': downloaded_path,
                                'url': img_url,
                                'filename': filename,
                                'type': 'visitor_review'
                            })
                            total_images += 1
                            print(f"  방문자리뷰사진 다운로드: {filename}")
            
            except Exception as e:
                print(f"  방문자리뷰사진 {img_idx+1} 처리 실패: {e}")
                continue
        
        print(f"방문자리뷰사진 다운로드 완료: 총 {total_images}개")
        return [img['path'] for img in downloaded_images]
        
    except Exception as e:
        print(f"방문자리뷰사진 수집 실패: {e}")
        return []

def get_review_info(review_images_dir, restaurant_name, max_reviews=None):
    """리뷰 탭에서 리뷰 정보 수집 (개선된 버전)"""
    if max_reviews is None:
        max_reviews = MAX_REVIEWS_PER_RESTAURANT
    
    review_list = []
    
    try:
        # 모든 리뷰 더보기 버튼 클릭
        click_all_review_more_buttons()
        
        # 리뷰 이미지 수집 (슬라이드 포함)
        review_slide_images = collect_review_images_with_slide(restaurant_name, review_images_dir)
        
        # 최종 스크롤하여 모든 리뷰 로드
        print("최종 스크롤로 모든 리뷰 로드 중...")
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # 리뷰 항목들 찾기 (여러 셀렉터 시도)
        review_selectors = [
            "li.place_apply_pui.EjjAW",
            "li.place_apply_pui",
            "div.pui__vn15t2",
            "li[class*='review']",
            "div[class*='review']"
        ]
        
        review_elements = []
        for selector in review_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    review_elements = elements[:max_reviews]
                    print(f"리뷰 요소 찾음: {selector} - {len(review_elements)}개")
                    break
            except:
                continue
        
        if not review_elements:
            print("리뷰 요소를 찾을 수 없습니다.")
            return review_list
        
        print(f"총 {len(review_elements)}개의 리뷰 처리 시작")
        
        for i, review_element in enumerate(review_elements):
            review_info = {}
            
            try:
                # 리뷰 텍스트 - 정확한 셀렉터 사용
                review_text = "리뷰 텍스트 없음"
                try:
                    # div.pui__vn15t2에서 리뷰 텍스트 찾기
                    text_element = review_element.find_element(By.CSS_SELECTOR, "div.pui__vn15t2")
                    review_text = text_element.text.strip()
                    if not review_text or len(review_text) < 5:
                        # 다른 텍스트 셀렉터들도 시도
                        text_selectors = [
                            "span.zPfVt", 
                            "div.pui__vn15t2 span",
                            "span[class*='review']", 
                            "div[class*='content']"
                        ]
                        
                        for text_selector in text_selectors:
                            try:
                                text_elements = review_element.find_elements(By.CSS_SELECTOR, text_selector)
                                for text_elem in text_elements:
                                    text_content = text_elem.text.strip()
                                    if text_content and len(text_content) > 10:
                                        review_text = text_content
                                        break
                                if review_text != "리뷰 텍스트 없음":
                                    break
                            except:
                                continue
                except:
                    # 여전히 텍스트를 못찾았다면 전체 텍스트에서 추출 시도
                    try:
                        full_text = review_element.text.strip()
                        if len(full_text) > 20:
                            sentences = full_text.split('\n')
                            for sentence in sentences:
                                if len(sentence.strip()) > 10:
                                    review_text = sentence.strip()
                                    break
                    except:
                        pass
                
                review_info['text'] = review_text
                
                # 리뷰 날짜 - 정확한 셀렉터 사용
                review_date = "날짜 정보 없음"
                try:
                    # span.pui__gfuUIT 안의 span.pui__blind에서 날짜 찾기
                    date_container = review_element.find_element(By.CSS_SELECTOR, "span.pui__gfuUIT")
                    date_element = date_container.find_element(By.CSS_SELECTOR, "span.pui__blind")
                    date_text = date_element.text.strip()
                    
                    if date_text:
                        # 날짜 패턴 추출
                        date_patterns = [
                            r'(\d{4}\.\d{1,2}\.\d{1,2})',  # 2024.01.15
                            r'(\d{4}-\d{1,2}-\d{1,2})',   # 2024-01-15
                            r'(\d{1,2}\.\d{1,2})',        # 01.15
                            r'(\d+일전)',                  # 3일전
                            r'(\d+주전)',                  # 2주전
                            r'(\d+개월전)',                # 1개월전
                            r'(\d+년전)',                  # 1년전
                            r'(오늘)',                     # 오늘
                            r'(어제)',                     # 어제
                            r'(\d{4}년\s*\d{1,2}월\s*\d{1,2}일)'  # 2024년 1월 15일
                        ]
                        
                        for pattern in date_patterns:
                            date_match = re.search(pattern, date_text)
                            if date_match:
                                review_date = date_match.group(1)
                                break
                        
                        if review_date == "날짜 정보 없음" and len(date_text) < 30:
                            review_date = date_text
                except:
                    # 대체 날짜 셀렉터들 시도
                    date_selectors = [
                        "time", 
                        "div.pui__QKE5Pr", 
                        "span[class*='date']",
                        "div[class*='date']",
                        "span.time"
                    ]
                    
                    for date_selector in date_selectors:
                        try:
                            date_element = review_element.find_element(By.CSS_SELECTOR, date_selector)
                            date_text = date_element.text.strip()
                            
                            if date_text and len(date_text) < 30:
                                review_date = date_text
                                break
                        except:
                            continue
                
                review_info['date'] = review_date
                
                # 리뷰 평점 (별점) 수집
                try:
                    rating_selectors = [
                        "div.pui__rating span",
                        "span[class*='rating']",
                        "div[class*='star']",
                        ".rating"
                    ]
                    
                    review_rating = "평점 정보 없음"
                    for rating_selector in rating_selectors:
                        try:
                            rating_element = review_element.find_element(By.CSS_SELECTOR, rating_selector)
                            rating_text = rating_element.text.strip()
                            if rating_text and ("점" in rating_text or "★" in rating_text):
                                review_rating = rating_text
                                break
                        except:
                            continue
                    
                    review_info['rating'] = review_rating
                except:
                    review_info['rating'] = "평점 정보 없음"
                
                # 리뷰 이미지들 (개별 리뷰에서) - alt="방문자리뷰사진" 우선 사용
                review_images = []
                try:
                    # 1. 우선적으로 alt="방문자리뷰사진"인 이미지만 찾기
                    visitor_images = review_element.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
                    
                    for j, img_element in enumerate(visitor_images):
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            img_filename = f"개별방문자리뷰_{i+1:02d}_{j+1:02d}.jpg"
                            downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                            if downloaded_path:
                                review_images.append(downloaded_path)
                    
                    # 2. 방문자리뷰사진이 없는 경우에만 다른 이미지들 시도 (백업용)
                    if not visitor_images:
                        img_elements = review_element.find_elements(By.CSS_SELECTOR, "img")
                        for j, img_element in enumerate(img_elements):
                            img_url = img_element.get_attribute("src")
                            img_alt = img_element.get_attribute("alt") or ""
                            img_class = img_element.get_attribute("class") or ""
                            
                            # 프로필 사진이 아닌 실제 리뷰 이미지만 필터링 (백업용)
                            if (img_url and 
                                ("review" in img_url or "ldb-phinf" in img_url or "pstatic.net" in img_url) and
                                "profile" not in img_url.lower() and
                                "avatar" not in img_url.lower() and
                                "user" not in img_alt.lower() and
                                "profile" not in img_alt.lower() and
                                "pui__q2fg8o" not in img_class and
                                "pui__A7NplK" not in img_class):
                                
                                img_filename = f"개별리뷰_{i+1:02d}_{j+1:02d}.jpg"
                                downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                                if downloaded_path:
                                    review_images.append(downloaded_path)
                        
                except Exception as e:
                    print(f"리뷰 {i+1} 개별 이미지 수집 실패: {e}")
                
                review_info['individual_images'] = review_images
                review_list.append(review_info)
                
                # 진행 상황 출력
                if i % 10 == 0:
                    print(f"리뷰 처리 진행률: {i+1}/{len(review_elements)}")
                
                print(f"리뷰 {i+1}: {review_info['text'][:50]}..." if len(review_info['text']) > 50 else f"리뷰 {i+1}: {review_info['text']}")
                
            except Exception as e:
                print(f"리뷰 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"리뷰 정보 수집 실패: {e}")
        
    print(f"총 {len(review_list)}개의 리뷰 수집 완료")
    return review_list

def go_back_to_list():
    """목록으로 돌아가기"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 검색 결과 iframe으로 다시 전환
        search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
        driver.switch_to.frame(search_iframe)
        print("목록으로 돌아가기 완료")
        return True
        
    except Exception as e:
        print(f"목록으로 돌아가기 실패: {e}")
        return False

# =================== 메인 실행 부분 ===================
try:
    print("=" * 60)
    print(f"네이버 지도 크롤링 시작")
    print(f"검색어: {SEARCH_KEYWORD}")
    print(f"최대 가게 수: {MAX_RESTAURANTS}")
    print(f"가게당 최대 리뷰 수: {MAX_REVIEWS_PER_RESTAURANT}")
    print("=" * 60)
    
    # 1. 검색어 설정 및 접속
    encoded_keyword = urllib.parse.quote(SEARCH_KEYWORD)
    URL = f"https://map.naver.com/p/search/{encoded_keyword}"
    
    print(f"네이버 지도 접속 중: {URL}")
    driver.get(URL)
    time.sleep(5)
    
    # 2. 디렉토리 생성
    base_dir = create_directory_structure(SEARCH_KEYWORD)
    print(f"저장 디렉토리 생성: {base_dir}")
    
    # 3. 검색 결과 iframe 접근
    print("검색 결과 iframe으로 전환 중...")
    search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
    driver.switch_to.frame(search_iframe)
    
    # 4. 가게 리스트 수집
    restaurant_elements = get_restaurant_list()
    total_restaurants = len(restaurant_elements)
    print(f"총 {total_restaurants}개 가게 발견")
    
    # 5. 각 가게별로 세부 정보 수집
    max_restaurants = min(MAX_RESTAURANTS, total_restaurants)
    
    for i in range(max_restaurants):
        print(f"\n{'='*20} 가게 {i+1}/{max_restaurants} {'='*20}")
        
        try:
            # 가게 클릭
            if not click_restaurant(restaurant_elements[i], i+1):
                continue
            
            # 세부 정보 iframe으로 전환
            if not switch_to_detail_iframe():
                continue
            
            # 홈 정보 수집
            print("홈 정보 수집 중...")
            home_info = get_home_info()
            
            # 가게별 폴더 구조 생성
            restaurant_dir, menu_images_dir, review_images_dir = create_restaurant_directories(
                base_dir, home_info.get('name', f'restaurant_{i+1}')
            )
            
            # 메뉴 정보 수집
            menu_info = []
            if find_and_click_menu_tab():
                print("메뉴 정보 수집 중...")
                menu_info = get_menu_info(menu_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("메뉴 탭을 찾을 수 없어 메뉴 정보 수집을 건너뜁니다.")
            
            # 리뷰 정보 수집
            review_info = []
            if find_and_click_review_tab():
                print("리뷰 정보 수집 중...")
                review_info = get_review_info(review_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("리뷰 탭을 찾을 수 없어 리뷰 정보 수집을 건너뜁니다.")
            
            # 전체 정보 결합
            restaurant_data = {
                'index': i + 1,
                'home_info': home_info,
                'menu_info': menu_info,
                'review_info': review_info,
                'restaurant_dir': restaurant_dir,
                'collected_at': datetime.now().isoformat()
            }
            
            all_restaurant_data.append(restaurant_data)
            print(f"가게 {i+1} 정보 수집 완료!")
            print(f"- 메뉴 개수: {len(menu_info)}")
            print(f"- 리뷰 개수: {len(review_info)}")
            
            # 목록으로 돌아가기
            go_back_to_list()
            time.sleep(2)
            
        except Exception as e:
            print(f"가게 {i+1} 처리 중 오류: {e}")
            # 목록으로 돌아가기 시도
            try:
                go_back_to_list()
            except:
                pass
            continue
    
    # 6. 결과 저장
    print(f"\n{'='*60}")
    print(f"총 {len(all_restaurant_data)}개 가게 상세 정보 수집 완료!")
    
    # JSON 파일 저장
    safe_keyword = safe_filename(SEARCH_KEYWORD)
    json_filename = os.path.join(base_dir, f"{safe_keyword}_detailed_restaurants.json")
    with open(json_filename, 'w', encoding='utf-8') as jsonfile:
        json.dump(all_restaurant_data, jsonfile, ensure_ascii=False, indent=2)
    
    # CSV 파일 저장 (한글 헤더로 개선)
    csv_filename = os.path.join(base_dir, f"{safe_keyword}_restaurants_summary.csv")
    with open(csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:  # utf-8-sig로 BOM 추가
        fieldnames = ['순번', '가게명', '분류', '주소', '전화번호', '메뉴수', '리뷰수', '폴더경로']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            row = {
                '순번': restaurant['index'],
                '가게명': restaurant['home_info'].get('name', ''),
                '분류': restaurant['home_info'].get('category', ''),
                '주소': restaurant['home_info'].get('address', ''),
                '전화번호': restaurant['home_info'].get('phone', ''),
                '메뉴수': len(restaurant['menu_info']),
                '리뷰수': len(restaurant['review_info']),
                '폴더경로': restaurant.get('restaurant_dir', '')
            }
            writer.writerow(row)
    
    # 리뷰 상세 정보 CSV 저장
    reviews_csv_filename = os.path.join(base_dir, f"{safe_keyword}_reviews_detail.csv")
    with open(reviews_csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['가게명', '리뷰번호', '리뷰내용', '작성날짜', '평점', '이미지수']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            restaurant_name = restaurant['home_info'].get('name', '')
            for idx, review in enumerate(restaurant['review_info']):
                row = {
                    '가게명': restaurant_name,
                    '리뷰번호': idx + 1,
                    '리뷰내용': review.get('text', ''),
                    '작성날짜': review.get('date', ''),
                    '평점': review.get('rating', ''),
                    '이미지수': len(review.get('individual_images', []))
                }
                writer.writerow(row)
    
    # 통계 정보 출력
    total_menus = sum(len(restaurant['menu_info']) for restaurant in all_restaurant_data)
    total_reviews = sum(len(restaurant['review_info']) for restaurant in all_restaurant_data)
    total_images = 0
    
    # 이미지 파일 개수 세기
    for restaurant in all_restaurant_data:
        restaurant_dir = restaurant.get('restaurant_dir', '')
        if os.path.exists(restaurant_dir):
            for root, dirs, files in os.walk(restaurant_dir):
                image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
                total_images += len(image_files)
    
    print(f"\n파일 저장 완료:")
    print(f"- 상세 정보 (JSON): {json_filename}")
    print(f"- 요약 정보 (CSV): {csv_filename}")
    print(f"- 리뷰 상세 (CSV): {reviews_csv_filename}")
    print(f"- 이미지 폴더: 각 가게별 폴더")
    print(f"\n수집 통계:")
    print(f"- 총 가게 수: {len(all_restaurant_data)}")
    print(f"- 총 메뉴 수: {total_menus}")
    print(f"- 총 리뷰 수: {total_reviews}")
    print(f"- 총 이미지 수: {total_images}")
    
    print(f"\n{'='*60}")
    print("크롤링 완료!")

except Exception as e:
    print(f"전체 프로세스 오류 발생: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    # 드라이버 종료
    try:
        pass
#         driver.quit()
#         print("웹드라이버 종료 완료")
    except:
        pass
end = time.time()
print(f"실행 시간: {end_time - start_time:.2f}초")

네이버 지도 크롤링 시작
검색어: 신림동 햄버거
최대 가게 수: 50
가게당 최대 리뷰 수: 500
네이버 지도 접속 중: https://map.naver.com/p/search/%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0
저장 디렉토리 생성: 신림동 햄버거_last_test!plz_data
검색 결과 iframe으로 전환 중...
가게 리스트 수집 중...
스크롤 컨테이너 찾음
현재 로드된 가게 수: 10
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
더 이상 로드할 가게가 없음
총 50개 가게 발견

==================== 가게 1/50 ====================
가게 1 클릭 완료
세부 정보 iframe으로 전환 완료
홈 정보 수집 중...
가게명: 아토커피 신림점
분류: 카페,디저트
주소: 서울 관악구 관천로 79 1층
메뉴 탭 찾는 중...
탭 요소들 찾음: 6개
탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1539251371/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081628&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 2: 텍스트='소식', href='https://pcmap.place.naver.com/restaurant/1539251371/feed?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081628&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 3: 텍스트='메뉴', href='https://p

  이미지 다운로드 성공: 방문자리뷰_013.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_013.jpg
  이미지 다운로드 성공: 방문자리뷰_014.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_014.jpg
  이미지 다운로드 성공: 방문자리뷰_015.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_015.jpg
  이미지 다운로드 성공: 방문자리뷰_016.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_016.jpg
  이미지 다운로드 성공: 방문자리뷰_017.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_017.jpg
  이미지 다운로드 성공: 방문자리뷰_018.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_018.jpg
  이미지 다운로드 성공: 방문자리뷰_019.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_019.jpg
  이미지 다운로드 성공: 방문자리뷰_020.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_020.jpg
  이미지 다운로드 성공: 방문자리뷰_021.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_021.jpg
  이미지 다운로드 성공: 방문자리뷰_022.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_022.jpg
  이미지 다운로드 성공: 방문자리뷰_023.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_023.jpg
  이미지 다운로드 성공: 방문자리뷰_024.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_024.jpg
  이미지 다운로드 성공: 방문자리뷰_025.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_025.jpg
  이미지 다운로드 성공: 방문자리뷰_026.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_026.jpg
  이미지 다운로드 성공: 방문자리뷰_027.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_027.jpg
  이미지 다운로드 성공: 방문자리뷰_028.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_028.jpg
  이미지 다운로드 성공: 방문자리뷰_029.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_029.j

  이미지 다운로드 성공: 개별리뷰_56_01.jpg
  이미지 다운로드 성공: 개별리뷰_56_02.jpg
  이미지 다운로드 성공: 개별리뷰_56_03.jpg
리뷰 56: 아늑하니 조용하고 커피가 맛있어요😋
오렌지라떼랑 오샷추 맛도리~~
카야 생크림 와플 바삭하...
  이미지 다운로드 성공: 개별리뷰_57_01.jpg
  이미지 다운로드 성공: 개별리뷰_57_02.jpg
리뷰 57: 집 근처라 카페 데이트하러 떡볶이랑 레몬콕, TAK 주스 먹었는데 모두 너무 맛있네요 ㅠㅠ...
  이미지 다운로드 성공: 개별리뷰_58_01.jpg
  이미지 다운로드 성공: 개별리뷰_58_02.jpg
리뷰 58: 신림24시카페 아토커피 다녀왔어요. 산미가 가벼운 바디감이 좋은 오리지널 원두 아이메와 든...
  이미지 다운로드 성공: 개별리뷰_59_02.jpg
리뷰 59: 여름철 신메뉴 수박주스 시원해서 먹기 좋아요~
건강에 좋은 ABC주스도 맛있네요!
번창하세...
  이미지 다운로드 성공: 개별방문자리뷰_60_01.jpg
리뷰 60: 운동다니면서 본 곳인데 도림천 앞에 있어서 시야가 좋아요. 음료도 맛있고 커피도 맛있어요....
  이미지 다운로드 성공: 개별방문자리뷰_61_01.jpg
리뷰 처리 진행률: 61/110
리뷰 61: 도림천 돌아다니다가 배고파서 들어갔는데 에그마요 샌드위치가 너무 맛있었어요 ❤️ 프로틴 음...
  이미지 다운로드 성공: 개별리뷰_62_02.jpg
리뷰 62: 통유리 개방감 카페라 너무 좋고 이 근방에 24시 카페 있는 줄 몰랐는데 알게되어서 밤에도...
  이미지 다운로드 성공: 개별리뷰_63_02.jpg
리뷰 63: 새로 신림 24시 카페 생겼다해서 와봤는데 인테리어도 예쁘고 분위기도 넘 좋아요ㅠㅠ 토마토...
  이미지 다운로드 성공: 개별리뷰_64_02.jpg
리뷰 64: 신림 24시카페, 디저트 맛있고 음료도 다양해요. 요즘
카페가 일찍 닫아서 아쉬웠는데 24...
  이미지 다운로드 성공: 개별리뷰_65_01.jpg
  이미지 다

  이미지 다운로드 성공: 메뉴_29_어니언링.jpg
메뉴 29: 어니언링 - 3,600
  이미지 다운로드 성공: 메뉴_30_빅 치즈스틱수제기본.jpg
메뉴 30: 빅 치즈스틱(수제)(기본) - 3,900
  이미지 다운로드 성공: 메뉴_31_베이컨 치즈스틱.jpg
메뉴 31: 베이컨 치즈스틱 - 4,500
  이미지 다운로드 성공: 메뉴_32_락치킨 S4조각.jpg
메뉴 32: 락치킨 S(4조각) - 4,400
  이미지 다운로드 성공: 메뉴_33_펩시.jpg
메뉴 33: 펩시 - 2,500
  이미지 다운로드 성공: 메뉴_34_사이다.jpg
메뉴 34: 사이다 - 2,500
  이미지 다운로드 성공: 메뉴_35_제로 펩시.jpg
메뉴 35: 제로 펩시 - 2,500
리뷰 탭 찾는 중...
탭 요소들 찾음: 5개
탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1592020163/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081630&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 2: 텍스트='메뉴', href='https://pcmap.place.naver.com/restaurant/1592020163/menu?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081630&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 3: 텍스트='리뷰', href='https://pcmap.place.naver.com/restaurant/1592020163/review?entry=bmp&from=map&fromPanelNum=2&timestamp=2025070


==================== 가게 3/50 ====================
가게 3 클릭 완료
세부 정보 iframe으로 전환 완료
홈 정보 수집 중...
가게명: 버거운버거 서울대점
분류: 햄버거
주소: 서울 관악구 관악로 1 101동 1층 113호
메뉴 탭 찾는 중...
탭 요소들 찾음: 6개
탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1347648415/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081632&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 2: 텍스트='소식', href='https://pcmap.place.naver.com/restaurant/1347648415/feed?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081632&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 3: 텍스트='메뉴', href='https://pcmap.place.naver.com/restaurant/1347648415/menu?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081632&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
메뉴 탭 발견: 메뉴
메뉴 탭 클릭 완료
메뉴 정보 수집 중...
메뉴 더보기 버튼 클릭 완료
  이미지 다운로드 성공: 메뉴_01_한입강정 SET실속.jpg
메뉴 1: 한입강정 SET(실속) - 11,500
  이미지 다운로드 성

...

리뷰 탭 클릭 완료
리뷰 정보 수집 중...
리뷰 더보기 버튼들 찾는 중...
더보기 버튼 찾기 시도 1/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (1번째)
더보기 버튼 찾기 시도 2/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (2번째)
더보기 버튼 찾기 시도 3/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (3번째)
더보기 버튼 찾기 시도 4/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (4번째)
더보기 버튼 찾기 시도 5/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (5번째)
더보기 버튼 찾기 시도 6/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (6번째)
더보기 버튼 찾기 시도 7/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (7번째)
더보기 버튼 찾기 시도 8/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (8번째)
더보기 버튼 찾기 시도 9/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (9번째)
더보기 버튼 찾기 시도 10/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (10번째)
총 10개의 더보기 버튼을 클릭했습니다
방문자리뷰사진만 수집 시작...
방문자리뷰사진 발견: 25개
  이미지 다운로드 성공: 방문자리뷰_001.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_001.jpg
  이미지 다운로드 성공: 방문자리뷰_002.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_002.jpg
  이미지 다운로드 성공: 방문자리뷰_003.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_003.jpg
  이미지 다운로드 성공: 방문자리뷰_004.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_004.jpg
  이미지 다운로드 성공: 방문자리뷰_005.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_005.jpg
  이미지 다운로드 성공: 방문자리뷰_006.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_00

  이미지 다운로드 성공: 개별리뷰_57_02.jpg
리뷰 57: 수제버거를 좋아해서 왔습니다.
훈제향을 입힌 페티가 특별함을 선사하는거 같습니다.
불향이 ...
  이미지 다운로드 성공: 개별리뷰_58_02.jpg
리뷰 58: 남자친구랑 서울데이트하러 왔는데 너무 맛있는 햄버거 맛집 발견해서 좋아요!ㅎㅎ훈연향이 확나...
  이미지 다운로드 성공: 개별리뷰_59_01.jpg
  이미지 다운로드 성공: 개별리뷰_59_02.jpg
리뷰 59: 사당역에 이렇게 맛있는 수제버거집이 있다니 진짜 너무 맛있었어요ㅠ 부서지는거 없이 깔끔하게...
  이미지 다운로드 성공: 개별리뷰_60_01.jpg
  이미지 다운로드 성공: 개별리뷰_60_02.jpg
리뷰 60: 맛있어요!!! 오랜만에 왔는데 여전히 맛있네요.
  이미지 다운로드 성공: 개별방문자리뷰_61_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_61_02.jpg
리뷰 처리 진행률: 61/110
리뷰 61: 맛있어요😆
영상도 시청하고 재밌네요
  이미지 다운로드 성공: 개별리뷰_62_01.jpg
  이미지 다운로드 성공: 개별리뷰_62_02.jpg
리뷰 62: 사당역 최고 수제버거 맛집인 듯 합니다~!
  이미지 다운로드 성공: 개별리뷰_63_02.jpg
리뷰 63: 너무맛있고즐거운회식자리였어요.ㅊ
또갈게요.감사해요♡
  이미지 다운로드 성공: 개별리뷰_64_01.jpg
  이미지 다운로드 성공: 개별리뷰_64_02.jpg
리뷰 64: 정말 맛있어요. 옛날 이수역쪽에 있을때도 왔지만 단골이 생기는 이유는 꾸준한 버거맛. 훈연...
  이미지 다운로드 성공: 개별리뷰_65_01.jpg
  이미지 다운로드 성공: 개별리뷰_65_02.jpg
리뷰 65: 햄버거에 훈연향이....???!
수제버거 먹은것중에 진짜 손에 꼽아요
감튀 꼭 드세요오오오...
  이미지 다운로드 성공: 개별리뷰_66_02.jpg
리뷰 66: 우연히 알게 되었는데, 너무 맛있고 고급져요!
플레이팅도 예쁘고 깔끔하면서 대접받는 느낌이...
  

메뉴 탭 클릭 완료
메뉴 정보 수집 중...
리뷰 탭 찾는 중...
탭 요소들 찾음: 9개
탭 1: 텍스트='홈', href=''
탭 2: 텍스트='소식', href=''
탭 3: 텍스트='메뉴', href=''
탭 4: 텍스트='리뷰', href=''
리뷰 탭 발견: 리뷰
리뷰 탭 클릭 완료
리뷰 정보 수집 중...
리뷰 더보기 버튼들 찾는 중...
더보기 버튼 찾기 시도 1/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (1번째)
더보기 버튼 찾기 시도 2/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (2번째)
더보기 버튼 찾기 시도 3/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (3번째)
더보기 버튼 찾기 시도 4/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (4번째)
더보기 버튼 찾기 시도 5/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (5번째)
더보기 버튼 찾기 시도 6/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (6번째)
더보기 버튼 찾기 시도 7/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (7번째)
더보기 버튼 찾기 시도 8/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (8번째)
더보기 버튼 찾기 시도 9/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (9번째)
더보기 버튼 찾기 시도 10/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (10번째)
총 10개의 더보기 버튼을 클릭했습니다
방문자리뷰사진만 수집 시작...
방문자리뷰사진 발견: 10개
  이미지 다운로드 성공: 방문자리뷰_001.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_001.jpg
  이미지 다운로드 성공: 방문자리뷰_002.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_002.jpg
  이미지 다운로드 성공: 방문자리뷰_003.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_003.jpg
  이미지 다운로드 성공: 방문

  이미지 다운로드 성공: 개별리뷰_79_02.jpg
리뷰 79: 메인도 사이드도 너어무 맛있습니다… 특히 패티랑 빵이 최고에요!
  이미지 다운로드 성공: 개별리뷰_80_02.jpg
  이미지 다운로드 성공: 개별리뷰_80_03.jpg
리뷰 80: 늘 믿고 시켜먹는 알렉스 버거 입니다 ~! 👍 🤗
  이미지 다운로드 성공: 개별리뷰_81_01.jpg
  이미지 다운로드 성공: 개별리뷰_81_02.jpg
리뷰 처리 진행률: 81/110
리뷰 81: 말이필요없어요 넘 맛나요!!
  이미지 다운로드 성공: 개별리뷰_82_02.jpg
리뷰 82: 고기가 너무 기름지지도 않고 담백하고 맛있었어요!!! 빵도 진짜 바삭바삭합니다!!!! 햄버...
  이미지 다운로드 성공: 개별리뷰_83_02.jpg
리뷰 83: 맛있어요.. 빵이 엄청 맛있어요
  이미지 다운로드 성공: 개별리뷰_84_02.jpg
리뷰 84: 깔끔해요.오빠가 먹어보구 맛나다해서 직접방문 했어요~
  이미지 다운로드 성공: 개별리뷰_85_02.jpg
리뷰 85: 보라매 5년차 주민이 추천하는 보라매역 최고의 맛집 ♥️ 절대 후회 없을거에요 !!!
  이미지 다운로드 성공: 개별리뷰_86_02.jpg
리뷰 86: 보라매 오면 항상 방문하는 맛집입니다. 너무 맛있네요.
  이미지 다운로드 성공: 개별리뷰_87_02.jpg
리뷰 87: 
  이미지 다운로드 성공: 개별리뷰_88_02.jpg
리뷰 88: 친구랑 밥먹으러 왔는데, 수제버거집 가성비 너무 좋아요 !! 다음에도 또 오려고 합니다 😆...
  이미지 다운로드 성공: 개별리뷰_89_02.jpg
리뷰 89: 동작구에서 알렉스보다 맛있는 햄버거집을 못 봤어요~ 넘넘 맛있고 양도 많습니다 ㅎㅎ 포장도...
  이미지 다운로드 성공: 개별리뷰_90_01.jpg
  이미지 다운로드 성공: 개별리뷰_90_02.jpg
리뷰 90: 수제버거 맛집
아이가 먹구싶다해서 포장!!
언제 먹어도 맛있어요
감튀는 커서 더 맛있어요~...
  이미지 다운로드 성공: 개별리뷰_91_0

KeyboardInterrupt: 

In [17]:
### 제ㅈ발 프로필 이미지 뺴고 할 수 있게 해주세요.

from time import time 
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
import requests
import time
import os
import json
import csv
import urllib.parse
import re
from datetime import datetime
from PIL import Image
import io
start = time.time()
# =================== 설정 구역 ===================
# 여기서 검색어와 기타 설정을 변경하세요
SEARCH_KEYWORD = "신림동 햄버거"  # 원하는 검색어로 변경
MAX_RESTAURANTS = 50  # 크롤링할 최대 가게 수
MAX_REVIEWS_PER_RESTAURANT = 500  # 가게당 최대 리뷰 수
# ================================================

# 서비스 설정
service = Service(port=9999)
# 크롬 인터넷 설정
options = webdriver.ChromeOptions()
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36')
options.add_argument('window-size=1380,900')
# 드라이버 생성
driver = webdriver.Chrome(service=service, options=options)
# WebDriverWait 초기화
wait = WebDriverWait(driver, 10)

# 결과 저장할 리스트
all_restaurant_data = []

def create_directory_structure(keyword):
    """가게별 이미지 저장용 디렉토리 구조 생성"""
    base_dir = f"{keyword}_test50_sam_data"
    os.makedirs(base_dir, exist_ok=True)
    return base_dir

def create_restaurant_directories(base_dir, restaurant_name):
    """가게별 폴더 구조 생성"""
    safe_name = safe_filename(restaurant_name)
    restaurant_dir = os.path.join(base_dir, safe_name)
    menu_images_dir = os.path.join(restaurant_dir, "메뉴_이미지")
    review_images_dir = os.path.join(restaurant_dir, "리뷰_이미지")
    
    os.makedirs(menu_images_dir, exist_ok=True)
    os.makedirs(review_images_dir, exist_ok=True)
    
    return restaurant_dir, menu_images_dir, review_images_dir

def download_image_improved(url, filename, images_dir):
    """이미지 다운로드 (개선된 버전 - 이미지 검증 포함)"""
    try:
        # User-Agent 헤더 추가
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
            'Referer': 'https://map.naver.com/'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            # 이미지 데이터 검증
            try:
                # PIL로 이미지 검증
                img = Image.open(io.BytesIO(response.content))
                img.verify()  # 이미지가 유효한지 검증
                
                # 파일 저장
                filepath = os.path.join(images_dir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                print(f"  이미지 다운로드 성공: {filename}")
                return filepath
            except Exception as img_error:
                print(f"  이미지 검증 실패 ({filename}): {img_error}")
                return None
        else:
            print(f"  HTTP 오류 ({filename}): {response.status_code}")
            return None
    except Exception as e:
        print(f"  이미지 다운로드 실패 ({filename}): {e}")
        return None

def safe_filename(filename):
    """파일명에서 특수문자 제거"""
    return re.sub(r'[^\w\s-]', '', filename).strip()[:50]

def get_restaurant_list():
    """왼쪽 패널에서 가게 리스트 수집"""
    print("가게 리스트 수집 중...")
    
    # 스크롤하여 모든 가게 로드
    try:
        scroll_container = driver.find_element(By.CSS_SELECTOR, "div.Ryr1F")
        print("스크롤 컨테이너 찾음")
        
        previous_count = 0
        max_attempts = 15
        
        for attempt in range(max_attempts):
            restaurant_elements = driver.find_elements(By.CSS_SELECTOR, "li.UEzoS")
            current_count = len(restaurant_elements)
            
            print(f"현재 로드된 가게 수: {current_count}")
            
            if current_count == previous_count:
                if attempt >= 3:
                    print("더 이상 로드할 가게가 없음")
                    break
            else:
                previous_count = current_count
            
            # 스크롤 실행
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
            time.sleep(2)
        
        return restaurant_elements
        
    except Exception as e:
        print(f"가게 리스트 수집 실패: {e}")
        return []

def click_restaurant(restaurant_element, index):
    """가게 클릭하여 세부 정보 패널 열기"""
    try:
        # 가게 링크 찾기
        link_element = restaurant_element.find_element(By.CSS_SELECTOR, "a.place_bluelink")
        
        # 스크롤하여 요소가 보이도록 함
        driver.execute_script("arguments[0].scrollIntoView(true);", link_element)
        time.sleep(1)
        
        # 클릭
        ActionChains(driver).move_to_element(link_element).click().perform()
        print(f"가게 {index} 클릭 완료")
        
        # 세부 정보 패널 로딩 대기
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"가게 {index} 클릭 실패: {e}")
        return False

def switch_to_detail_iframe():
    """오른쪽 세부 정보 iframe으로 전환"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 세부 정보 iframe 찾기
        detail_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#entryIframe")))
        driver.switch_to.frame(detail_iframe)
        print("세부 정보 iframe으로 전환 완료")
        return True
        
    except Exception as e:
        print(f"세부 정보 iframe 전환 실패: {e}")
        return False

def get_home_info():
    """홈 탭에서 기본 정보 수집"""
    home_info = {}
    
    try:
        # 가게 이름
        name_element = driver.find_element(By.CSS_SELECTOR, "span.GHAhO")
        home_info['name'] = name_element.text.strip()
        print(f"가게명: {home_info['name']}")
        
        # 가게 분류
        category_element = driver.find_element(By.CSS_SELECTOR, "span.lnJFt")
        home_info['category'] = category_element.text.strip()
        print(f"분류: {home_info['category']}")
        
        # 가게 주소
        try:
            address_element = driver.find_element(By.CSS_SELECTOR, "span.LDgIH")
            home_info['address'] = address_element.text.strip()
            print(f"주소: {home_info['address']}")
        except:
            home_info['address'] = "주소 정보 없음"
        
        # 전화번호 (있다면)
        try:
            phone_element = driver.find_element(By.CSS_SELECTOR, "span.xlx7Q")
            home_info['phone'] = phone_element.text.strip()
        except:
            home_info['phone'] = "전화번호 정보 없음"
        
        # 영업시간 (있다면)
        try:
            hours_element = driver.find_element(By.CSS_SELECTOR, "time.H3ua4")
            home_info['hours'] = hours_element.text.strip()
        except:
            home_info['hours'] = "영업시간 정보 없음"
            
    except Exception as e:
        print(f"홈 정보 수집 실패: {e}")
        
    return home_info

def find_and_click_menu_tab():
    """메뉴 탭을 정확하게 찾아서 클릭"""
    try:
        print("메뉴 탭 찾는 중...")
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 메뉴 탭 찾기
        menu_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 메뉴 탭인지 판단
                is_menu_tab = (
                    "menu" in tab_href.lower() or
                    "메뉴" in tab_text or
                    "menu" in tab_text
                )
                
                if is_menu_tab:
                    menu_tab = tab
                    print(f"메뉴 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 메뉴 탭 클릭
        if menu_tab:
            try:
                # 스크롤하여 요소가 보이도록 함
                driver.execute_script("arguments[0].scrollIntoView(true);", menu_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", menu_tab)
                time.sleep(3)
                print("메뉴 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"메뉴 탭 클릭 실패: {e}")
                return False
        else:
            print("메뉴 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"메뉴 탭 찾기 실패: {e}")
        return False

def get_menu_info(menu_images_dir, restaurant_name):
    """메뉴 탭에서 메뉴 정보 수집"""
    menu_list = []
    
    try:
        # 메뉴 더보기 버튼 클릭
        try:
            more_button_selectors = [
                "a.fvwqf",
                "button.fvwqf",
                "a[class*='more']",
                "//a[contains(text(), '더보기')]"
            ]
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_button = driver.find_element(By.XPATH, selector)
                    else:
                        more_button = driver.find_element(By.CSS_SELECTOR, selector)
                    
                    if more_button.is_displayed() and more_button.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", more_button)
                        time.sleep(2)
                        print("메뉴 더보기 버튼 클릭 완료")
                        break
                except:
                    continue
        except:
            print("메뉴 더보기 버튼이 없거나 이미 모든 메뉴가 표시됨")
        
        # 메뉴 항목들 찾기
        menu_elements = driver.find_elements(By.CSS_SELECTOR, "li.E2jtL")
        
        for i, menu_element in enumerate(menu_elements):
            menu_info = {}
            
            try:
                # 메뉴명
                name_element = menu_element.find_element(By.CSS_SELECTOR, "span.lPzHi")
                menu_info['name'] = name_element.text.strip()
                
                # 메뉴 가격
                try:
                    price_element = menu_element.find_element(By.CSS_SELECTOR, "div.GXS1X em")
                    menu_info['price'] = price_element.text.strip()
                except:
                    menu_info['price'] = "가격 정보 없음"
                
                # 메뉴 설명
                try:
                    desc_element = menu_element.find_element(By.CSS_SELECTOR, "div.TRxGt")
                    menu_info['description'] = desc_element.text.strip()
                except:
                    menu_info['description'] = "설명 없음"
                
                # 메뉴 이미지
                try:
                    img_selectors = ["img.K0PDV", "img"]
                    img_element = None
                    
                    for selector in img_selectors:
                        try:
                            img_element = menu_element.find_element(By.CSS_SELECTOR, selector)
                            break
                        except:
                            continue
                    
                    if img_element:
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            # 이미지 파일명 생성 (특수문자 제거)
                            safe_menu_name = safe_filename(menu_info['name'])[:20]
                            img_filename = f"메뉴_{i+1:02d}_{safe_menu_name}.jpg"
                            
                            # 이미지 다운로드
                            downloaded_path = download_image_improved(img_url, img_filename, menu_images_dir)
                            menu_info['image_path'] = downloaded_path
                        else:
                            menu_info['image_path'] = None
                    else:
                        menu_info['image_path'] = None
                except:
                    menu_info['image_path'] = None
                
                menu_list.append(menu_info)
                print(f"메뉴 {i+1}: {menu_info['name']} - {menu_info['price']}")
                
            except Exception as e:
                print(f"메뉴 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"메뉴 정보 수집 실패: {e}")
    
    return menu_list

def find_and_click_review_tab():
    """리뷰 탭을 정확하게 찾아서 클릭"""
    try:
        print("리뷰 탭 찾는 중...")
        
        # 페이지 스크롤하여 탭이 보이도록 함
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 리뷰 탭 찾기
        review_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 리뷰 탭인지 판단
                is_review_tab = (
                    "review" in tab_href.lower() or
                    "리뷰" in tab_text or
                    "review" in tab_text
                )
                
                if is_review_tab:
                    review_tab = tab
                    print(f"리뷰 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 리뷰 탭 클릭
        if review_tab:
            try:
                # 요소가 보이도록 스크롤
                driver.execute_script("arguments[0].scrollIntoView(true);", review_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", review_tab)
                time.sleep(3)
                print("리뷰 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"리뷰 탭 클릭 실패: {e}")
                return False
        else:
            print("리뷰 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"리뷰 탭 찾기 실패: {e}")
        return False

def click_all_review_more_buttons():
    """모든 리뷰 더보기 버튼 클릭 (개선된 버전)"""
    try:
        print("리뷰 더보기 버튼들 찾는 중...")
        
        # 지정된 셀렉터로 더보기 버튼 찾기
        more_button_selectors = [
            "div.lfH3O.fvwqf",  # 사용자가 제공한 정확한 셀렉터
            "div.fvwqf",
            "a.fvwqf",
            "button.fvwqf",
            "//a[contains(text(), '더보기')]",
            "//button[contains(text(), '더보기')]",
            "//div[contains(text(), '더보기')]"
        ]
        
        clicked_count = 0
        max_attempts = 10  # 최대 10번 시도
        
        for attempt in range(max_attempts):
            print(f"더보기 버튼 찾기 시도 {attempt + 1}/{max_attempts}")
            
            button_found = False
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_buttons = driver.find_elements(By.XPATH, selector)
                    else:
                        more_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    
                    for button in more_buttons:
                        if button.is_displayed() and button.is_enabled():
                            try:
                                # 버튼이 보이도록 스크롤
                                driver.execute_script("arguments[0].scrollIntoView(true);", button)
                                time.sleep(1)
                                
                                # 버튼 텍스트 확인
                                button_text = button.text.strip()
                                print(f"발견된 버튼 텍스트: '{button_text}'")
                                
                                if "더보기" in button_text or "more" in button_text.lower():
                                    # JavaScript로 클릭
                                    driver.execute_script("arguments[0].click();", button)
                                    time.sleep(3)  # 로딩 대기
                                    clicked_count += 1
                                    button_found = True
                                    print(f"더보기 버튼 클릭 완료 ({clicked_count}번째)")
                                    break
                            except Exception as click_error:
                                print(f"버튼 클릭 실패: {click_error}")
                                continue
                    
                    if button_found:
                        break
                        
                except Exception as e:
                    continue
            
            if not button_found:
                print("더 이상 더보기 버튼을 찾을 수 없음")
                break
                
            # 페이지 하단으로 스크롤하여 새로운 리뷰 로드 확인
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        print(f"총 {clicked_count}개의 더보기 버튼을 클릭했습니다")
        return clicked_count > 0
        
    except Exception as e:
        print(f"리뷰 더보기 버튼 처리 실패: {e}")
        return False

def collect_review_images_with_slide(restaurant_name, review_images_dir):
    """리뷰 이미지 수집 - 프로필 영역 제외하고 방문자리뷰사진만 수집"""
    downloaded_images = []
    
    try:
        print("방문자리뷰사진만 수집 시작 (프로필 영역 제외)...")
        
        # 전체 페이지에서 alt="방문자리뷰사진"인 이미지들 찾기
        all_visitor_images = driver.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
        print(f"전체 방문자리뷰사진 발견: {len(all_visitor_images)}개")
        
        # 프로필 영역의 이미지들 제외
        profile_containers = driver.find_elements(By.CSS_SELECTOR, 'div.pui__q2fg8o.pui__A7NplK')
        profile_images = []
        
        for container in profile_containers:
            container_images = container.find_elements(By.CSS_SELECTOR, 'img')
            profile_images.extend(container_images)
        
        print(f"프로필 영역 이미지 발견: {len(profile_images)}개 (제외 예정)")
        
        total_images = 0
        excluded_count = 0
        
        for img_idx, img in enumerate(all_visitor_images):
            try:
                # 이 이미지가 프로필 영역에 속하는지 확인
                is_profile_image = False
                for profile_img in profile_images:
                    try:
                        if (img.get_attribute("src") == profile_img.get_attribute("src") or
                            driver.execute_script("return arguments[0].contains(arguments[1])", 
                                                profile_containers[profile_images.index(profile_img) // 10 if profile_images.index(profile_img) < len(profile_containers) * 10 else 0], img)):
                            is_profile_image = True
                            break
                    except:
                        continue
                
                # 프로필 영역의 이미지가 아닌 경우에만 다운로드
                if not is_profile_image:
                    img_url = img.get_attribute("src")
                    img_alt = img.get_attribute("alt")
                    
                    # alt 속성이 정확히 "방문자리뷰사진"인지 다시 한번 확인
                    if img_url and img_alt == "방문자리뷰사진":
                        filename = f"방문자리뷰_{total_images+1:03d}.jpg"
                        
                        # 이미지가 이미 다운로드되었는지 확인 (URL 기준)
                        if img_url not in [img_info.get('url') for img_info in downloaded_images]:
                            downloaded_path = download_image_improved(img_url, filename, review_images_dir)
                            if downloaded_path:
                                downloaded_images.append({
                                    'path': downloaded_path,
                                    'url': img_url,
                                    'filename': filename,
                                    'type': 'visitor_review'
                                })
                                total_images += 1
                                print(f"  방문자리뷰사진 다운로드: {filename}")
                else:
                    excluded_count += 1
                    print(f"  프로필 영역 이미지 제외: {excluded_count}개")
            
            except Exception as e:
                print(f"  이미지 {img_idx+1} 처리 실패: {e}")
                continue
        
        print(f"방문자리뷰사진 다운로드 완료: 총 {total_images}개")
        print(f"프로필 영역 이미지 제외: {excluded_count}개")
        return [img['path'] for img in downloaded_images]
        
    except Exception as e:
        print(f"방문자리뷰사진 수집 실패: {e}")
        return []

def get_review_info(review_images_dir, restaurant_name, max_reviews=None):
    """리뷰 탭에서 리뷰 정보 수집 (개선된 버전)"""
    if max_reviews is None:
        max_reviews = MAX_REVIEWS_PER_RESTAURANT
    
    review_list = []
    
    try:
        # 모든 리뷰 더보기 버튼 클릭
        click_all_review_more_buttons()
        
        # 리뷰 이미지 수집 (슬라이드 포함)
        review_slide_images = collect_review_images_with_slide(restaurant_name, review_images_dir)
        
        # 최종 스크롤하여 모든 리뷰 로드
        print("최종 스크롤로 모든 리뷰 로드 중...")
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # 리뷰 항목들 찾기 (여러 셀렉터 시도)
        review_selectors = [
            "li.place_apply_pui.EjjAW",
            "li.place_apply_pui",
            "div.pui__vn15t2",
            "li[class*='review']",
            "div[class*='review']"
        ]
        
        review_elements = []
        for selector in review_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    review_elements = elements[:max_reviews]
                    print(f"리뷰 요소 찾음: {selector} - {len(review_elements)}개")
                    break
            except:
                continue
        
        if not review_elements:
            print("리뷰 요소를 찾을 수 없습니다.")
            return review_list
        
        print(f"총 {len(review_elements)}개의 리뷰 처리 시작")
        
        for i, review_element in enumerate(review_elements):
            review_info = {}
            
            try:
                # 리뷰 텍스트 - 정확한 셀렉터 사용
                review_text = "리뷰 텍스트 없음"
                try:
                    # div.pui__vn15t2에서 리뷰 텍스트 찾기
                    text_element = review_element.find_element(By.CSS_SELECTOR, "div.pui__vn15t2")
                    review_text = text_element.text.strip()
                    if not review_text or len(review_text) < 5:
                        # 다른 텍스트 셀렉터들도 시도
                        text_selectors = [
                            "span.zPfVt", 
                            "div.pui__vn15t2 span",
                            "span[class*='review']", 
                            "div[class*='content']"
                        ]
                        
                        for text_selector in text_selectors:
                            try:
                                text_elements = review_element.find_elements(By.CSS_SELECTOR, text_selector)
                                for text_elem in text_elements:
                                    text_content = text_elem.text.strip()
                                    if text_content and len(text_content) > 10:
                                        review_text = text_content
                                        break
                                if review_text != "리뷰 텍스트 없음":
                                    break
                            except:
                                continue
                except:
                    # 여전히 텍스트를 못찾았다면 전체 텍스트에서 추출 시도
                    try:
                        full_text = review_element.text.strip()
                        if len(full_text) > 20:
                            sentences = full_text.split('\n')
                            for sentence in sentences:
                                if len(sentence.strip()) > 10:
                                    review_text = sentence.strip()
                                    break
                    except:
                        pass
                
                review_info['text'] = review_text
                
                # 리뷰 날짜 - 정확한 셀렉터 사용
                review_date = "날짜 정보 없음"
                try:
                    # span.pui__gfuUIT 안의 span.pui__blind에서 날짜 찾기
                    date_container = review_element.find_element(By.CSS_SELECTOR, "span.pui__gfuUIT")
                    date_element = date_container.find_element(By.CSS_SELECTOR, "span.pui__blind")
                    date_text = date_element.text.strip()
                    
                    if date_text:
                        # 날짜 패턴 추출
                        date_patterns = [
                            r'(\d{4}\.\d{1,2}\.\d{1,2})',  # 2024.01.15
                            r'(\d{4}-\d{1,2}-\d{1,2})',   # 2024-01-15
                            r'(\d{1,2}\.\d{1,2})',        # 01.15
                            r'(\d+일전)',                  # 3일전
                            r'(\d+주전)',                  # 2주전
                            r'(\d+개월전)',                # 1개월전
                            r'(\d+년전)',                  # 1년전
                            r'(오늘)',                     # 오늘
                            r'(어제)',                     # 어제
                            r'(\d{4}년\s*\d{1,2}월\s*\d{1,2}일)'  # 2024년 1월 15일
                        ]
                        
                        for pattern in date_patterns:
                            date_match = re.search(pattern, date_text)
                            if date_match:
                                review_date = date_match.group(1)
                                break
                        
                        if review_date == "날짜 정보 없음" and len(date_text) < 30:
                            review_date = date_text
                except:
                    # 대체 날짜 셀렉터들 시도
                    date_selectors = [
                        "time", 
                        "div.pui__QKE5Pr", 
                        "span[class*='date']",
                        "div[class*='date']",
                        "span.time"
                    ]
                    
                    for date_selector in date_selectors:
                        try:
                            date_element = review_element.find_element(By.CSS_SELECTOR, date_selector)
                            date_text = date_element.text.strip()
                            
                            if date_text and len(date_text) < 30:
                                review_date = date_text
                                break
                        except:
                            continue
                
                review_info['date'] = review_date
                
                # 리뷰 평점 (별점) 수집
                try:
                    rating_selectors = [
                        "div.pui__rating span",
                        "span[class*='rating']",
                        "div[class*='star']",
                        ".rating"
                    ]
                    
                    review_rating = "평점 정보 없음"
                    for rating_selector in rating_selectors:
                        try:
                            rating_element = review_element.find_element(By.CSS_SELECTOR, rating_selector)
                            rating_text = rating_element.text.strip()
                            if rating_text and ("점" in rating_text or "★" in rating_text):
                                review_rating = rating_text
                                break
                        except:
                            continue
                    
                    review_info['rating'] = review_rating
                except:
                    review_info['rating'] = "평점 정보 없음"
                
                # 리뷰 이미지들 (개별 리뷰에서) - 오직 alt="방문자리뷰사진"만
                review_images = []
                try:
                    # 오직 alt="방문자리뷰사진"인 이미지만 찾기
                    visitor_images = review_element.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
                    
                    for j, img_element in enumerate(visitor_images):
                        img_url = img_element.get_attribute("src")
                        img_alt = img_element.get_attribute("alt")
                        
                        # alt 속성이 정확히 "방문자리뷰사진"인지 다시 한번 확인
                        if img_url and img_alt == "방문자리뷰사진":
                            img_filename = f"개별방문자리뷰_{i+1:02d}_{j+1:02d}.jpg"
                            downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                            if downloaded_path:
                                review_images.append(downloaded_path)
                        
                except Exception as e:
                    print(f"리뷰 {i+1} 방문자리뷰사진 수집 실패: {e}")
                
                review_info['individual_images'] = review_images
                review_list.append(review_info)
                
                # 진행 상황 출력
                if i % 10 == 0:
                    print(f"리뷰 처리 진행률: {i+1}/{len(review_elements)}")
                
                print(f"리뷰 {i+1}: {review_info['text'][:50]}..." if len(review_info['text']) > 50 else f"리뷰 {i+1}: {review_info['text']}")
                
            except Exception as e:
                print(f"리뷰 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"리뷰 정보 수집 실패: {e}")
        
    print(f"총 {len(review_list)}개의 리뷰 수집 완료")
    return review_list

def go_back_to_list():
    """목록으로 돌아가기"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 검색 결과 iframe으로 다시 전환
        search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
        driver.switch_to.frame(search_iframe)
        print("목록으로 돌아가기 완료")
        return True
        
    except Exception as e:
        print(f"목록으로 돌아가기 실패: {e}")
        return False

# =================== 메인 실행 부분 ===================
try:
    print("=" * 60)
    print(f"네이버 지도 크롤링 시작")
    print(f"검색어: {SEARCH_KEYWORD}")
    print(f"최대 가게 수: {MAX_RESTAURANTS}")
    print(f"가게당 최대 리뷰 수: {MAX_REVIEWS_PER_RESTAURANT}")
    print("=" * 60)
    
    # 1. 검색어 설정 및 접속
    encoded_keyword = urllib.parse.quote(SEARCH_KEYWORD)
    URL = f"https://map.naver.com/p/search/{encoded_keyword}"
    
    print(f"네이버 지도 접속 중: {URL}")
    driver.get(URL)
    time.sleep(5)
    
    # 2. 디렉토리 생성
    base_dir = create_directory_structure(SEARCH_KEYWORD)
    print(f"저장 디렉토리 생성: {base_dir}")
    
    # 3. 검색 결과 iframe 접근
    print("검색 결과 iframe으로 전환 중...")
    search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
    driver.switch_to.frame(search_iframe)
    
    # 4. 가게 리스트 수집
    restaurant_elements = get_restaurant_list()
    total_restaurants = len(restaurant_elements)
    print(f"총 {total_restaurants}개 가게 발견")
    
    # 5. 각 가게별로 세부 정보 수집
    max_restaurants = min(MAX_RESTAURANTS, total_restaurants)
    
    for i in range(max_restaurants):
        print(f"\n{'='*20} 가게 {i+1}/{max_restaurants} {'='*20}")
        
        try:
            # 가게 클릭
            if not click_restaurant(restaurant_elements[i], i+1):
                continue
            
            # 세부 정보 iframe으로 전환
            if not switch_to_detail_iframe():
                continue
            
            # 홈 정보 수집
            print("홈 정보 수집 중...")
            home_info = get_home_info()
            
            # 가게별 폴더 구조 생성
            restaurant_dir, menu_images_dir, review_images_dir = create_restaurant_directories(
                base_dir, home_info.get('name', f'restaurant_{i+1}')
            )
            
            # 메뉴 정보 수집
            menu_info = []
            if find_and_click_menu_tab():
                print("메뉴 정보 수집 중...")
                menu_info = get_menu_info(menu_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("메뉴 탭을 찾을 수 없어 메뉴 정보 수집을 건너뜁니다.")
            
            # 리뷰 정보 수집
            review_info = []
            if find_and_click_review_tab():
                print("리뷰 정보 수집 중...")
                review_info = get_review_info(review_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("리뷰 탭을 찾을 수 없어 리뷰 정보 수집을 건너뜁니다.")
            
            # 전체 정보 결합
            restaurant_data = {
                'index': i + 1,
                'home_info': home_info,
                'menu_info': menu_info,
                'review_info': review_info,
                'restaurant_dir': restaurant_dir,
                'collected_at': datetime.now().isoformat()
            }
            
            all_restaurant_data.append(restaurant_data)
            print(f"가게 {i+1} 정보 수집 완료!")
            print(f"- 메뉴 개수: {len(menu_info)}")
            print(f"- 리뷰 개수: {len(review_info)}")
            
            # 목록으로 돌아가기
            go_back_to_list()
            time.sleep(2)
            
        except Exception as e:
            print(f"가게 {i+1} 처리 중 오류: {e}")
            # 목록으로 돌아가기 시도
            try:
                go_back_to_list()
            except:
                pass
            continue
    
    # 6. 결과 저장
    print(f"\n{'='*60}")
    print(f"총 {len(all_restaurant_data)}개 가게 상세 정보 수집 완료!")
    
    # JSON 파일 저장
    safe_keyword = safe_filename(SEARCH_KEYWORD)
    json_filename = os.path.join(base_dir, f"{safe_keyword}_detailed_restaurants.json")
    with open(json_filename, 'w', encoding='utf-8') as jsonfile:
        json.dump(all_restaurant_data, jsonfile, ensure_ascii=False, indent=2)
    
    # CSV 파일 저장 (한글 헤더로 개선)
    csv_filename = os.path.join(base_dir, f"{safe_keyword}_restaurants_summary.csv")
    with open(csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:  # utf-8-sig로 BOM 추가
        fieldnames = ['순번', '가게명', '분류', '주소', '전화번호', '메뉴수', '리뷰수', '폴더경로']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            row = {
                '순번': restaurant['index'],
                '가게명': restaurant['home_info'].get('name', ''),
                '분류': restaurant['home_info'].get('category', ''),
                '주소': restaurant['home_info'].get('address', ''),
                '전화번호': restaurant['home_info'].get('phone', ''),
                '메뉴수': len(restaurant['menu_info']),
                '리뷰수': len(restaurant['review_info']),
                '폴더경로': restaurant.get('restaurant_dir', '')
            }
            writer.writerow(row)
    
    # 리뷰 상세 정보 CSV 저장
    reviews_csv_filename = os.path.join(base_dir, f"{safe_keyword}_reviews_detail.csv")
    with open(reviews_csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['가게명', '리뷰번호', '리뷰내용', '작성날짜', '평점', '이미지수']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            restaurant_name = restaurant['home_info'].get('name', '')
            for idx, review in enumerate(restaurant['review_info']):
                row = {
                    '가게명': restaurant_name,
                    '리뷰번호': idx + 1,
                    '리뷰내용': review.get('text', ''),
                    '작성날짜': review.get('date', ''),
                    '평점': review.get('rating', ''),
                    '이미지수': len(review.get('individual_images', []))
                }
                writer.writerow(row)
    
    # 통계 정보 출력
    total_menus = sum(len(restaurant['menu_info']) for restaurant in all_restaurant_data)
    total_reviews = sum(len(restaurant['review_info']) for restaurant in all_restaurant_data)
    total_images = 0
    
    # 이미지 파일 개수 세기
    for restaurant in all_restaurant_data:
        restaurant_dir = restaurant.get('restaurant_dir', '')
        if os.path.exists(restaurant_dir):
            for root, dirs, files in os.walk(restaurant_dir):
                image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
                total_images += len(image_files)
    
    print(f"\n파일 저장 완료:")
    print(f"- 상세 정보 (JSON): {json_filename}")
    print(f"- 요약 정보 (CSV): {csv_filename}")
    print(f"- 리뷰 상세 (CSV): {reviews_csv_filename}")
    print(f"- 이미지 폴더: 각 가게별 폴더")
    print(f"\n수집 통계:")
    print(f"- 총 가게 수: {len(all_restaurant_data)}")
    print(f"- 총 메뉴 수: {total_menus}")
    print(f"- 총 리뷰 수: {total_reviews}")
    print(f"- 총 이미지 수: {total_images}")
    
    print(f"\n{'='*60}")
    print("크롤링 완료!")

except Exception as e:
    print(f"전체 프로세스 오류 발생: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    # 드라이버 종료
    try:
        pass
#         driver.quit()
#         print("웹드라이버 종료 완료")
    except:
        pass
end = time.time()
print(f"실행 시간: {end - start:.2f}초")

네이버 지도 크롤링 시작
검색어: 신림동 햄버거
최대 가게 수: 50
가게당 최대 리뷰 수: 500
네이버 지도 접속 중: https://map.naver.com/p/search/%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0
저장 디렉토리 생성: 신림동 햄버거_test50_sam_data
검색 결과 iframe으로 전환 중...
가게 리스트 수집 중...
스크롤 컨테이너 찾음
현재 로드된 가게 수: 10
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
더 이상 로드할 가게가 없음
총 50개 가게 발견

==================== 가게 1/50 ====================
가게 1 클릭 완료
세부 정보 iframe으로 전환 완료
홈 정보 수집 중...
가게명: 아토커피 신림점
분류: 카페,디저트
주소: 서울 관악구 관천로 79 1층
메뉴 탭 찾는 중...
탭 요소들 찾음: 6개
탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1539251371/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081659&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 2: 텍스트='소식', href='https://pcmap.place.naver.com/restaurant/1539251371/feed?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081659&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 3: 텍스트='메뉴', href='https://pcma

  이미지 다운로드 성공: 방문자리뷰_011.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_011.jpg
  이미지 다운로드 성공: 방문자리뷰_012.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_012.jpg
  이미지 다운로드 성공: 방문자리뷰_013.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_013.jpg
  이미지 다운로드 성공: 방문자리뷰_014.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_014.jpg
  이미지 다운로드 성공: 방문자리뷰_015.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_015.jpg
  이미지 다운로드 성공: 방문자리뷰_016.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_016.jpg
  이미지 다운로드 성공: 방문자리뷰_017.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_017.jpg
  이미지 다운로드 성공: 방문자리뷰_018.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_018.jpg
  이미지 다운로드 성공: 방문자리뷰_019.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_019.jpg
  이미지 다운로드 성공: 방문자리뷰_020.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_020.jpg
  이미지 다운로드 성공: 방문자리뷰_021.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_021.jpg
  이미지 다운로드 성공: 방문자리뷰_022.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_022.jpg
  이미지 다운로드 성공: 방문자리뷰_023.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_023.jpg
  이미지 다운로드 성공: 방문자리뷰_024.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_024.jpg
  이미지 다운로드 성공: 방문자리뷰_025.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_025.jpg
  이미지 다운로드 성공: 방문자리뷰_026.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_026.jpg
  이미지 다운로드 성공: 방문자리뷰_027.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_027.j

리뷰 88: 굿
리뷰 89: 아지트가될수있을꺼같은 ^^
  이미지 다운로드 성공: 개별방문자리뷰_90_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_90_02.jpg
리뷰 90: 분위기 정말 좋고 음료랑 디저트도 완전 맛나요💙
이런 서비스를 24시간 즐길수 있다니
대박...
  이미지 다운로드 성공: 개별방문자리뷰_91_01.jpg
리뷰 처리 진행률: 91/110
리뷰 91: 
리뷰 92: 연유라떼 너무너무 밍밍해요.... ㅠ
리뷰 93: 비오는 연휴에 아이들과 디저트 먹으러 방문해보았는데
불고기핫도그도 너무 맛있었고, 와플이 ...
리뷰 94: 지나가는길에 힙한 장소에 나도 모르게 이끌려 들어왔습니다.
사장님 잘생김에 반하고 직원의 ...
리뷰 95: 진짜 너무맛있어요
수제햄버거...
음료 모두진짜
감동 그리구테라스처럼
되어잇어서 아이데리구...
리뷰 96: 굿
리뷰 97: 24시간 이용할 수 있는 곳
새벽시간에도 테이크아웃 되니 너무 좋아요
커피맛 좋아요!
추천...
리뷰 98: 음료가 엄청 진해서 너무 맛있었어요!
리뷰 99: ㅎㅎㅎㅎㅎㅎㅎ
  이미지 다운로드 성공: 개별방문자리뷰_100_01.jpg
리뷰 100: 술 오진탕먹고 갈증해소엔 토마토주스랑 수박주스 최고ㅠㅠ 여지껏 먹어본 과일주스 중에서 최고...
리뷰 처리 진행률: 101/110
리뷰 101: 😬😬😬😬😬
리뷰 102: 
리뷰 103: 매장이 너무 이쁘고 집중하기 좋네요!
리뷰 104: 첫 방문 본이 아니게 오래 있다가 감. 혼자 2시간을 어떻게 기다리지 했는데 음악이 좋아서...
리뷰 105: ㅎ
리뷰 106: 굿
리뷰 107: 대용량커피 좋아요
리뷰 108: 굿
리뷰 109: 붐위기기 좋아요 좋아요
리뷰 110: 머그컵 손잡이가 겁나 불푠하지만
커피는 맛있어요
총 110개의 리뷰 수집 완료
가게 1 정보 수집 완료!
- 메뉴 개수: 74
- 리뷰 개수: 110
목록으로 돌아가기 완료

==================== 가게 2/50 ====================
가게 2 

  이미지 다운로드 성공: 개별방문자리뷰_03_04.jpg
리뷰 3: 새로 생겨서 궁금해서 와봤는데 매장도 깔끔하고 맛있는 수제버거 메뉴가 많아요!
패티자체도 ...
  이미지 다운로드 성공: 개별방문자리뷰_04_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_04_02.jpg
  이미지 다운로드 성공: 개별방문자리뷰_04_03.jpg
  이미지 다운로드 성공: 개별방문자리뷰_04_04.jpg
  이미지 다운로드 성공: 개별방문자리뷰_04_05.jpg
리뷰 4: 신림 버거 맛집 버거 맛탱입니다 고기 너무 푸짐하고 맛있고 아메리칸 버거 정말 좋아해서 당...
  이미지 다운로드 성공: 개별방문자리뷰_05_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_05_02.jpg
  이미지 다운로드 성공: 개별방문자리뷰_05_03.jpg
리뷰 5: 자극적이지 않고 맛있는 수제버거! 🍔 해쉬딥치킨버거는 두툼하고 약간 매콤해서 맵찔이도 가능...
  이미지 다운로드 성공: 개별방문자리뷰_06_01.jpg
리뷰 6: 베이컨해쉬치즈버거를 먹었어요. 이벤트가 있어서 세트를 할인받아 먹었어요. 버거 엄청 두께가...
  이미지 다운로드 성공: 개별방문자리뷰_07_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_07_02.jpg
리뷰 7: 이제 신림에서도 미쿡의 맛을 느낄 수 있게되었어요! 패티 육즙 가득, 햄버거 번도 맛있었고...
리뷰 8: 보통 햄버거는 사진이랑 실제가 차이가 많이 나는 경우가 흔한데, 여긴 똑같네요ㅎㅎ 알찹니다...
  이미지 다운로드 성공: 개별방문자리뷰_09_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_09_02.jpg
  이미지 다운로드 성공: 개별방문자리뷰_09_03.jpg
  이미지 다운로드 성공: 개별방문자리뷰_09_04.jpg
  이미지 다운로드 성공: 개별방문자리뷰_09_05.jpg
리뷰 9: 햄버거 너무 맛있었습니다ㅠㅠ 가격대비 퀄리티가 너무 좋은 집이었어요! 또 방문하고 싶은 곳...
  이미지 다운로드 성공: 개별방문

리뷰 탭 클릭 완료
리뷰 정보 수집 중...
리뷰 더보기 버튼들 찾는 중...
더보기 버튼 찾기 시도 1/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (1번째)
더보기 버튼 찾기 시도 2/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (2번째)
더보기 버튼 찾기 시도 3/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (3번째)
더보기 버튼 찾기 시도 4/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (4번째)
더보기 버튼 찾기 시도 5/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (5번째)
더보기 버튼 찾기 시도 6/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (6번째)
더보기 버튼 찾기 시도 7/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (7번째)
더보기 버튼 찾기 시도 8/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (8번째)
더보기 버튼 찾기 시도 9/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (9번째)
더보기 버튼 찾기 시도 10/10
발견된 버튼 텍스트: '더보기'
더보기 버튼 클릭 완료 (10번째)
총 10개의 더보기 버튼을 클릭했습니다
방문자리뷰사진만 수집 시작 (프로필 영역 제외)...
전체 방문자리뷰사진 발견: 18개
프로필 영역 이미지 발견: 118개 (제외 예정)
  이미지 다운로드 성공: 방문자리뷰_001.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_001.jpg
  이미지 다운로드 성공: 방문자리뷰_002.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_002.jpg
  이미지 다운로드 성공: 방문자리뷰_003.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_003.jpg
  이미지 다운로드 성공: 방문자리뷰_004.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_004.jpg
  이미지 다운로드 성공: 방문자리뷰_005.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_005.jpg
  이미지 다운로드

...

리뷰 25: 친절하고 신속한 서비스가 좋았어요
리뷰 26: 핫 크리스피 통다리 존맛탱
리뷰 27: 아이스커피 핫커피 둘 다 같은 컵에 줘서 헷갈렸어요
리뷰 28: 
리뷰 29: 주문 누락되고 한참 기다려도 안나와서 물어보면 사과와 함께 빨리 드린다고 하는게 정상인데 ...
리뷰 30: 런치킨박스 맛 굿
  이미지 다운로드 성공: 개별방문자리뷰_31_01.jpg
리뷰 처리 진행률: 31/110
리뷰 31: 징거타워세트 맛있어요
리뷰 32: 
리뷰 33: 런치박스 메뉴가 키오스크에서 음료 변경이 안돼서 구두로 여쭤보니 정말 귀찮은 눈빛과 말투로...
리뷰 34: 맛있어요
리뷰 35: 치밥이 좋아서 자주 이용하고 있습니다
리뷰 36: 런치킨박스 맛있어요
리뷰 37: 매장이 넓고 음식이 빨리 나와요
리뷰 38: 
리뷰 39: 좋아요
리뷰 40: 행사 있을때 먹기 좋아요
  이미지 다운로드 성공: 개별방문자리뷰_41_01.jpg
리뷰 처리 진행률: 41/110
리뷰 41: 원플원해서 먹음 맛있어요
리뷰 42: ㅇ
리뷰 43: 1인용 좌석이 많아 혼자 이용하기 좋습니당
리뷰 44: 켄티밥 너무 맛있음. 추천!
리뷰 45: KFC 치킨 넘 맛있어요💙
리뷰 46: 맛있어요 저렴하고 간단하게 즐기기 굳굳
리뷰 47: 너무 맛있어요
리뷰 48: 커넬오리지널타워팩 +프렌치프라이 먹었는데
배터지는줄요ㅎㅎ타워팩세트구성좋은데 버거2개중 하나...
리뷰 49: 굳굳굳 ㅎㅎ
리뷰 50: 
  이미지 다운로드 성공: 개별방문자리뷰_51_01.jpg
리뷰 처리 진행률: 51/110
리뷰 51: 
리뷰 52: 맛있어요
리뷰 53: 올데이라 갔는데 치킨 너무 늦게 나옴.
40분 기다림...
리뷰 54: 조아요
리뷰 55: 치밥 최고입니다
리뷰 56: 역시 치킨이 맛있어요.
리뷰 57: 좋아요
리뷰 58: 치밥 맛있어요!
리뷰 59: 양념치킨치밥 맛있네요
리뷰 60: 오리지널 항상 맛있음
  이미지 다운로드 성공: 개별방문자리뷰_61_01.jpg
리뷰 처리 진행률: 61/110
리뷰 61: ガーリ

리뷰 17: 새로 오픈해서 와퍼를 빅세일하네요. 하지만 맛은 똑같아서 맛있어요. 오픈 세일 때문에 직원...
리뷰 18: 설문후 업그레이드 쿠폰= 3천원 할인
야채가 부족해요
콜라 텀블러 픽업했어요. 키오스크 선...
리뷰 19: 매장 넓고 진짜 시원합니다!!!!!!!!!
주문한 것도 금방금방 나와요
리뷰 20: 동네에 생겨서 방문해봅니다.
23시에 버거라니^^;;
  이미지 다운로드 성공: 개별방문자리뷰_21_01.jpg
리뷰 처리 진행률: 21/76
리뷰 21: 집앞에 버거킹 있으니깐

너~무 좋다 스윽 걸어나와서

햄버거 냠냠.
더보기
리뷰 22: 
리뷰 23: 동네에 생겨서 다녀왔어요 종종생각남 직접 먹으러
리뷰 24: 햄버거 좋아하는 내겐 일상ㅎ
통창에 뷰도 트여있고 좋아요:)
리뷰 25: 굳
리뷰 26: 굿
리뷰 27: 가성비 좋고 버가가 맛있습니다
리뷰 28: 매장이 테이블 과 바닥이 끈적거려요 힘내세요
리뷰 29: 감튀랑 햄버거도 맛있고 좋아요..
리뷰 30: 쾌적하고 좋아요:)
  이미지 다운로드 성공: 개별방문자리뷰_31_01.jpg
리뷰 처리 진행률: 31/76
리뷰 31: 햄버거 좋아해서 주1회는 오는듯요ㅎㅎ
리뷰 32: 굳
리뷰 33: 처음 매장 가봤는데 사람 많음 좀 기달려야 됨 음식은 맛있음👍🏻👍🏻
리뷰 34: 
리뷰 35: 가성비 좋은 햄버거 좋아요~
5월은 할인행사가 있어 콰치와주+통새우와주
할인으로 즐길수 있...
리뷰 36: 맛있어요!!배달시간도짧아요♡
리뷰 37: 버거킹 와퍼 맛나요
자주 들릴게요
리뷰 38: 오픈날에 방문했는데 영수증에 설문조사코드 안되서 현장에서 물어봤었는데 다음날 등록하면 될거...
리뷰 39: 맛있어요!! 배달도시간정확하게왔어요♡
리뷰 40: 
리뷰 처리 진행률: 41/76
리뷰 41: 소스가 더 많았으면 좋겠어요~
리뷰 42: 키오스크 4대 있어서 편함
매장 넓음
리뷰 43: 쿠폰받아서 먹었는데 단짠단짠 맛있게 먹었어요 삼성페이가 안되는게 아쉬웠어요
리뷰 44: 새로 생겨서 좋네요
리뷰 45: 가격이

리뷰 77: 버거킹
지점마다 미세하게맛이다르다고생각하는데
개인적으로
신대방삼거리지점과 교대점을 선호해요...
리뷰 78: 맛있어요
리뷰 79: 행사해서 조금 저렴하게 먹었어요
리뷰 80: good~!!
  이미지 다운로드 성공: 개별방문자리뷰_81_01.jpg
리뷰 처리 진행률: 81/110
리뷰 81: 
리뷰 82: 
리뷰 83: 버거가 너무 작아요
리뷰 84: 쿠폰을 사용하려고 방문했습니다 키오스크가 많아서 기다림 없이 주문할 수 있어요 햄버거 감튀...
리뷰 85: good~!!
리뷰 86: 햄버거에 토마토 들어가 있는걸 좋아하는 사람으로
토마토 추가(0원)를 항상한다
맛있습니다
리뷰 87: 매장내 혼밥하기 괜찮아요.
사람은 많지만 자리도 넉넉해서 바로 앉아서 먹을수 있어요. 매장...
리뷰 88: 
리뷰 89: 맛있어요
리뷰 90: 
  이미지 다운로드 성공: 개별방문자리뷰_91_01.jpg
리뷰 처리 진행률: 91/110
리뷰 91: good~!!
리뷰 92: 저녁먹고 출출해서 간식먹어요.
와퍼주니어 넘 작아요
리뷰 93: 굿~~~~
리뷰 94: 매장도 넓고 햄버거는 역시 버거킹이죠!!!
리뷰 95: 통새우와퍼 맛있어요
리뷰 96: 
리뷰 97: 버거킹은 자주가는곳
햄버거 감자 맛있어요
리뷰 98: 
리뷰 99: good~!!
리뷰 100: 오징어게임을 접목해서 팔던데 재미있네요.
리뷰 처리 진행률: 101/110
리뷰 101: 좋아요
리뷰 102: 굿~~~~~
리뷰 103: 버거는...버거킹♡
리뷰 104: 햄버거는 버거킹이죠
리뷰 105: 무난하게 가기 좋아용 ㅎㅎ
리뷰 106: 햄버거 너무 작아요
리뷰 107: 굿
리뷰 108: 
리뷰 109: 10월 31일 목요일 신대방삼거리 버거킹. 선데이 2,200원. 치즈와퍼쮸니어. 밤에도 손...
리뷰 110: 맛있어요
총 110개의 리뷰 수집 완료
가게 47 정보 수집 완료!
- 메뉴 개수: 12
- 리뷰 개수: 110
목록으로 돌아가기 완료

==================== 가게 48/50 ==

리뷰 처리 진행률: 81/110
리뷰 81: 쉐이크 다 맛있음.
리뷰 82: 수제버거 머쉬룸버거에 채소추가해서 맛있게 먹었어요
리뷰 83: 컨트리버거2개,칠리치즈프라이세트.베이컨렌치프라이세트.
맛 보장된 메뉴입니다. 재료 뭐뭐 들...
리뷰 84: 동네 버거 맛집!
가격이 좀 있어도 맛있어서 또 사먹고 싶은 맛입니다. 매장 좀 여기저기 ...
리뷰 85: 버거 맛있어요!
리뷰 86: 오~~ 햄버거 맛집입니당~~!!!
리뷰 87: 
리뷰 88: 매장 깔끔해서 좋아요. 너무 맛있어요. 매장에 계신 분들이 매우 친절해서 좋아요. 신랑이 ...
리뷰 89: 프랜차이즈 버거에 비해 가격이 좀 비싸긴한데 버거가 정말 맛있어요. 감튀는 매장에서 직접 ...
리뷰 90: 맛있고 좋은데 양상추가 채썰어져 있어 먹기 불편ㅠㅠ환기가 너무 안되서 옷에 냄새가 가득.....
리뷰 처리 진행률: 91/110
리뷰 91: 맛있어요
리뷰 92: 맛있어요
리뷰 93: 리얼 고급진 수제버거 맛집
음료수 무한리필은 덤
리뷰 94: 인테리어가 너무 수수해서 들어가기 좀 망설여졌어요. 그런데 맛에 집중한 곳이었군요. 패티 ...
리뷰 95: 버거가 맛있어요
리뷰 96: 항상 너무 맛있음
리뷰 97: 육즙 가득하고 맛있어요! 살짝 덜익긴했는데 먹었더니 맛있었어요ㅎㅎ
어니언링 맛있는데 기름기...
리뷰 98: 언제나 맛있는 이 동네의 hidden gem. 늘 번창하세요:)
리뷰 99: 넘나 맛있어요 햄버거 짱짱 존맛탱~~
리뷰 100: 미국냄새남
리뷰 처리 진행률: 101/110
리뷰 101: 진짜 이 지역 최고 버거 맛집!!!
리뷰 102: 굳
리뷰 103: 머쉬룸버거 먹었는데 JMT
매장도 깔끔하고 맛있어서 좋아요
종종 수제버거 먹으러 가야겠오요...
리뷰 104: 기대안하고 간단히 먹으러 간 곳인데
너무 맛있어서 놀랬습니다.
리뷰 105: 정말맛있어요~
처음이라 치즈버거시켰는데 패티도 육즙가득에 채소도 신선하네요^^
자주 방문할...
리뷰 106: 동네 이런 곳이 있었네요~~과하지 않고 맛있었어

프로필 영역 이미지 발견: 162개 (제외 예정)
  이미지 다운로드 성공: 방문자리뷰_001.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_001.jpg
  이미지 다운로드 성공: 방문자리뷰_002.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_002.jpg
  이미지 다운로드 성공: 방문자리뷰_003.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_003.jpg
  이미지 다운로드 성공: 방문자리뷰_004.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_004.jpg
방문자리뷰사진 다운로드 완료: 총 4개
프로필 영역 이미지 제외: 0개
최종 스크롤로 모든 리뷰 로드 중...
리뷰 요소 찾음: li.place_apply_pui.EjjAW - 110개
총 110개의 리뷰 처리 시작
리뷰 처리 진행률: 1/110
리뷰 1: 파파이스 골든 넘버세트 12900원/루이 골든 휠레 버거6200원/슈프링클 치킨 1조각39...
리뷰 2: 루이버거 진짜 맛있어요, 음료 체리콕 추천합니다!!!!!
리뷰 3: 처음 먹어보는 파파이스였는데
가격 대비 조금 아쉬웠어요.

역시 맘스터치, 맥도날드, 버거...
리뷰 4: 파파이스 오랜만에 갔습니다^^
역시 비스켓 꼭 먹어줘야 하거든요...
리뷰 5: 호다닥 먹기 편해요
리뷰 6: 20분 넘게 기다렸는데 주문이 안들어갔다고 대응. 추가로 13분 더 기다림
리뷰 7: 치킨 신선 바삭하고 따뜻해서 너무 맛있어요!
리뷰 8: 파파이스! 감튀가 좀 느끼한 편이네..
리뷰 9: 맛있어요.
리뷰 10: 매장이 많이 없어서 자주 못 먹는데 오랜만에 먹으니
더 맛있었습니다. 튀김도 바삭바삭 하고...
  이미지 다운로드 성공: 개별방문자리뷰_11_01.jpg
  이미지 다운로드 성공: 개별방문자리뷰_11_02.jpg
리뷰 처리 진행률: 11/110
리뷰 11: 2인석도 많아서 혼밥 하긴 좋은데요.
제가 콜라를 흘린 것도 아닌데 햄버거 밑에 콜라로 젖...
리뷰 12: 클래식 치킨 샌드위치 맛있어요 기본의 맛
리뷰 13: 오늘은 버거나 치킨사이즈가 좀

In [18]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.service import Service
import requests
import time
import os
import json
import csv
import urllib.parse
import re
from datetime import datetime
from PIL import Image
import io

# =================== 설정 구역 ===================
# 여기서 검색어와 기타 설정을 변경하세요
SEARCH_KEYWORD = "신림동 햄버거"  # 원하는 검색어로 변경
MAX_RESTAURANTS = 1  # 크롤링할 최대 가게 수
MAX_REVIEWS_PER_RESTAURANT = 10  # 가게당 최대 리뷰 수
# ================================================

# 서비스 설정
service = Service(port=9999)
# 크롬 인터넷 설정
options = webdriver.ChromeOptions()
options.add_argument('user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36')
options.add_argument('window-size=1380,900')
# 드라이버 생성
driver = webdriver.Chrome(service=service, options=options)
# WebDriverWait 초기화
wait = WebDriverWait(driver, 10)

# 결과 저장할 리스트
all_restaurant_data = []

def create_directory_structure(keyword):
    """가게별 이미지 저장용 디렉토리 구조 생성"""
    base_dir = f"{keyword}_테스트(1개)_data"
    os.makedirs(base_dir, exist_ok=True)
    return base_dir

def create_restaurant_directories(base_dir, restaurant_name):
    """가게별 폴더 구조 생성"""
    safe_name = safe_filename(restaurant_name)
    restaurant_dir = os.path.join(base_dir, safe_name)
    menu_images_dir = os.path.join(restaurant_dir, "메뉴_이미지")
    review_images_dir = os.path.join(restaurant_dir, "리뷰_이미지")
    
    os.makedirs(menu_images_dir, exist_ok=True)
    os.makedirs(review_images_dir, exist_ok=True)
    
    return restaurant_dir, menu_images_dir, review_images_dir

def download_image_improved(url, filename, images_dir):
    """이미지 다운로드 (개선된 버전 - 이미지 검증 포함)"""
    try:
        # User-Agent 헤더 추가
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36',
            'Referer': 'https://map.naver.com/'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code == 200:
            # 이미지 데이터 검증
            try:
                # PIL로 이미지 검증
                img = Image.open(io.BytesIO(response.content))
                img.verify()  # 이미지가 유효한지 검증
                
                # 파일 저장
                filepath = os.path.join(images_dir, filename)
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                
                print(f"  이미지 다운로드 성공: {filename}")
                return filepath
            except Exception as img_error:
                print(f"  이미지 검증 실패 ({filename}): {img_error}")
                return None
        else:
            print(f"  HTTP 오류 ({filename}): {response.status_code}")
            return None
    except Exception as e:
        print(f"  이미지 다운로드 실패 ({filename}): {e}")
        return None

def safe_filename(filename):
    """파일명에서 특수문자 제거"""
    return re.sub(r'[^\w\s-]', '', filename).strip()[:50]

def get_restaurant_list():
    """왼쪽 패널에서 가게 리스트 수집"""
    print("가게 리스트 수집 중...")
    
    # 스크롤하여 모든 가게 로드
    try:
        scroll_container = driver.find_element(By.CSS_SELECTOR, "div.Ryr1F")
        print("스크롤 컨테이너 찾음")
        
        previous_count = 0
        max_attempts = 15
        
        for attempt in range(max_attempts):
            restaurant_elements = driver.find_elements(By.CSS_SELECTOR, "li.UEzoS")
            current_count = len(restaurant_elements)
            
            print(f"현재 로드된 가게 수: {current_count}")
            
            if current_count == previous_count:
                if attempt >= 3:
                    print("더 이상 로드할 가게가 없음")
                    break
            else:
                previous_count = current_count
            
            # 스크롤 실행
            driver.execute_script("arguments[0].scrollTop = arguments[0].scrollHeight", scroll_container)
            time.sleep(2)
        
        return restaurant_elements
        
    except Exception as e:
        print(f"가게 리스트 수집 실패: {e}")
        return []

def click_restaurant(restaurant_element, index):
    """가게 클릭하여 세부 정보 패널 열기"""
    try:
        # 가게 링크 찾기
        link_element = restaurant_element.find_element(By.CSS_SELECTOR, "a.place_bluelink")
        
        # 스크롤하여 요소가 보이도록 함
        driver.execute_script("arguments[0].scrollIntoView(true);", link_element)
        time.sleep(1)
        
        # 클릭
        ActionChains(driver).move_to_element(link_element).click().perform()
        print(f"가게 {index} 클릭 완료")
        
        # 세부 정보 패널 로딩 대기
        time.sleep(3)
        return True
        
    except Exception as e:
        print(f"가게 {index} 클릭 실패: {e}")
        return False

def switch_to_detail_iframe():
    """오른쪽 세부 정보 iframe으로 전환"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 세부 정보 iframe 찾기
        detail_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#entryIframe")))
        driver.switch_to.frame(detail_iframe)
        print("세부 정보 iframe으로 전환 완료")
        return True
        
    except Exception as e:
        print(f"세부 정보 iframe 전환 실패: {e}")
        return False

def get_home_info():
    """홈 탭에서 기본 정보 수집"""
    home_info = {}
    
    try:
        # 가게 이름
        name_element = driver.find_element(By.CSS_SELECTOR, "span.GHAhO")
        home_info['name'] = name_element.text.strip()
        print(f"가게명: {home_info['name']}")
        
        # 가게 분류
        category_element = driver.find_element(By.CSS_SELECTOR, "span.lnJFt")
        home_info['category'] = category_element.text.strip()
        print(f"분류: {home_info['category']}")
        
        # 가게 주소
        try:
            address_element = driver.find_element(By.CSS_SELECTOR, "span.LDgIH")
            home_info['address'] = address_element.text.strip()
            print(f"주소: {home_info['address']}")
        except:
            home_info['address'] = "주소 정보 없음"
        
        # 전화번호 (있다면)
        try:
            phone_element = driver.find_element(By.CSS_SELECTOR, "span.xlx7Q")
            home_info['phone'] = phone_element.text.strip()
        except:
            home_info['phone'] = "전화번호 정보 없음"
        
        # 영업시간 (있다면)
        try:
            hours_element = driver.find_element(By.CSS_SELECTOR, "time.H3ua4")
            home_info['hours'] = hours_element.text.strip()
        except:
            home_info['hours'] = "영업시간 정보 없음"
            
    except Exception as e:
        print(f"홈 정보 수집 실패: {e}")
        
    return home_info

def find_and_click_menu_tab():
    """메뉴 탭을 정확하게 찾아서 클릭"""
    try:
        print("메뉴 탭 찾는 중...")
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 메뉴 탭 찾기
        menu_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 메뉴 탭인지 판단
                is_menu_tab = (
                    "menu" in tab_href.lower() or
                    "메뉴" in tab_text or
                    "menu" in tab_text
                )
                
                if is_menu_tab:
                    menu_tab = tab
                    print(f"메뉴 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 메뉴 탭 클릭
        if menu_tab:
            try:
                # 스크롤하여 요소가 보이도록 함
                driver.execute_script("arguments[0].scrollIntoView(true);", menu_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", menu_tab)
                time.sleep(3)
                print("메뉴 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"메뉴 탭 클릭 실패: {e}")
                return False
        else:
            print("메뉴 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"메뉴 탭 찾기 실패: {e}")
        return False

def get_menu_info(menu_images_dir, restaurant_name):
    """메뉴 탭에서 메뉴 정보 수집"""
    menu_list = []
    
    try:
        # 메뉴 더보기 버튼 클릭
        try:
            more_button_selectors = [
                "a.fvwqf",
                "button.fvwqf",
                "a[class*='more']",
                "//a[contains(text(), '더보기')]"
            ]
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_button = driver.find_element(By.XPATH, selector)
                    else:
                        more_button = driver.find_element(By.CSS_SELECTOR, selector)
                    
                    if more_button.is_displayed() and more_button.is_enabled():
                        driver.execute_script("arguments[0].scrollIntoView(true);", more_button)
                        time.sleep(1)
                        driver.execute_script("arguments[0].click();", more_button)
                        time.sleep(2)
                        print("메뉴 더보기 버튼 클릭 완료")
                        break
                except:
                    continue
        except:
            print("메뉴 더보기 버튼이 없거나 이미 모든 메뉴가 표시됨")
        
        # 메뉴 항목들 찾기
        menu_elements = driver.find_elements(By.CSS_SELECTOR, "li.E2jtL")
        
        for i, menu_element in enumerate(menu_elements):
            menu_info = {}
            
            try:
                # 메뉴명
                name_element = menu_element.find_element(By.CSS_SELECTOR, "span.lPzHi")
                menu_info['name'] = name_element.text.strip()
                
                # 메뉴 가격
                try:
                    price_element = menu_element.find_element(By.CSS_SELECTOR, "div.GXS1X em")
                    menu_info['price'] = price_element.text.strip()
                except:
                    menu_info['price'] = "가격 정보 없음"
                
                # 메뉴 설명
                try:
                    desc_element = menu_element.find_element(By.CSS_SELECTOR, "div.TRxGt")
                    menu_info['description'] = desc_element.text.strip()
                except:
                    menu_info['description'] = "설명 없음"
                
                # 메뉴 이미지
                try:
                    img_selectors = ["img.K0PDV", "img"]
                    img_element = None
                    
                    for selector in img_selectors:
                        try:
                            img_element = menu_element.find_element(By.CSS_SELECTOR, selector)
                            break
                        except:
                            continue
                    
                    if img_element:
                        img_url = img_element.get_attribute("src")
                        if img_url:
                            # 이미지 파일명 생성 (특수문자 제거)
                            safe_menu_name = safe_filename(menu_info['name'])[:20]
                            img_filename = f"메뉴_{i+1:02d}_{safe_menu_name}.jpg"
                            
                            # 이미지 다운로드
                            downloaded_path = download_image_improved(img_url, img_filename, menu_images_dir)
                            menu_info['image_path'] = downloaded_path
                        else:
                            menu_info['image_path'] = None
                    else:
                        menu_info['image_path'] = None
                except:
                    menu_info['image_path'] = None
                
                menu_list.append(menu_info)
                print(f"메뉴 {i+1}: {menu_info['name']} - {menu_info['price']}")
                
            except Exception as e:
                print(f"메뉴 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"메뉴 정보 수집 실패: {e}")
    
    return menu_list

def find_and_click_review_tab():
    """리뷰 탭을 정확하게 찾아서 클릭"""
    try:
        print("리뷰 탭 찾는 중...")
        
        # 페이지 스크롤하여 탭이 보이도록 함
        driver.execute_script("window.scrollTo(0, 0);")
        time.sleep(1)
        
        # 1단계: 모든 탭 요소들 찾기
        tab_selectors = [
            "a._tab-menu",
            "div.YYh8o a",
            "div[class*='tab'] a",
            "nav a",
            "ul li a"
        ]
        
        all_tabs = []
        for selector in tab_selectors:
            try:
                tabs = driver.find_elements(By.CSS_SELECTOR, selector)
                if tabs:
                    all_tabs = tabs
                    print(f"탭 요소들 찾음: {len(tabs)}개")
                    break
            except:
                continue
        
        if not all_tabs:
            print("탭 요소를 찾을 수 없습니다")
            return False
        
        # 2단계: 각 탭의 텍스트와 URL 확인하여 리뷰 탭 찾기
        review_tab = None
        for i, tab in enumerate(all_tabs):
            try:
                # 탭 텍스트 확인
                tab_text = tab.text.strip().lower()
                # href 속성 확인
                tab_href = tab.get_attribute("href") or ""
                
                print(f"탭 {i+1}: 텍스트='{tab_text}', href='{tab_href}'")
                
                # 리뷰 탭인지 판단
                is_review_tab = (
                    "review" in tab_href.lower() or
                    "리뷰" in tab_text or
                    "review" in tab_text
                )
                
                if is_review_tab:
                    review_tab = tab
                    print(f"리뷰 탭 발견: {tab_text}")
                    break
                    
            except Exception as e:
                print(f"탭 {i+1} 확인 중 오류: {e}")
                continue
        
        # 3단계: 리뷰 탭 클릭
        if review_tab:
            try:
                # 요소가 보이도록 스크롤
                driver.execute_script("arguments[0].scrollIntoView(true);", review_tab)
                time.sleep(1)
                
                # JavaScript로 클릭
                driver.execute_script("arguments[0].click();", review_tab)
                time.sleep(3)
                print("리뷰 탭 클릭 완료")
                return True
                
            except Exception as e:
                print(f"리뷰 탭 클릭 실패: {e}")
                return False
        else:
            print("리뷰 탭을 찾을 수 없습니다")
            return False
            
    except Exception as e:
        print(f"리뷰 탭 찾기 실패: {e}")
        return False

def click_all_review_more_buttons():
    """모든 리뷰 더보기 버튼 클릭 (개선된 버전)"""
    try:
        print("리뷰 더보기 버튼들 찾는 중...")
        
        # 지정된 셀렉터로 더보기 버튼 찾기
        more_button_selectors = [
            "div.lfH3O.fvwqf",  # 사용자가 제공한 정확한 셀렉터
            "div.fvwqf",
            "a.fvwqf",
            "button.fvwqf",
            "//a[contains(text(), '더보기')]",
            "//button[contains(text(), '더보기')]",
            "//div[contains(text(), '더보기')]"
        ]
        
        clicked_count = 0
        max_attempts = 10  # 최대 10번 시도
        
        for attempt in range(max_attempts):
            print(f"더보기 버튼 찾기 시도 {attempt + 1}/{max_attempts}")
            
            button_found = False
            
            for selector in more_button_selectors:
                try:
                    if selector.startswith("//"):
                        more_buttons = driver.find_elements(By.XPATH, selector)
                    else:
                        more_buttons = driver.find_elements(By.CSS_SELECTOR, selector)
                    
                    for button in more_buttons:
                        if button.is_displayed() and button.is_enabled():
                            try:
                                # 버튼이 보이도록 스크롤
                                driver.execute_script("arguments[0].scrollIntoView(true);", button)
                                time.sleep(1)
                                
                                # 버튼 텍스트 확인
                                button_text = button.text.strip()
                                print(f"발견된 버튼 텍스트: '{button_text}'")
                                
                                if "더보기" in button_text or "more" in button_text.lower():
                                    # JavaScript로 클릭
                                    driver.execute_script("arguments[0].click();", button)
                                    time.sleep(3)  # 로딩 대기
                                    clicked_count += 1
                                    button_found = True
                                    print(f"더보기 버튼 클릭 완료 ({clicked_count}번째)")
                                    break
                            except Exception as click_error:
                                print(f"버튼 클릭 실패: {click_error}")
                                continue
                    
                    if button_found:
                        break
                        
                except Exception as e:
                    continue
            
            if not button_found:
                print("더 이상 더보기 버튼을 찾을 수 없음")
                break
                
            # 페이지 하단으로 스크롤하여 새로운 리뷰 로드 확인
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
        
        print(f"총 {clicked_count}개의 더보기 버튼을 클릭했습니다")
        return clicked_count > 0
        
    except Exception as e:
        print(f"리뷰 더보기 버튼 처리 실패: {e}")
        return False

def collect_review_images_with_slide(restaurant_name, review_images_dir):
    """리뷰 이미지 수집 - 프로필 영역 제외하고 방문자리뷰사진만 수집"""
    downloaded_images = []
    
    try:
        print("방문자리뷰사진만 수집 시작 (프로필 영역 제외)...")
        
        # 전체 페이지에서 alt="방문자리뷰사진"인 이미지들 찾기
        all_visitor_images = driver.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
        print(f"전체 방문자리뷰사진 발견: {len(all_visitor_images)}개")
        
        # 프로필 영역의 이미지들 제외
        profile_containers = driver.find_elements(By.CSS_SELECTOR, 'div.pui__q2fg8o.pui__A7NplK')
        profile_images = []
        
        for container in profile_containers:
            container_images = container.find_elements(By.CSS_SELECTOR, 'img')
            profile_images.extend(container_images)
        
        print(f"프로필 영역 이미지 발견: {len(profile_images)}개 (제외 예정)")
        
        total_images = 0
        excluded_count = 0
        
        for img_idx, img in enumerate(all_visitor_images):
            try:
                # 이 이미지가 프로필 영역에 속하는지 확인
                is_profile_image = False
                for profile_img in profile_images:
                    try:
                        if (img.get_attribute("src") == profile_img.get_attribute("src") or
                            driver.execute_script("return arguments[0].contains(arguments[1])", 
                                                profile_containers[profile_images.index(profile_img) // 10 if profile_images.index(profile_img) < len(profile_containers) * 10 else 0], img)):
                            is_profile_image = True
                            break
                    except:
                        continue
                
                # 프로필 영역의 이미지가 아닌 경우에만 다운로드
                if not is_profile_image:
                    img_url = img.get_attribute("src")
                    img_alt = img.get_attribute("alt")
                    
                    # alt 속성이 정확히 "방문자리뷰사진"인지 다시 한번 확인
                    if img_url and img_alt == "방문자리뷰사진":
                        filename = f"방문자리뷰_{total_images+1:03d}.jpg"
                        
                        # 이미지가 이미 다운로드되었는지 확인 (URL 기준)
                        if img_url not in [img_info.get('url') for img_info in downloaded_images]:
                            downloaded_path = download_image_improved(img_url, filename, review_images_dir)
                            if downloaded_path:
                                downloaded_images.append({
                                    'path': downloaded_path,
                                    'url': img_url,
                                    'filename': filename,
                                    'type': 'visitor_review'
                                })
                                total_images += 1
                                print(f"  방문자리뷰사진 다운로드: {filename}")
                else:
                    excluded_count += 1
                    print(f"  프로필 영역 이미지 제외: {excluded_count}개")
            
            except Exception as e:
                print(f"  이미지 {img_idx+1} 처리 실패: {e}")
                continue
        
        print(f"방문자리뷰사진 다운로드 완료: 총 {total_images}개")
        print(f"프로필 영역 이미지 제외: {excluded_count}개")
        return [img['path'] for img in downloaded_images]
        
    except Exception as e:
        print(f"방문자리뷰사진 수집 실패: {e}")
        return []

def get_review_info(review_images_dir, restaurant_name, max_reviews=None):
    """리뷰 탭에서 리뷰 정보 수집 (개선된 버전)"""
    if max_reviews is None:
        max_reviews = MAX_REVIEWS_PER_RESTAURANT
    
    review_list = []
    
    try:
        # 모든 리뷰 더보기 버튼 클릭
        click_all_review_more_buttons()
        
        # 리뷰 이미지 수집 (슬라이드 포함)
        review_slide_images = collect_review_images_with_slide(restaurant_name, review_images_dir)
        
        # 최종 스크롤하여 모든 리뷰 로드
        print("최종 스크롤로 모든 리뷰 로드 중...")
        last_height = driver.execute_script("return document.body.scrollHeight")
        
        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(2)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height
        
        # 리뷰 항목들 찾기 (여러 셀렉터 시도)
        review_selectors = [
            "li.place_apply_pui.EjjAW",
            "li.place_apply_pui",
            "div.pui__vn15t2",
            "li[class*='review']",
            "div[class*='review']"
        ]
        
        review_elements = []
        for selector in review_selectors:
            try:
                elements = driver.find_elements(By.CSS_SELECTOR, selector)
                if elements:
                    review_elements = elements[:max_reviews]
                    print(f"리뷰 요소 찾음: {selector} - {len(review_elements)}개")
                    break
            except:
                continue
        
        if not review_elements:
            print("리뷰 요소를 찾을 수 없습니다.")
            return review_list
        
        print(f"총 {len(review_elements)}개의 리뷰 처리 시작")
        
        for i, review_element in enumerate(review_elements):
            review_info = {}
            
            try:
                # 리뷰 텍스트 - 정확한 셀렉터 사용
                review_text = "리뷰 텍스트 없음"
                try:
                    # div.pui__vn15t2에서 리뷰 텍스트 찾기
                    text_element = review_element.find_element(By.CSS_SELECTOR, "div.pui__vn15t2")
                    review_text = text_element.text.strip()
                    if not review_text or len(review_text) < 5:
                        # 다른 텍스트 셀렉터들도 시도
                        text_selectors = [
                            "span.zPfVt", 
                            "div.pui__vn15t2 span",
                            "span[class*='review']", 
                            "div[class*='content']"
                        ]
                        
                        for text_selector in text_selectors:
                            try:
                                text_elements = review_element.find_elements(By.CSS_SELECTOR, text_selector)
                                for text_elem in text_elements:
                                    text_content = text_elem.text.strip()
                                    if text_content and len(text_content) > 10:
                                        review_text = text_content
                                        break
                                if review_text != "리뷰 텍스트 없음":
                                    break
                            except:
                                continue
                except:
                    # 여전히 텍스트를 못찾았다면 전체 텍스트에서 추출 시도
                    try:
                        full_text = review_element.text.strip()
                        if len(full_text) > 20:
                            sentences = full_text.split('\n')
                            for sentence in sentences:
                                if len(sentence.strip()) > 10:
                                    review_text = sentence.strip()
                                    break
                    except:
                        pass
                
                review_info['text'] = review_text
                
                # 리뷰 날짜 - 두 번째 span.pui__blind 요소에서 실제 날짜 추출
                review_date = "날짜 정보 없음"
                try:
                    # span.pui__gfuUIT 안의 모든 span.pui__blind 요소들 찾기
                    date_container = review_element.find_element(By.CSS_SELECTOR, "span.pui__gfuUIT")
                    date_elements = date_container.find_elements(By.CSS_SELECTOR, "span.pui__blind")
                    
                    # 두 번째 span.pui__blind 요소에서 날짜 추출 (첫 번째는 "방문일"이므로 제외)
                    if len(date_elements) >= 2:
                        date_text = date_elements[1].text.strip()  # 두 번째 요소 사용
                        
                        if date_text:
                            # 날짜 패턴 추출
                            date_patterns = [
                                r'(\d{4}년\s*\d{1,2}월\s*\d{1,2}일)',  # 2025년 5월 15일
                                r'(\d{4}\.\d{1,2}\.\d{1,2})',         # 2024.01.15
                                r'(\d{4}-\d{1,2}-\d{1,2})',          # 2024-01-15
                                r'(\d{1,2}\.\d{1,2})',               # 01.15
                                r'(\d+일전)',                         # 3일전
                                r'(\d+주전)',                         # 2주전
                                r'(\d+개월전)',                       # 1개월전
                                r'(\d+년전)',                         # 1년전
                                r'(오늘)',                            # 오늘
                                r'(어제)'                             # 어제
                            ]
                            
                            for pattern in date_patterns:
                                date_match = re.search(pattern, date_text)
                                if date_match:
                                    review_date = date_match.group(1)
                                    break
                            
                            # 패턴에 매칭되지 않는 경우 전체 텍스트 사용 (요일 포함된 경우 등)
                            if review_date == "날짜 정보 없음" and len(date_text) < 50:
                                review_date = date_text
                    
                    elif len(date_elements) == 1:
                        # span.pui__blind가 하나뿐인 경우, 그것이 날짜일 가능성
                        date_text = date_elements[0].text.strip()
                        if date_text and "방문일" not in date_text:
                            review_date = date_text
                        
                except:
                    # 대체 날짜 셀렉터들 시도
                    date_selectors = [
                        "time", 
                        "div.pui__QKE5Pr", 
                        "span[class*='date']",
                        "div[class*='date']",
                        "span.time"
                    ]
                    
                    for date_selector in date_selectors:
                        try:
                            date_element = review_element.find_element(By.CSS_SELECTOR, date_selector)
                            date_text = date_element.text.strip()
                            
                            if date_text and len(date_text) < 30 and "방문일" not in date_text:
                                review_date = date_text
                                break
                        except:
                            continue
                
                review_info['date'] = review_date
                
                # 리뷰 평점 (별점) 수집
                try:
                    rating_selectors = [
                        "div.pui__rating span",
                        "span[class*='rating']",
                        "div[class*='star']",
                        ".rating"
                    ]
                    
                    review_rating = "평점 정보 없음"
                    for rating_selector in rating_selectors:
                        try:
                            rating_element = review_element.find_element(By.CSS_SELECTOR, rating_selector)
                            rating_text = rating_element.text.strip()
                            if rating_text and ("점" in rating_text or "★" in rating_text):
                                review_rating = rating_text
                                break
                        except:
                            continue
                    
                    review_info['rating'] = review_rating
                except:
                    review_info['rating'] = "평점 정보 없음"
                
                # 리뷰 이미지들 (개별 리뷰에서) - 프로필 영역 제외
                review_images = []
                try:
                    # 현재 리뷰 요소 내의 프로필 영역 찾기
                    profile_containers = review_element.find_elements(By.CSS_SELECTOR, 'div.pui__q2fg8o.pui__A7NplK')
                    profile_images = []
                    
                    for container in profile_containers:
                        container_images = container.find_elements(By.CSS_SELECTOR, 'img')
                        profile_images.extend(container_images)
                    
                    # alt="방문자리뷰사진"인 이미지만 찾기
                    visitor_images = review_element.find_elements(By.CSS_SELECTOR, 'img[alt="방문자리뷰사진"]')
                    
                    for j, img_element in enumerate(visitor_images):
                        # 이 이미지가 프로필 영역에 속하는지 확인
                        is_profile_image = False
                        for profile_img in profile_images:
                            try:
                                if img_element.get_attribute("src") == profile_img.get_attribute("src"):
                                    is_profile_image = True
                                    break
                            except:
                                continue
                        
                        # 프로필 이미지가 아닌 경우에만 다운로드
                        if not is_profile_image:
                            img_url = img_element.get_attribute("src")
                            img_alt = img_element.get_attribute("alt")
                            
                            # alt 속성이 정확히 "방문자리뷰사진"인지 다시 한번 확인
                            if img_url and img_alt == "방문자리뷰사진":
                                img_filename = f"개별방문자리뷰_{i+1:02d}_{j+1:02d}.jpg"
                                downloaded_path = download_image_improved(img_url, img_filename, review_images_dir)
                                if downloaded_path:
                                    review_images.append(downloaded_path)
                        
                except Exception as e:
                    print(f"리뷰 {i+1} 방문자리뷰사진 수집 실패: {e}")
                
                review_info['individual_images'] = review_images
                review_list.append(review_info)
                
                # 진행 상황 출력
                if i % 10 == 0:
                    print(f"리뷰 처리 진행률: {i+1}/{len(review_elements)}")
                
                print(f"리뷰 {i+1}: {review_info['text'][:50]}..." if len(review_info['text']) > 50 else f"리뷰 {i+1}: {review_info['text']}")
                
            except Exception as e:
                print(f"리뷰 {i+1} 정보 수집 실패: {e}")
                continue
                
    except Exception as e:
        print(f"리뷰 정보 수집 실패: {e}")
        
    print(f"총 {len(review_list)}개의 리뷰 수집 완료")
    return review_list

def go_back_to_list():
    """목록으로 돌아가기"""
    try:
        # 메인으로 돌아가기
        driver.switch_to.default_content()
        
        # 검색 결과 iframe으로 다시 전환
        search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
        driver.switch_to.frame(search_iframe)
        print("목록으로 돌아가기 완료")
        return True
        
    except Exception as e:
        print(f"목록으로 돌아가기 실패: {e}")
        return False

# =================== 메인 실행 부분 ===================
try:
    print("=" * 60)
    print(f"네이버 지도 크롤링 시작")
    print(f"검색어: {SEARCH_KEYWORD}")
    print(f"최대 가게 수: {MAX_RESTAURANTS}")
    print(f"가게당 최대 리뷰 수: {MAX_REVIEWS_PER_RESTAURANT}")
    print("=" * 60)
    
    # 1. 검색어 설정 및 접속
    encoded_keyword = urllib.parse.quote(SEARCH_KEYWORD)
    URL = f"https://map.naver.com/p/search/{encoded_keyword}"
    
    print(f"네이버 지도 접속 중: {URL}")
    driver.get(URL)
    time.sleep(5)
    
    # 2. 디렉토리 생성
    base_dir = create_directory_structure(SEARCH_KEYWORD)
    print(f"저장 디렉토리 생성: {base_dir}")
    
    # 3. 검색 결과 iframe 접근
    print("검색 결과 iframe으로 전환 중...")
    search_iframe = wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "iframe#searchIframe")))
    driver.switch_to.frame(search_iframe)
    
    # 4. 가게 리스트 수집
    restaurant_elements = get_restaurant_list()
    total_restaurants = len(restaurant_elements)
    print(f"총 {total_restaurants}개 가게 발견")
    
    # 5. 각 가게별로 세부 정보 수집
    max_restaurants = min(MAX_RESTAURANTS, total_restaurants)
    
    for i in range(max_restaurants):
        print(f"\n{'='*20} 가게 {i+1}/{max_restaurants} {'='*20}")
        
        try:
            # 가게 클릭
            if not click_restaurant(restaurant_elements[i], i+1):
                continue
            
            # 세부 정보 iframe으로 전환
            if not switch_to_detail_iframe():
                continue
            
            # 홈 정보 수집
            print("홈 정보 수집 중...")
            home_info = get_home_info()
            
            # 가게별 폴더 구조 생성
            restaurant_dir, menu_images_dir, review_images_dir = create_restaurant_directories(
                base_dir, home_info.get('name', f'restaurant_{i+1}')
            )
            
            # 메뉴 정보 수집
            menu_info = []
            if find_and_click_menu_tab():
                print("메뉴 정보 수집 중...")
                menu_info = get_menu_info(menu_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("메뉴 탭을 찾을 수 없어 메뉴 정보 수집을 건너뜁니다.")
            
            # 리뷰 정보 수집
            review_info = []
            if find_and_click_review_tab():
                print("리뷰 정보 수집 중...")
                review_info = get_review_info(review_images_dir, home_info.get('name', f'restaurant_{i+1}'))
            else:
                print("리뷰 탭을 찾을 수 없어 리뷰 정보 수집을 건너뜁니다.")
            
            # 전체 정보 결합
            restaurant_data = {
                'index': i + 1,
                'home_info': home_info,
                'menu_info': menu_info,
                'review_info': review_info,
                'restaurant_dir': restaurant_dir,
                'collected_at': datetime.now().isoformat()
            }
            
            all_restaurant_data.append(restaurant_data)
            print(f"가게 {i+1} 정보 수집 완료!")
            print(f"- 메뉴 개수: {len(menu_info)}")
            print(f"- 리뷰 개수: {len(review_info)}")
            
            # 목록으로 돌아가기
            go_back_to_list()
            time.sleep(2)
            
        except Exception as e:
            print(f"가게 {i+1} 처리 중 오류: {e}")
            # 목록으로 돌아가기 시도
            try:
                go_back_to_list()
            except:
                pass
            continue
    
    # 6. 결과 저장
    print(f"\n{'='*60}")
    print(f"총 {len(all_restaurant_data)}개 가게 상세 정보 수집 완료!")
    
    # JSON 파일 저장
    safe_keyword = safe_filename(SEARCH_KEYWORD)
    json_filename = os.path.join(base_dir, f"{safe_keyword}_detailed_restaurants.json")
    with open(json_filename, 'w', encoding='utf-8') as jsonfile:
        json.dump(all_restaurant_data, jsonfile, ensure_ascii=False, indent=2)
    
    # CSV 파일 저장 (한글 헤더로 개선)
    csv_filename = os.path.join(base_dir, f"{safe_keyword}_restaurants_summary.csv")
    with open(csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:  # utf-8-sig로 BOM 추가
        fieldnames = ['순번', '가게명', '분류', '주소', '전화번호', '메뉴수', '리뷰수', '폴더경로']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            row = {
                '순번': restaurant['index'],
                '가게명': restaurant['home_info'].get('name', ''),
                '분류': restaurant['home_info'].get('category', ''),
                '주소': restaurant['home_info'].get('address', ''),
                '전화번호': restaurant['home_info'].get('phone', ''),
                '메뉴수': len(restaurant['menu_info']),
                '리뷰수': len(restaurant['review_info']),
                '폴더경로': restaurant.get('restaurant_dir', '')
            }
            writer.writerow(row)
    
    # 리뷰 상세 정보 CSV 저장
    reviews_csv_filename = os.path.join(base_dir, f"{safe_keyword}_reviews_detail.csv")
    with open(reviews_csv_filename, 'w', newline='', encoding='utf-8-sig') as csvfile:
        fieldnames = ['가게명', '리뷰번호', '리뷰내용', '작성날짜', '평점', '이미지수']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)
        writer.writeheader()
        
        for restaurant in all_restaurant_data:
            restaurant_name = restaurant['home_info'].get('name', '')
            for idx, review in enumerate(restaurant['review_info']):
                row = {
                    '가게명': restaurant_name,
                    '리뷰번호': idx + 1,
                    '리뷰내용': review.get('text', ''),
                    '작성날짜': review.get('date', ''),
                    '평점': review.get('rating', ''),
                    '이미지수': len(review.get('individual_images', []))
                }
                writer.writerow(row)
    
    # 통계 정보 출력
    total_menus = sum(len(restaurant['menu_info']) for restaurant in all_restaurant_data)
    total_reviews = sum(len(restaurant['review_info']) for restaurant in all_restaurant_data)
    total_images = 0
    
    # 이미지 파일 개수 세기
    for restaurant in all_restaurant_data:
        restaurant_dir = restaurant.get('restaurant_dir', '')
        if os.path.exists(restaurant_dir):
            for root, dirs, files in os.walk(restaurant_dir):
                image_files = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png', '.gif'))]
                total_images += len(image_files)
    
    print(f"\n파일 저장 완료:")
    print(f"- 상세 정보 (JSON): {json_filename}")
    print(f"- 요약 정보 (CSV): {csv_filename}")
    print(f"- 리뷰 상세 (CSV): {reviews_csv_filename}")
    print(f"- 이미지 폴더: 각 가게별 폴더")
    print(f"\n수집 통계:")
    print(f"- 총 가게 수: {len(all_restaurant_data)}")
    print(f"- 총 메뉴 수: {total_menus}")
    print(f"- 총 리뷰 수: {total_reviews}")
    print(f"- 총 이미지 수: {total_images}")
    
    print(f"\n{'='*60}")
    print("크롤링 완료!")

except Exception as e:
    print(f"전체 프로세스 오류 발생: {e}")
    import traceback
    traceback.print_exc()
    
finally:
    # 드라이버 종료
    try:
        pass
#         driver.quit()
#         print("웹드라이버 종료 완료")
    except:
        pass

네이버 지도 크롤링 시작
검색어: 신림동 햄버거
최대 가게 수: 1
가게당 최대 리뷰 수: 10
네이버 지도 접속 중: https://map.naver.com/p/search/%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0
저장 디렉토리 생성: 신림동 햄버거_테스트(1개)_data
검색 결과 iframe으로 전환 중...
가게 리스트 수집 중...
스크롤 컨테이너 찾음
현재 로드된 가게 수: 10
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
현재 로드된 가게 수: 50
더 이상 로드할 가게가 없음
총 50개 가게 발견

==================== 가게 1/1 ====================
가게 1 클릭 완료
세부 정보 iframe으로 전환 완료
홈 정보 수집 중...
가게명: 아토커피 신림점
분류: 카페,디저트
주소: 서울 관악구 관천로 79 1층
메뉴 탭 찾는 중...
탭 요소들 찾음: 6개
탭 1: 텍스트='홈', href='https://pcmap.place.naver.com/restaurant/1539251371/home?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081918&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 2: 텍스트='소식', href='https://pcmap.place.naver.com/restaurant/1539251371/feed?entry=bmp&from=map&fromPanelNum=2&timestamp=202507081918&locale=ko&svcName=map_pcv5&searchText=%EC%8B%A0%EB%A6%BC%EB%8F%99%20%ED%96%84%EB%B2%84%EA%B1%B0'
탭 3: 텍스트='메뉴', href='https://pcmap.plac

  이미지 다운로드 성공: 방문자리뷰_011.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_011.jpg
  이미지 다운로드 성공: 방문자리뷰_012.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_012.jpg
  이미지 다운로드 성공: 방문자리뷰_013.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_013.jpg
  이미지 다운로드 성공: 방문자리뷰_014.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_014.jpg
  이미지 다운로드 성공: 방문자리뷰_015.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_015.jpg
  이미지 다운로드 성공: 방문자리뷰_016.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_016.jpg
  이미지 다운로드 성공: 방문자리뷰_017.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_017.jpg
  이미지 다운로드 성공: 방문자리뷰_018.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_018.jpg
  이미지 다운로드 성공: 방문자리뷰_019.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_019.jpg
  이미지 다운로드 성공: 방문자리뷰_020.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_020.jpg
  이미지 다운로드 성공: 방문자리뷰_021.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_021.jpg
  이미지 다운로드 성공: 방문자리뷰_022.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_022.jpg
  이미지 다운로드 성공: 방문자리뷰_023.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_023.jpg
  이미지 다운로드 성공: 방문자리뷰_024.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_024.jpg
  이미지 다운로드 성공: 방문자리뷰_025.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_025.jpg
  이미지 다운로드 성공: 방문자리뷰_026.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_026.jpg
  이미지 다운로드 성공: 방문자리뷰_027.jpg
  방문자리뷰사진 다운로드: 방문자리뷰_027.j